# KLTN — Kiểm chứng rò rỉ không gian: notebook đọc code cho nhóm

**Ngày đóng gói: 11/10/2026.** Phạm vi bắt đầu từ audit split đầu tiên, qua ownership M3/M5 và frozen sensitivity, đến suite GAN can thiệp nguồn lúc train hiện tại. Không chứa training baseline A/B hoặc nhánh source coverage/collision/ROS.

Notebook này chứa đầy đủ các module nghiên cứu, code kiểm và driver trong phạm vi trên. Code gốc hiển thị trực tiếp dưới từng chú thích; không cần tải các `.py` riêng. **Chỉ mở notebook là đọc được; không cần chạy.**

**Cách Run All mặc định:** `REVIEW_ACTION = "read_only"`; notebook chỉ lưu các source trong thư mục `_leakage_team_review_4c08e7e92312`, kiểm SHA và định nghĩa driver. Không train, không load checkpoint, không chấm validation/test. Dòng `%%luu_ma` đầu cell là lệnh Jupyter lưu source, không thực thi đoạn Python bên dưới. Giữ nguyên source khi review để SHA/release không đổi.

| Phần | Câu hỏi | Mốc đã có |
|---|---|---|
| 01 | Split cũ có dùng chung raw sources? Full11×11 có chia đủ16 lớp thành3tập disjoint? | Audit CPU và bất khả thi ở lớp1/7/9/16 |
| 02 | DIP/GT và A/B có cùng identity? | 161 nhãn bất đồng; B final5528/4721 |
| 03 | Đổi observation ownership có giữ16 lớp và zero source intersection? | M3/M5 CPU passed; chưa classification |
| 04 | Ownership cố định còn dư địa giữ context? | M3:40,4625; cận40,6327 nguồn/mẫu |
| 05 | Classifier B nhạy với shared-context perturbation thế nào? | Frozen sensitivity seed42,2962mẫu/14lớp |
| 06 | Chặn source reuse lúc train ảnh hưởng whole HyperGAMO thế nào? | Benchmark400/nhánh passed; full30jobs và test chưa có |
| 07 | Code kiểm/builder và thiết kế phụ | Source QA; crossedM3 chưa có runner36jobs |

**Mốc khoa học:** giao nguồn đã được xác nhận. Những phép thử cũ chưa xác định số điểm accuracy do leakage. Suite mới thay observation khi train rồi chấm cùng test; nó vẫn có confound content/hướng và không tách hiệu ứng generator riêng.


### Cell cấu hình — chỉ chọn action khi muốn thực thi

**Mục đích:** tách đọc code khỏi các lần chạy nghiên cứu. **Đầu vào:** đường dẫn nguồn theo từng action. **Đầu ra mặc định:** các hằng số, chưa mở data.

Các action CPU: `split_audit`, `identity`, `m3_m5`, `context_bound`, `probe_feasibility`. Action `frozen_sensitivity` chỉ inference B đã lưu. Action `gan_prepare` chạy CPU; `gan_benchmark`, `gan_train`, `gan_test` dùng GPU. Chỉ chọn một action mỗi lần; giữ `read_only` để gửi nhóm.


In [ ]:
REVIEW_ACTION = "read_only"
HSI_PATH = None               # .mat corrected HSI đúng SHA; None tự tìm dưới /kaggle/input
GT_PATH = None                # .mat standard GT đúng SHA
COMPARISON_SOURCE = None      # Thư mục/ZIP comparison_4methods_20fffe805617 cho split_audit
COMPARISON_ZIP = None         # ZIP run A cũ cho audit M3/M5
B_OUTPUT_SOURCE = None        # Output B đầy đủ50e988fae260 cho frozen_sensitivity
OUTPUT_ROOT = "/kaggle/working" if __import__("pathlib").Path("/kaggle/working").is_dir() else str(__import__("pathlib").Path.cwd()/"leakage_review_outputs")
IMPACT_DEVICE = "/GPU:0"
RESUME_RUN_PATH = None        # Output suite GAN đầy đủ; REPORT.zip không có weights
MAX_WORKERS = 2
GPU_IDS = None
SESSION_HOURS = 8
SAVE_EVERY = 100
BENCHMARK_C_UPDATES = 400
JOB_FILTER = None             # Mặc định30 jobs; chỉ dùng filter nếu thực sự cần


### Cell hạ tầng — lưu module nguyên bản và kiểm SHA

**Mục đích:** tránh lẫn namespace hoặc lỗi `__file__` khi ghép nhiều script. **Đầu vào:** source hiển thị ở các cell tiếp theo. **Đầu ra:** thư mục source riêng của notebook. Dòng magic chỉ lưu chữ, không execute hàm/main. SHA giữ cả comments và xuống dòng để suite có thể resume cùng release.


In [ ]:
# Cell hạ tầng: chỉ lưu mã nguồn; không import TensorFlow hoặc chạy thí nghiệm.
from pathlib import Path
import ast
import hashlib
import json
import sys
from IPython.core.magic import register_cell_magic

REVIEW_CODE_ROOT = Path.cwd() / "_leakage_team_review_4c08e7e92312"
REVIEW_CODE_ROOT.mkdir(parents=True, exist_ok=True)
REVIEW_SOURCE_SPECS = {'cell_1_read_run.py': {'sha256': 'af3a4939dd6b3c657d0ad2ce3aba384c664039e78c09c93ab685a04497fde813',
                        'newline': 'LF',
                        'parts': 1},
 'cell_2_geometry.py': {'sha256': '8287e57ba5e079ff630e478c9195f3e8aa2a3e78c58e439c474bcd8cc4543778',
                        'newline': 'LF',
                        'parts': 1},
 'cell_3_report.py': {'sha256': '113fe2e56e030d5372dd4fcc727bc3b3e97c201d851c2f5e497ff531b8fae61f',
                      'newline': 'LF',
                      'parts': 1},
 'team_review_adapter.py': {'sha256': 'ec79196be40d092228167edb24e9c47ea7a17bbabd4703be29a44045c10cd669',
                            'newline': 'LF',
                            'parts': 1},
 'spatial_leakage_core.py': {'sha256': '0aa6c374aa8aa66b2208cb5c69678abd577379782c0c99260e94c1bfed09d297',
                             'newline': 'LF',
                             'parts': 1},
 'audit_real_data.py': {'sha256': '09535b9830ab7b79c7ededdf62c6a16ad88a6667142c900b55077660e125dbd3',
                        'newline': 'LF',
                        'parts': 2},
 'context_retention_bound.py': {'sha256': 'ff39a65ce3bca7a542690b22162578daa798b565406e3a6c353e01b84abe20b3',
                                'newline': 'LF',
                                'parts': 1},
 'impact_probe_feasibility.py': {'sha256': '3e13db21b44ed90250b14b7faf40e19b70a2cf1a2f1b61b4e6827154c8321a3a',
                                 'newline': 'LF',
                                 'parts': 1},
 'impact_probe_core.py': {'sha256': 'f78728c533c08715eebf3add1675163dbe33e4b13542d12f1b7f252334d9214f',
                          'newline': 'LF',
                          'parts': 2},
 'impact_probe_bootstrap_fragment.py': {'sha256': 'd480092e2f562928f11a3aaf105e26469d5f5330da86a1aaac155ae0375801cb',
                                        'newline': 'CRLF',
                                        'parts': 4},
 'impact_probe_main_fragment.py': {'sha256': 'abe5d03887e1f5c66210a3b13db2a0351e49e783a9f8ab6ab84d0ab96e69339a',
                                   'newline': 'LF',
                                   'parts': 1},
 'gan_exposure_data.py': {'sha256': 'bf7cde0d7305f430d8ab85866b7e3b66ea99903cf7d5ff8fac5d6bbe71920733',
                          'newline': 'LF',
                          'parts': 7},
 'gan_exposure_training.py': {'sha256': '8a1ef7dd13702665cfa56c30ef2653b43076a9c2af67d6a31af17aea50ad5c63',
                              'newline': 'LF',
                              'parts': 9},
 'gan_exposure_runner.py': {'sha256': 'bc5616400f49be3a40d86fa39f69ce019941f2ac9f1886414e2a22a3f465f5be',
                            'newline': 'LF',
                            'parts': 5},
 'gan_exposure_region_selection.json': {'sha256': 'c91ea95a6d91df3751b3b13643fb4545234b5e7565960e6b255cef4a887c912c',
                                        'newline': 'CRLF',
                                        'parts': 1},
 'verify_cells.py': {'sha256': '2e8217f03ea442f96cedb2d690b29adbe7ebef8cd2513eeb1dfeb9292d559fdd',
                     'newline': 'LF',
                     'parts': 1},
 'verify_impact_probe.py': {'sha256': 'e32300405e105e4e4ae61a6332331d147dabad6606c70b74e24ffb9cd5db403e',
                            'newline': 'LF',
                            'parts': 2},
 'exposure_data_verify.py': {'sha256': '1d4eaff41b4c195e1d4a2bce992469810b6d508db27836f55c575ad5471ef7cf',
                             'newline': 'LF',
                             'parts': 1},
 'exposure_review_verify.py': {'sha256': '2e48e6455adc6925948e73cbc414845578ce35ca642c526bc112d65a877493cf',
                               'newline': 'LF',
                               'parts': 2},
 'exposure_runner_verify.py': {'sha256': '6b644f34da149dac5e1ca525084e2e1cae3c9627cc362489b1d3d7b38669f207',
                               'newline': 'LF',
                               'parts': 2},
 'audit_received_exposure_report.py': {'sha256': '69310f7a8a9331079e9dc991bdc8b032a254d346238e2eff97d3b91f96be66f4',
                                       'newline': 'LF',
                                       'parts': 1},
 'build_impact_probe.py': {'sha256': '65ff4127536c24badcd7a8b1efc2cd013df7ff124bdd113a8446ba6676a33e8e',
                           'newline': 'LF',
                           'parts': 1},
 'build_exposure_notebook.py': {'sha256': '153c30fdc87a3368751f080efdb4b268e51f892d5e3860f0cdfc3d64fe5e776a',
                                'newline': 'LF',
                                'parts': 1},
 'package_exposure_release.py': {'sha256': 'f6c4d7f64e41bcfaf5b0209e41f381cf7c7ea8650645b2fd72bcda4ad1791362',
                                 'newline': 'LF',
                                 'parts': 1},
 'plot_certificate.py': {'sha256': 'a1ecf1e62af54450dc4e4611da448e118a5b9600c9c3df956545c48e9f0bc569',
                         'newline': 'LF',
                         'parts': 1},
 'strong_gan_design_geometry.json': {'sha256': '5d70af33da8993996bd45d39ec069dfb603a86fddc408743c118509199aa711d',
                                     'newline': 'CRLF',
                                     'parts': 1},
 'gan_exposure_release.py': {'sha256': '813627d0357de6b79f7991489b9241291c98555bf039f1ee47a7dc47464c8bdf',
                             'newline': 'LF',
                             'parts': 0}}
_review_parts = {}


def _review_write_exact(name, blob):
    if Path(name).name != name or name not in REVIEW_SOURCE_SPECS:
        raise ValueError("Tên nguồn không nằm trong manifest: " + name)
    expected = REVIEW_SOURCE_SPECS[name]["sha256"]
    if hashlib.sha256(blob).hexdigest() != expected:
        raise ValueError("Nguồn bị sửa hoặc thiếu phần: " + name)
    path = REVIEW_CODE_ROOT/name
    if path.exists() and path.read_bytes() != blob:
        raise RuntimeError("Thư mục review chứa source khác; chọn thư mục review mới: " + str(path))
    if not path.exists():
        path.write_bytes(blob)
    return path


@register_cell_magic
def luu_ma(line, cell):
    """%%luu_ma only stages text. Original Python starts after the first line."""
    name, part, edges, expected = line.split()
    index, total = map(int, part.split("/"))
    leading, trailing = map(int, edges.split(":"))
    spec = REVIEW_SOURCE_SPECS[name]
    if total != spec["parts"] or not 1 <= index <= total:
        raise ValueError("Sai thứ tự phần nguồn")
    body = cell.replace("\r\n", "\n").strip("\n")
    normalized = "\n"*leading + body + "\n"*trailing
    if hashlib.sha256(normalized.encode("utf-8")).hexdigest() != expected:
        raise ValueError("Nội dung cell khác bản đã khóa: " + name)
    _review_parts.setdefault(name, {})[index] = normalized
    if len(_review_parts[name]) == total:
        source = "".join(_review_parts[name][i] for i in range(1, total+1))
        if name.endswith(".py"):
            ast.parse(source, filename=name)
        elif name.endswith(".json"):
            json.loads(source)
        if spec["newline"] == "CRLF":
            source = source.replace("\n", "\r\n")
        _review_write_exact(name, source.encode("utf-8"))
    # No exec/import/main/fit/predict inside this magic.


## 01 — Audit split và raw-source overlap ban đầu

Run A:4557 train/1132val/4711test. Lớp1/7/9/16 không có bộ ba tâm đôi một cách nhau đủ để ba full patches11×11 rời nguồn. Sáu pruning khảo sát làm mất phần lớn tập; chưa được chọn thành protocol.

### Code — `cell_1_read_run.py` · phần 1/1: sa_require, sa_json

**Mục đích:** Đọc metadata, khóa run A và tọa độ split; không đọc weights.

**Đầu vào:** Output/ZIP comparison_4methods_20fffe805617. **Đầu ra:** CSV/JSON/mask và ZIP split_audit.

**Trạng thái:** Đã chạy CPU trên nguồn thật. **Giới hạn/cần review:** Ba cell cần chung namespace. Pruning không phải chia tối ưu; không sửa nhãn/split gốc.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma cell_1_read_run.py 1/1 0:1 af3a4939dd6b3c657d0ad2ce3aba384c664039e78c09c93ab685a04497fde813
# CELL 1/3 — Đọc metadata và tọa độ của đúng run (CPU).
# Dán vào một cell Code mới ở cuối notebook. Chỉ chạy 3 cell mới theo thứ tự.
from pathlib import Path
from datetime import datetime, timezone
import hashlib, io, itertools, json, zipfile
import numpy as np
import pandas as pd
from scipy.ndimage import maximum_filter
from IPython.display import display, FileLink

SA_SOURCE = ""  # Để trống tự tìm; hoặc điền thư mục run / đường dẫn ZIP.
SA_RUN_NAME = "comparison_4methods_20fffe805617"
SA_OUTPUT_PARENT = Path("/kaggle/working")
# Ngưỡng khảo sát số mẫu, KHÔNG phải bảo đảm đủ độ tin cậy thống kê.
SA_MIN_COUNTS = (20, 5, 5)  # train, validation, test; không dùng để tự bỏ lớp.

def sa_require(condition, message):
    if not condition:
        raise ValueError(message)

if not SA_SOURCE:
    sa_preferred = Path("/kaggle/input/notebooks/minhlongg/notebook708cde343b") / SA_RUN_NAME
    if (sa_preferred / "split_indices.npz").is_file():
        SA_SOURCE = str(sa_preferred)
    else:
        sa_dirs, sa_zips = [], []
        for sa_root in (Path("/kaggle/input"), Path("/kaggle/working")):
            if sa_root.exists():
                sa_dirs.extend(p.parent for p in sa_root.rglob("comparison_config.json")
                               if p.parent.name == SA_RUN_NAME)
                sa_zips.extend(sa_root.rglob(SA_RUN_NAME + "*.zip"))
        sa_candidates = sorted(set(sa_dirs or sa_zips))
        sa_require(len(sa_candidates) == 1,
                   "Điền SA_SOURCE để chọn đúng nguồn. Tìm thấy: " + str(sa_candidates))
        SA_SOURCE = str(sa_candidates[0])

sa_source = Path(SA_SOURCE)
sa_require(sa_source.exists(), "Không thấy nguồn: " + str(sa_source))
sa_files = ("config.json", "comparison_config.json", "manifest.json", "split_indices.npz")
sa_blobs = {}
if sa_source.is_file():
    with zipfile.ZipFile(sa_source) as sa_z:
        sa_prefixes = [n[:-len("comparison_config.json")] for n in sa_z.namelist()
                       if n in ("comparison_config.json", SA_RUN_NAME + "/comparison_config.json")
                       or n.endswith("/" + SA_RUN_NAME + "/comparison_config.json")]
        sa_require(len(sa_prefixes) == 1, "ZIP không xác định được duy nhất thư mục run.")
        for sa_name in sa_files:
            sa_info = sa_z.getinfo(sa_prefixes[0] + sa_name)
            sa_require(sa_info.file_size < 4_000_000, "Metadata lớn bất thường: " + sa_name)
            sa_blobs[sa_name] = sa_z.read(sa_info)
else:
    sa_require(sa_source.name == SA_RUN_NAME, "SA_SOURCE phải trỏ đúng thư mục run.")
    for sa_name in sa_files:
        sa_require((sa_source / sa_name).stat().st_size < 4_000_000,
                   "Metadata lớn bất thường: " + sa_name)
        sa_blobs[sa_name] = (sa_source / sa_name).read_bytes()

sa_cfg = json.loads(sa_blobs["config.json"])
sa_comparison = json.loads(sa_blobs["comparison_config.json"])
sa_manifest = json.loads(sa_blobs["manifest.json"])
sa_require(sa_cfg["protocol"] == "dip_train_only_validation"
           and sa_cfg["accepted_class_ids"] == list(range(1, 17))
           and (sa_cfg["patch_size"], sa_cfg["pca_components"]) == (11, 40)
           and sa_comparison["seeds"] == [42, 43, 44], "Nguồn khác protocol đã xác nhận.")
with np.load(io.BytesIO(sa_blobs["split_indices.npz"]), allow_pickle=False) as sa_d:
    sa_split = sa_d["split"].copy()
    sa_labels = sa_d["labels_protocol"].copy()
    sa_gt = sa_d["standard_gt"].copy()
    sa_require(sa_split.shape == sa_labels.shape == sa_gt.shape == (145, 145), "Sai shape.")
    sa_require(all(np.issubdtype(a.dtype, np.integer) for a in (sa_split, sa_labels, sa_gt)),
               "Tọa độ/nhãn phải là số nguyên.")
    sa_require(set(np.unique(sa_split)) <= {-1, 0, 1, 2}, "Mã split không hợp lệ.")
    sa_require(set(np.unique(sa_labels)) == set(range(17))
               and set(np.unique(sa_gt)) == set(range(17)), "Thiếu hoặc sai nhãn lớp.")
    sa_require(np.all((sa_labels > 0) == (sa_split >= 0)), "Nhãn không khớp mặt nạ split.")
    for sa_k, sa_field in enumerate(("coords_train", "coords_val", "coords_test")):
        np.testing.assert_array_equal(sa_d[sa_field], np.argwhere(sa_split == sa_k))
    np.testing.assert_array_equal(sa_d["class_ids"], np.arange(1, 17))
sa_require([int((sa_split == k).sum()) for k in range(3)] == [4557, 1132, 4711],
           "Số mẫu khác run đã xác nhận.")

SA_PATCH = int(sa_cfg["patch_size"])
SA_NAMES = ["Alfalfa", "Corn-notill", "Corn-mintill", "Corn", "Grass-pasture",
            "Grass-trees", "Grass-pasture-mowed", "Hay-windrowed", "Oats",
            "Soybean-notill", "Soybean-mintill", "Soybean-clean", "Wheat", "Woods",
            "Buildings-Grass-Trees-Drives", "Stone-Steel-Towers"]
sa_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
sa_out = SA_OUTPUT_PARENT / ("split_audit_" + SA_RUN_NAME[-12:] + "_" + sa_tag)
sa_out.mkdir(parents=True, exist_ok=False)

def sa_json(name, value):
    (sa_out / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
                               encoding="utf-8")

sa_json("provenance.json", dict(source=str(sa_source), run=SA_RUN_NAME,
    config=sa_cfg, comparison=sa_comparison, minimum_counts=list(SA_MIN_COUNTS),
    metadata_sha256={k: hashlib.sha256(v).hexdigest() for k, v in sa_blobs.items()},
    reads="Metadata, tọa độ và nhãn chia tập; không đọc cache ảnh hay dự đoán test.",
    training_performed=False, test_performance_evaluated=False))
print("Đã đọc đúng nguồn:", sa_source)
print("Train/validation/test: 4557 / 1132 / 4711. Batch CNN đã lưu:", sa_comparison["batch_size"])
print("Báo cáo riêng:", sa_out)
print("CELL 1 XONG — chạy cell 2.")


### Code — `cell_2_geometry.py` · phần 1/1: sa_overlap, sa_pair_triple

**Mục đích:** Đếm giao raw pixels, đối chiếu nhãn, tìm pair/triple tâm không chồng patch.

**Đầu vào:** Output/ZIP comparison_4methods_20fffe805617. **Đầu ra:** CSV/JSON/mask và ZIP split_audit.

**Trạng thái:** Đã chạy CPU trên nguồn thật. **Giới hạn/cần review:** Ba cell cần chung namespace. Pruning không phải chia tối ưu; không sửa nhãn/split gốc.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma cell_2_geometry.py 1/1 0:1 8287e57ba5e079ff630e478c9195f3e8aa2a3e78c58e439c474bcd8cc4543778
# CELL 2/3 — Audit split cũ + điều kiện hình học, không huấn luyện.
sa_require("sa_out" in globals(), "Chạy cell 1 trước.")

def sa_overlap(mask, patch):
    result = {}
    footprints = [maximum_filter((mask == k).astype(np.uint8), size=patch,
                                 mode="constant", cval=0) > 0 for k in range(3)]
    for a, b in ((0, 1), (0, 2), (1, 2)):
        key = ("train", "validation", "test")[a] + "_" + ("train", "validation", "test")[b]
        result[key + "_shared_raw_pixels"] = int((footprints[a] & footprints[b]).sum())
        near = maximum_filter((mask == a).astype(np.uint8), size=2 * patch - 1,
                              mode="constant", cval=0) > 0
        result[key + "_overlapping_centers_in_second_set"] = int((near & (mask == b)).sum())
    return result

sa_old_overlap = sa_overlap(sa_split, SA_PATCH)
for sa_key, sa_value in sa_old_overlap.items():
    sa_require(sa_manifest["overlap"][sa_key] == sa_value, "Audit khác manifest: " + sa_key)
sa_json("existing_overlap.json", sa_old_overlap)
print("Đã đối chiếu lại tọa độ với audit cũ:")
display(pd.DataFrame([sa_old_overlap]).T.rename(columns={0: "count"}))

sa_train_val = np.isin(sa_split, (0, 1))
sa_label_audit = dict(
    train_vs_standard_gt_disagreements=int((sa_train_val & (sa_labels != sa_gt)).sum()),
    assigned_on_gt_background=int((sa_train_val & (sa_gt == 0)).sum()),
    assigned_different_nonzero_label=int((sa_train_val & (sa_gt > 0) & (sa_labels != sa_gt)).sum()))
sa_require(sa_label_audit == sa_manifest["label_audit"], "Audit nhãn khác manifest.")
sa_json("label_audit.json", sa_label_audit)
print("Khác biệt nhãn:", sa_label_audit)

def sa_pair_triple(coords, patch):
    """Tìm chính xác 2/3 tâm có patch đôi một không giao nhau, trong MỘT lớp.
    Khoảng cách Chebyshev > patch-1; khớp zero padding của run nguồn.
    Có bộ ba chỉ là điều kiện cần cho toàn split.
    """
    coords = np.asarray(coords, dtype=np.int32)
    n = len(coords)
    if n < 2:
        return None, None
    far = (np.abs(coords[:, None, 0] - coords[None, :, 0]) >= patch)
    far |= (np.abs(coords[:, None, 1] - coords[None, :, 1]) >= patch)
    pair = None
    for i in range(n - 1):
        neighbors = np.flatnonzero(far[i, i+1:]) + i + 1
        if len(neighbors) and pair is None:
            pair = [int(i), int(neighbors[0])]
        for j in neighbors:
            k = np.flatnonzero(far[i, j+1:] & far[j, j+1:])
            if len(k):
                return pair, [int(i), int(j), int(j + 1 + k[0])]
    return pair, None

sa_geometry, sa_witnesses = [], {}
for sa_basis, sa_map in (("DIP_labels", sa_labels), ("standard_gt", sa_gt)):
    for sa_cid in range(1, 17):
        sa_coords = np.argwhere(sa_map == sa_cid)
        sa_pair, sa_triple = sa_pair_triple(sa_coords, SA_PATCH)
        sa_diameter = int(np.ptp(sa_coords, axis=0).max()) if len(sa_coords) else 0
        sa_geometry.append(dict(label_basis=sa_basis, class_id=sa_cid,
            class_name=SA_NAMES[sa_cid-1], samples=len(sa_coords),
            max_chebyshev_distance=sa_diameter, has_disjoint_pair=sa_pair is not None,
            has_disjoint_triple=sa_triple is not None))
        sa_witnesses[f"{sa_basis}_{sa_cid}"] = dict(
            pair=None if sa_pair is None else sa_coords[sa_pair].tolist(),
            triple=None if sa_triple is None else sa_coords[sa_triple].tolist())
sa_geometry = pd.DataFrame(sa_geometry)
sa_geometry.to_csv(sa_out / "geometry_by_class.csv", index=False)
sa_json("geometry_witnesses.json", sa_witnesses)
display(sa_geometry)
for sa_basis in ("DIP_labels", "standard_gt"):
    sa_rows = sa_geometry[sa_geometry.label_basis == sa_basis]
    sa_blocked = sa_rows.loc[~sa_rows.has_disjoint_triple, "class_id"].tolist()
    print(sa_basis, "— lớp KHÔNG có nổi 3 tâm đôi một không chồng lấn:", sa_blocked)
print("Nếu một lớp không có bộ ba: không thể có đủ lớp đó ở cả train/val/test không chồng lấn.")
print("Có bộ ba chưa bảo đảm chia được toàn cảnh hoặc đủ số mẫu. Không tự đổi nhãn/lớp/patch.")
print("CELL 2 XONG — chạy cell 3 để xuất báo cáo.")


### Code — `cell_3_report.py` · phần 1/1: sa_counts, sa_prune

**Mục đích:** Khảo sát sáu thứ tự pruning và xuất báo cáo/mask riêng.

**Đầu vào:** Output/ZIP comparison_4methods_20fffe805617. **Đầu ra:** CSV/JSON/mask và ZIP split_audit.

**Trạng thái:** Đã chạy CPU trên nguồn thật. **Giới hạn/cần review:** Ba cell cần chung namespace. Pruning không phải chia tối ưu; không sửa nhãn/split gốc.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma cell_3_report.py 1/1 0:1 113fe2e56e030d5372dd4fcc727bc3b3e97c201d851c2f5e497ff531b8fae61f
# CELL 3/3 — Khảo sát loại patch chồng lấn từ split cũ, lưu bảng/hình/ZIP.
sa_require("sa_geometry" in globals(), "Chạy cell 2 trước.")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import html

def sa_counts(mask, labels):
    return np.array([[int(((mask == k) & (labels == c)).sum()) for k in range(3)]
                     for c in range(1, 17)], dtype=int)

def sa_prune(mask, order, patch):
    # Giữ nguyên tập có ưu tiên cao; các tập sau bỏ tâm giao với patch đã giữ.
    # Đây là khảo sát bảo thủ 6 thứ tự, KHÔNG phải tối ưu toàn cục.
    result = np.full(mask.shape, -1, dtype=np.int8)
    kept = np.zeros(mask.shape, dtype=bool)
    for k in order:
        forbidden = maximum_filter(kept.astype(np.uint8), size=2 * patch - 1,
                                   mode="constant", cval=0) > 0
        retain = (mask == k) & ~forbidden
        result[retain] = k
        kept |= retain
    return result

sa_min_raw = np.asarray(SA_MIN_COUNTS)
sa_require(sa_min_raw.shape == (3,) and np.issubdtype(sa_min_raw.dtype, np.integer)
           and (sa_min_raw >= 1).all(), "Ngưỡng mẫu phải gồm 3 số nguyên dương.")
sa_min = sa_min_raw.astype(int)
sa_before = sa_counts(sa_split, sa_labels)
sa_per_class, sa_variants, sa_masks, sa_prune_audits = [], [], {}, {}
for sa_order in itertools.permutations(range(3)):
    sa_key = "_".join(("train", "val", "test")[k] for k in sa_order)
    sa_mask = sa_prune(sa_split, sa_order, SA_PATCH)
    sa_audit = sa_overlap(sa_mask, SA_PATCH)
    sa_require(not any(sa_audit.values()), "Lỗi kiểm tra loại chồng lấn: " + sa_key)
    sa_after = sa_counts(sa_mask, sa_labels)
    sa_masks[sa_key] = sa_mask
    sa_prune_audits[sa_key] = sa_audit
    sa_variants.append(dict(priority=sa_key, train=int(sa_after[:, 0].sum()),
        validation=int(sa_after[:, 1].sum()), test=int(sa_after[:, 2].sum()),
        classes_present_in_all_three=int((sa_after > 0).all(axis=1).sum()),
        classes_meeting_threshold=int((sa_after >= sa_min).all(axis=1).sum()),
        removed=int(sa_before.sum() - sa_after.sum())))
    for sa_i in range(16):
        sa_per_class.append(dict(priority=sa_key, class_id=sa_i+1, class_name=SA_NAMES[sa_i],
            train_before=int(sa_before[sa_i, 0]), val_before=int(sa_before[sa_i, 1]),
            test_before=int(sa_before[sa_i, 2]), train_after=int(sa_after[sa_i, 0]),
            val_after=int(sa_after[sa_i, 1]), test_after=int(sa_after[sa_i, 2]),
            meets_threshold=bool((sa_after[sa_i] >= sa_min).all())))
sa_variants = pd.DataFrame(sa_variants)
pd.DataFrame(sa_per_class).to_csv(sa_out / "pruning_per_class.csv", index=False)
sa_variants.to_csv(sa_out / "pruning_summary.csv", index=False)
sa_json("pruning_overlap.json", sa_prune_audits)
# Tệp chẩn đoán riêng, không ghi đè split_indices.npz của run gốc.
np.savez_compressed(sa_out / "diagnostic_pruned_masks_NOT_FOR_TRAINING.npz", **sa_masks)
display(sa_variants)
print("Cả 6 phương án đều có 0 pixel dùng chung giữa các tập; xem số lớp/mẫu còn lại.")
print("Các mask chỉ để khảo sát, chưa được chọn làm protocol huấn luyện.")

sa_fig, sa_axes = plt.subplots(2, 4, figsize=(16, 8))
sa_cmap = ListedColormap(["#dddddd", "#258955", "#e1a82e", "#397ac1"])
for sa_ax, (sa_title, sa_mask) in zip(sa_axes.flat,
        [("Original DIP (overlap)", sa_split)] + list(sa_masks.items())):
    sa_ax.imshow(sa_mask + 1, cmap=sa_cmap, vmin=0, vmax=3, interpolation="nearest")
    sa_ax.set_title(sa_title, fontsize=9)
    sa_ax.axis("off")
sa_axes.flat[-1].axis("off")
sa_axes.flat[-1].text(0, .6, "Gray: excluded/background\nGreen: train\nYellow: validation\nBlue: test\n\nDiagnostics only", fontsize=11)
sa_fig.tight_layout()
sa_fig.savefig(sa_out / "split_diagnostics.png", dpi=160)
plt.show()
plt.close(sa_fig)

sa_blocks = {basis: sa_geometry.loc[(sa_geometry.label_basis == basis)
             & ~sa_geometry.has_disjoint_triple, "class_id"].tolist()
             for basis in ("DIP_labels", "standard_gt")}
sa_conclusion = {
    "run": SA_RUN_NAME, "patch_size": SA_PATCH,
    "padding_geometry": "Zero padding như run gốc; audit pixel thật trong biên ảnh.",
    "classes_without_disjoint_triple": sa_blocks,
    "all_16_three_way_disjoint_impossible_by_geometry": {k: bool(v) for k, v in sa_blocks.items()},
    "minimum_counts_exploratory_only": list(SA_MIN_COUNTS),
    "new_training_protocol_selected": False, "training_performed": False,
    "test_performance_evaluated": False,
    "limits": "Per-class triple is necessary, not sufficient. Six pruning orders are not a global search. Labels/coordinates of all splits were inspected for geometry, not model scores.",
    "next_step": "Nếu có lớp bị chặn, thống nhất thay đổi yêu cầu (patch/phạm vi/dữ liệu hoặc chấp nhận giới hạn DIP) trước khi tạo protocol mới; không tự bỏ lớp."}
sa_json("conclusion.json", sa_conclusion)
sa_html = ("<!doctype html><meta charset='utf-8'><title>Kiểm tra split</title>"
    "<h1>Kiểm tra split — DIP 16 lớp, patch 11×11</h1><p>Nguồn: "
    + html.escape(str(sa_source)) + "</p><p>Chỉ khảo sát; chưa chọn protocol mới.</p>"
    + "<h2>Điều kiện hình học theo từng lớp</h2>" + sa_geometry.to_html(index=False)
    + "<h2>Loại chồng lấn theo sáu thứ tự</h2>" + sa_variants.to_html(index=False)
    + "<img src='split_diagnostics.png' style='max-width:100%'><h2>Kết luận</h2><pre>"
    + html.escape(json.dumps(sa_conclusion, ensure_ascii=False, indent=2)) + "</pre>")
(sa_out / "BAO_CAO_SPLIT.html").write_text(sa_html, encoding="utf-8")
sa_zip = sa_out.with_suffix(".zip")
with zipfile.ZipFile(sa_zip, "w", zipfile.ZIP_DEFLATED) as sa_z:
    for sa_file in sorted(sa_out.iterdir()):
        if sa_file.is_file():
            sa_z.write(sa_file, sa_out.name + "/" + sa_file.name)
with zipfile.ZipFile(sa_zip) as sa_z:
    sa_require(sa_z.testzip() is None, "ZIP báo cáo lỗi CRC.")
print("HOÀN TẤT. Lớp chặn yêu cầu 3 tập không chồng lấn:", sa_blocks)
print("ZIP BÁO CÁO:", sa_zip)
print("Gửi bảng geometry_by_class + pruning_summary hoặc ZIP nhỏ này để chốt bước tiếp.")
display(FileLink(str(sa_zip)))


## 02 — Identity nhãn A/B và adapter notebook

A author train5689/test4711; B chỉ giữ DIP đồng ý GT chuẩn:5528/4721. B development4422/1106. Làm sạch nhãn không giải quyết spatial overlap. Adapter mới chỉ nối đường dẫn/literals; source gốc không bị đổi.

### Code — `team_review_adapter.py` · phần 1/1: find_mat, build_identity, write_report, run_split_audit

**Mục đích:** Trích literals dữ liệu B; ghép đường dẫn để các bước lịch sử chạy trong notebook.

**Đầu vào:** Hai MAT đúng SHA; comparison ZIP/Output B theo từng action. **Đầu ra:** Identity A/B và các hàm driver.

**Trạng thái:** Adapter mới để đóng gói; thuật toán gốc ở các module bên dưới. **Giới hạn/cần review:** Không execute baseline v2_code.py; không coi adapter là thí nghiệm mới đã chạy.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma team_review_adapter.py 1/1 0:1 ec79196be40d092228167edb24e9c47ea7a17bbabd4703be29a44045c10cd669
"""Notebook adapter (2026-10-11), separate from historical frozen modules.

Only source/path wiring and CPU report reconstruction are new here.
No baseline training, model selection or accuracy evaluation happens here.
"""
from pathlib import Path
import ast
import base64
import hashlib
import io
import json
import sys
import types
import zlib

# Literal extracted from v2_code.py; no execution of the baseline notebook.
DATA_CONFIG = {'patch_size': 11, 'development_seed': 42, 'development_val_fraction': 0.2}
DIP_B64 = 'eJzt201vmzAYwHHDWvKyjeS6G7tvUxupk5JbutOkRZq23KtIoRpTl0RNJvVr7BuPBAjEL/AYG9sk/kuozQvwkwvFNOpsOv82vQ/uPt0Es+n842P0FAbfnxa7x/Xzn0mwWW+jlw/Bl+dwsQuXwXo1Cea//gY/w00w+hzcjiej8eRuFIxubseoRs7XmR9/efmBkBd/7cZLL33pOn38L12ceHkXL9FqGS1Wm2gVbh+W0fb3OlrtHpbbcJeu58bL+3h77r46JqxXSeIbciWJJIPERbJBLNHrNOzpbtzJBuSDGCKNoEzkHMJBmEgfKBedoAwB5apLAhFhIEJ08SAHB+Ei/SAEAiH5INcwUHqeUUCUFIBcs0CF30RGgFw9ID/fAo2TPVsEsSYCUkA+A4RfPEgQKZIB8hkgVxPIZ4BcAoRIECEqgLI4QX4joLf7aoF8HhAiQbiIAuLKZ4O05MNAfAfl1Wn0N3VoiYAcJ7+UXJUmE0ScMHRQuUgeiHIGawF5SYkIBioVoWRXhoGKOxUDecfKQKVR910LhCyIDUqeAB9DqkCHMz/JOFAXsykEFeMAkRvPpis8oNMYswmqiBwhJqh+PNOPFoAov5EtCNuhetBwaBJouK8UhBSChmmmgIYwEDJthHBQgwFBlMsE8wVK2R9IIbddZoIKj1lXe6Wg4o2RdhDaD0/hcs8CUachYBBvrQTRpiFtA3E7+EC0O1W5DFGQ9HHhBQFn9QpB6mLdNWrLgqo6dxDwz5EAkCSRBSkCycHUBg0GBoEGh4wBDbLkCcRqF8irCh1nIYyJAf8RLQbyeEEcI0QTQUCIA1TNEQblH0UAQBBPOQggqwDxNwCJAENkQVUiJaDy47lxECkCgTwZoF6cGaBeHgDEUy1Qr9cukJDIgrSAREQtApVMQSxIGCRDZAyoThSQjPp9paDSN/eTzAEh40BIH8ihlYo0gKic5Bw3DmTUWSaYJBA5RhZ0yaDqz9wUH9RQEHnvfNqbOP7PD2GgDmWltoJ4RQKgqowDofR/XxoBdciVwCA+0TmBKn+yjYI6xEpwEJeoWVCNKCDgShJ2fr6gJqsLuqanDcTwaAOxPPJAfCsxPQ2DWGPH9kgDlWLxd5R4FIIK72kWVI0lx0gv6PidASCbzWaz2Ww2pf0H3l+xCA=='
BASELINE_SOURCE_SHA256 = '0231a5eeb5bc47997d28b0f616cc659807f953eed568b33295e352ae4dde87c6'
RAW_HSI_SHA256 = "ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939"
RAW_GT_SHA256 = "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c"
DIP_MAT_SHA256 = "430c37ce56a160d410378aa136e5e696e2436e2fe03e275f306c3502f622e77b"


def find_mat(explicit, expected, roots):
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError("MAT thiếu hoặc sai SHA: " + str(path))
        return path
    candidates = []
    for root in roots:
        root = Path(root)
        if root.exists():
            for path in root.rglob("*.mat"):
                if hashlib.sha256(path.read_bytes()).hexdigest() == expected:
                    candidates.append(path.resolve())
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError("Điền đường dẫn MAT gốc hoặc thêm Indian Pines vào Input.")
    # Byte-identical duplicates are interchangeable; choose deterministic path.
    return candidates[0]


def build_identity(hsi_path, gt_path):
    import numpy as np
    from scipy.io import loadmat
    assert hashlib.sha256(Path(hsi_path).read_bytes()).hexdigest() == RAW_HSI_SHA256
    assert hashlib.sha256(Path(gt_path).read_bytes()).hexdigest() == RAW_GT_SHA256
    cube = loadmat(hsi_path)["indian_pines_corrected"].astype(np.float32)
    gt = loadmat(gt_path)["indian_pines_gt"].astype(np.int16)
    assert cube.shape == (145, 145, 200) and gt.shape == (145, 145)
    assert set(np.unique(gt)) == set(range(17))
    blob = zlib.decompress(base64.b64decode(DIP_B64))
    assert hashlib.sha256(blob).hexdigest() == DIP_MAT_SHA256
    encoded = loadmat(io.BytesIO(blob))["indianpines_disjoint_dset"].astype(np.int16)
    mapping = np.array([0, 2, 3, 5, 6, 8, 10, 11, 12, 14, 1, 4, 7, 9, 13, 15, 16])
    assert encoded.shape == gt.shape and encoded.min() >= 0 and encoded.max() < len(mapping)
    author = mapping[encoded]
    a_train = author > 0
    a_test = (author == 0) & (gt > 0)
    b_train = (author > 0) & (gt > 0) & (author == gt)
    b_test = (gt > 0) & ~b_train
    mismatch = (author > 0) & (author != gt)
    assert (int(a_train.sum()), int(a_test.sum())) == (5689, 4711)
    assert (int(b_train.sum()), int(b_test.sum())) == (5528, 4721)
    assert (int(mismatch.sum()), int((mismatch & (gt == 0)).sum()),
            int((mismatch & (gt > 0)).sum())) == (161, 151, 10)
    dev_train, dev_val = b_train.copy(), np.zeros_like(b_train)
    rng = np.random.default_rng(DATA_CONFIG["development_seed"])
    for cid in range(1, 17):
        ids = np.flatnonzero((b_train & (gt == cid)).ravel())
        nval = min(len(ids)-1, max(2, int(round(len(ids)*DATA_CONFIG["development_val_fraction"]))))
        chosen = rng.permutation(ids)[:nval]
        rr, cc = np.unravel_index(chosen, gt.shape, order="C")
        dev_train[rr, cc], dev_val[rr, cc] = False, True
    assert (int(dev_train.sum()), int(dev_val.sum())) == (4422, 1106)
    report = {"A_train_test": [5689, 4711], "B_final_train_test": [5528, 4721],
              "B_development_train_validation": [4422, 1106],
              "label_disagreements": {"total": 161, "background": 151, "other_nonzero": 10},
              "baseline_source_sha256": BASELINE_SOURCE_SHA256,
              "adapter": "new CPU wiring; baseline code not executed",
              "spatial_overlap_resolved_by_label_cleaning": False}
    return cube, gt, b_train, b_test, dev_train, dev_val, report


def write_report(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
    print("Đã lưu:", path)


def run_split_audit(source_root, source, output_root):
    """Execute original three cells together, overriding two path assignments only."""
    first = Path(source_root)/"cell_1_read_run.py"
    tree = ast.parse(first.read_text(encoding="utf-8"))
    replacements = {"SA_SOURCE": ast.Constant(str(source or "")),
                    "SA_OUTPUT_PARENT": ast.Call(func=ast.Name(id="Path", ctx=ast.Load()),
                                                 args=[ast.Constant(str(output_root))], keywords=[])}
    changed = set()
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            name = node.targets[0].id
            if name in replacements:
                node.value = replacements[name]
                changed.add(name)
    assert changed == set(replacements)
    namespace = {"__name__": "review_split_audit", "__file__": str(first)}
    exec(compile(ast.fix_missing_locations(tree), str(first), "exec"), namespace)
    for name in ("cell_2_geometry.py", "cell_3_report.py"):
        path = Path(source_root)/name
        namespace["__file__"] = str(path)
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)


def run_m3_m5(source_root, hsi_path, gt_path, comparison_zip, output_root):
    import audit_real_data as audit
    if not comparison_zip or not Path(comparison_zip).is_file():
        raise FileNotFoundError("M3/M5 cần comparison ZIP cũ chứa standard_gt để đối chiếu.")
    audit.ROOT = Path(source_root)
    audit.HSI_PATH, audit.GT_PATH = Path(hsi_path), Path(gt_path)
    audit.SOURCE_ZIP = Path(comparison_zip)
    audit.OUT = Path(output_root)/"m3_m5_cpu_certificate"
    audit.OUT.parent.mkdir(parents=True, exist_ok=True)
    audit.main()


def run_bound(gt, output_root):
    import numpy as np
    import context_retention_bound as bound
    rr, cc = np.indices((13, 14))
    eligible_fixture = ((rr*7+cc*11) % 5) != 0
    owner_fixture = (rr+cc) % 3
    masks = [eligible_fixture & (owner_fixture == s) for s in range(3)]
    assert np.array_equal(bound.demands(masks, 11), bound.brute_demands(masks, 11))
    rr, cc = np.indices(gt.shape)
    result = {"scope": "fixed centers/folds/window; sole owner per raw spectrum; keep centers",
              "adapter": "same original solve, new explicit GT path wiring",
              "independent_fixture_checked": True,
              "variants": [bound.solve(gt > 0, (rr+cc) % m, groups)
                           for m, groups in [(3, [0, 1, 2]), (5, [0, 0, 0, 1, 2])]],
              "accuracy_measured": False, "universal_HSI_bound": False}
    write_report(Path(output_root)/"context_retention_bound.json", result)


def run_feasibility(gt, train, test, dev_train, dev_val, output_root):
    from impact_probe_feasibility import probe_geometry
    result = {"purpose": "geometry only; no classification or causal-effect estimate",
              "adapter": "same original probe_geometry, explicit B masks from verified literals",
              "fixed_final_B_test": probe_geometry(train, test, gt),
              "fixed_development_B_validation": probe_geometry(dev_train, dev_val, gt)}
    write_report(Path(output_root)/"impact_probe_feasibility.json", result)


def run_frozen_sensitivity(source_root, source_run, output_root, device):
    """One shared namespace, same payload hash and registered probe defaults."""
    names = ("impact_probe_core.py", "impact_probe_bootstrap_fragment.py", "impact_probe_main_fragment.py")
    payload = "\n\n".join((Path(source_root)/name).read_text(encoding="utf-8") for name in names)
    module = types.ModuleType("review_frozen_sensitivity")
    module.__file__ = str(Path(source_root)/"CELL_KIEM_TRA_DO_NHAY_B.py")
    module.__dict__.update(IMPACT_SOURCE_RUN_PATH=source_run, IMPACT_OUTPUT_ROOT=str(output_root),
        IMPACT_DEVICE=device, IMPACT_METHODS=("c_real_only_matched", "hypergamo"),
        IMPACT_MODEL_SEEDS=(42,), IMPACT_K=5, IMPACT_MASK_SEEDS=(173, 947), IMPACT_EVAL_BATCH=64,
        IMPACT_IMPLEMENTATION_SHA256=hashlib.sha256(payload.encode()).hexdigest())
    sys.modules[module.__name__] = module
    exec(compile(payload, module.__file__, "exec"), module.__dict__)
    return module.impact_run_probe()


## 03 — Prototype observation ownership M3/M5

M3 có41/121 và M5 có25/121 vị trí nội vùng mỗi observation. Zero raw-source intersection, đủ16 lớp ở3folds; các tâm vẫn sát nhau. Không gọi đây là geographic independence hoặc classifier improvement.

### Code — `spatial_leakage_core.py` · phần 1/1: Lattice, source_indices, partition_centers, fit_center_pca

**Mục đích:** Ownership lattice M3/M5, chia tâm và extractor chỉ đọc raw spectra cho phép.

**Đầu vào:** Cube, tọa độ/eligibility và lattice. **Đầu ra:** Observation, validity mask, source IDs và digest.

**Trạng thái:** CPU checks đã qua; chưa train classifier. **Giới hạn/cần review:** Source-disjoint không đồng nghĩa độc lập địa lý.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma spatial_leakage_core.py 1/1 0:1 0aa6c374aa8aa66b2208cb5c69678abd577379782c0c99260e94c1bfed09d297
"""CPU research prototype: exclusive raw observations on a fixed lattice.

This changes observation support, not merely the split of dense patches.
No classifier, no GPU, no inpainting, and no geographic-independence claim.
"""
from dataclasses import dataclass
import hashlib
import numpy as np


@dataclass(frozen=True)
class Lattice:
    modulus: int
    phase_to_split: tuple
    row_coefficient: int = 1
    col_coefficient: int = 1
    patch_size: int = 11

    def __post_init__(self):
        if self.modulus < 3 or len(self.phase_to_split) != self.modulus:
            raise ValueError("Invalid phase partition")
        if set(self.phase_to_split) != {0, 1, 2}:
            raise ValueError("Three nonempty split groups required")
        if self.patch_size < 1 or self.patch_size % 2 != 1:
            raise ValueError("Odd positive patch size required")
        if np.gcd(np.gcd(self.row_coefficient, self.col_coefficient), self.modulus) != 1:
            raise ValueError("Phase map must reach all residues")

    def phase_map(self, shape):
        rr, cc = np.indices(shape)
        return (self.row_coefficient * rr + self.col_coefficient * cc) % self.modulus

    def owner_map(self, shape):
        return np.asarray(self.phase_to_split, dtype=np.int8)[self.phase_map(shape)]

    def offset_mask(self):
        radius = self.patch_size // 2
        rr, cc = np.mgrid[-radius:radius + 1, -radius:radius + 1]
        return (self.row_coefficient * rr + self.col_coefficient * cc) % self.modulus == 0


def source_indices(shape, coordinates, lattice):
    """Flat raw-coordinate IDs, -1 for missing/constant padding.

    Coordinates only; no spectra or class labels are accepted here.
    Every sample uses its individual phase, including multi-phase train groups.
    """
    coordinates = np.asarray(coordinates, dtype=np.int64)
    if coordinates.ndim != 2 or coordinates.shape[1] != 2:
        raise ValueError("Coordinates must have shape N,2")
    if np.any(coordinates < 0) or np.any(coordinates >= np.array(shape)):
        raise ValueError("Center outside image")
    size = lattice.patch_size
    radius = size // 2
    dr, dc = np.mgrid[-radius:radius + 1, -radius:radius + 1]
    rr = coordinates[:, 0, None, None] + dr
    cc = coordinates[:, 1, None, None] + dc
    valid = ((rr >= 0) & (rr < shape[0]) & (cc >= 0) & (cc < shape[1])
             & lattice.offset_mask()[None])
    return np.where(valid, rr * shape[1] + cc, -1).astype(np.int32)


def partition_centers(labeled_eligibility, lattice):
    """Geometry sees labeled/background eligibility, never class IDs."""
    labeled = np.asarray(labeled_eligibility)
    if labeled.ndim != 2 or labeled.dtype != np.bool_:
        raise ValueError("A two-dimensional boolean eligibility mask is required")
    owner = lattice.owner_map(labeled.shape)
    return owner, [np.argwhere(labeled & (owner == s)) for s in range(3)]


def fit_center_pca(cube, train_coordinates, components=40):
    """Fit only labeled train centers; no spatial smoothing/full-scene fit."""
    centers = np.asarray(train_coordinates, dtype=np.int64)
    observed = np.asarray(cube[centers[:, 0], centers[:, 1]], dtype=np.float64)
    if not 1 <= components <= min(observed.shape):
        raise ValueError("Invalid PCA dimension")
    mean = observed.mean(axis=0)
    centered = observed - mean
    covariance = centered.T @ centered / max(1, len(observed) - 1)
    values, vectors = np.linalg.eigh(covariance)
    projection = vectors[:, np.argsort(values)[-components:][::-1]]
    # Deterministic sign convention for reproducible probes.
    pivots = np.argmax(np.abs(projection), axis=0)
    signs = np.sign(projection[pivots, np.arange(components)])
    projection = projection * np.where(signs == 0, 1, signs)
    return {"mean": mean, "projection": projection}


def extract_observations(cube, coordinates, lattice, preprocessing=None):
    """Gather only permitted spectra, then transform each spectrum locally.

    Missing entries are exact constants and accompanied by a validity mask.
    Applying a full-scene spatial filter before this function is forbidden.
    """
    if np.asarray(cube).ndim != 3:
        raise ValueError("Cube must have shape H,W,B")
    ids = source_indices(cube.shape[:2], coordinates, lattice)
    valid = ids >= 0
    gathered = np.asarray(cube).reshape(-1, cube.shape[2])[ids[valid]]
    if preprocessing is not None:
        gathered = (gathered - preprocessing["mean"]) @ preprocessing["projection"]
    output = np.zeros(ids.shape + (gathered.shape[1],), dtype=np.float32)
    output[valid] = gathered
    return output, valid, ids


def observation_digest(cube, coordinates, lattice, preprocessing=None, batch_size=96):
    """Streaming audit digest; never retains the whole research tensor cache."""
    digest = hashlib.sha256()
    for start in range(0, len(coordinates), batch_size):
        values, valid, ids = extract_observations(
            cube, coordinates[start:start + batch_size], lattice, preprocessing)
        for value in (values, valid, ids):
            digest.update(np.ascontiguousarray(value).tobytes())
    return digest.hexdigest()


def source_unions(shape, split_coordinates, lattice):
    unions = []
    for coordinates in split_coordinates:
        ids = source_indices(shape, coordinates, lattice)
        unions.append(np.unique(ids[ids >= 0]))
    return unions


def array_hash(array):
    a = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str(a.dtype).encode())
    digest.update(str(a.shape).encode())
    digest.update(a.tobytes())
    return digest.hexdigest()


### Code — `audit_real_data.py` · phần 1/2: sha, write_json, write_csv, brute_source_set

**Mục đích:** Oracle tọa độ độc lập, all16 class counts, biên ảnh và counterfactual spectra/nhãn.

**Đầu vào:** Hai MAT + comparison ZIP cũ. **Đầu ra:** cpu_certificate, geometry NPZ và probe PCA.

**Trạng thái:** CPU certificate thật đã có. **Giới hạn/cần review:** PCA prototype riêng; không dùng lại preprocessing B hoặc chấm classification.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma audit_real_data.py 1/2 0:3 70e99c7896cc27b167d437e02d8bae3c550973a2a29f03d89f0b221135b358ee
"""Audits actual Indian Pines input geometry and counterfactual extraction.

No model training/test scoring. Internal local script, not a Kaggle cell.
"""
from pathlib import Path
import csv
import hashlib
import io
import json
import zipfile
import numpy as np
from scipy.io import loadmat
from scipy.ndimage import maximum_filter
from scipy.spatial import cKDTree
from spatial_leakage_core import (
    Lattice, source_indices, fit_center_pca, extract_observations,
    observation_digest, source_unions, array_hash, partition_centers,
)

ROOT = Path(__file__).resolve().parent
SOURCE_ZIP = Path("C:/Users/ADMIN/OneDrive/Desktop/reproduct - 3dhyper/output/comparison_4methods_20fffe805617-ver2.zip")
MAT_ROOT = Path("C:/Users/ADMIN/OneDrive/Desktop/reproduct - 3dhyper/3D-HyperGAMO-main/3D-HyperGAMO-main/HSI-datasets/Classification")
HSI_PATH = MAT_ROOT / "indian_pines_corrected.mat"
GT_PATH = MAT_ROOT / "indian_pines_corrected_gt.mat"
OUT = ROOT / "cpu_certificate"

CLASS_NAMES = ["Alfalfa", "Corn-notill", "Corn-mintill", "Corn", "Grass-pasture",
               "Grass-trees", "Grass-pasture-mowed", "Hay-windrowed", "Oats",
               "Soybean-notill", "Soybean-mintill", "Soybean-clean", "Wheat",
               "Woods", "Buildings-Grass-Trees-Drives", "Stone-Steel-Towers"]


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_json(name, value):
    (OUT / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def write_csv(name, rows):
    with (OUT / name).open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def brute_source_set(shape, coordinates, lattice):
    # Independent oracle: enumerate physical coordinates and compare phases,
    # rather than reuse the relative offset test implemented by the extractor.
    result = set()
    phases = lattice.phase_map(shape)
    radius = lattice.patch_size // 2
    for row, col in coordinates:
        for rr in range(max(0, row - radius), min(shape[0], row + radius + 1)):
            for cc in range(max(0, col - radius), min(shape[1], col + radius + 1)):
                if phases[rr, cc] == phases[row, col]:
                    result.add(int(rr * shape[1] + cc))
    return np.array(sorted(result), dtype=np.int32)


def fixture_checks():
    lattice = Lattice(5, (0, 0, 0, 1, 2))
    shape = (13, 14)
    owner = lattice.owner_map(shape)
    coordinates = [np.argwhere(owner == s) for s in range(3)]
    for coords, actual in zip(coordinates, source_unions(shape, coordinates, lattice)):
        np.testing.assert_array_equal(actual, brute_source_set(shape, coords, lattice))
    # Negative control: reflection of a nominally permitted offset can move
    # it to another phase. Constant padding must not do that.
    bad_reflection_examples = []
    for rr, cc in np.argwhere(lattice.offset_mask()):
        dr, dc = rr - 5, cc - 5
        if dr < 0 or dc < 0:
            physical = (abs(dr), abs(dc))
            if lattice.phase_map(shape)[physical] != lattice.phase_map(shape)[0, 0]:
                bad_reflection_examples.append([int(dr), int(dc), *map(int, physical)])
    assert bad_reflection_examples
    ids = source_indices(shape, np.array([[0, 0]]), lattice)
    assert np.all(ids[0, :5, :] == -1) and np.all(ids[0, :, :5] == -1)
    return {"brute_physical_coordinate_oracle": True,
            "multi_phase_train_group_checked": True,
            "reflection_counterexample": bad_reflection_examples[0],
            "constant_padding_checked": True}


def main():
    OUT.mkdir(exist_ok=True)
    expected_hsi = "ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939"
    expected_gt = "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c"
    assert sha(HSI_PATH) == expected_hsi and sha(GT_PATH) == expected_gt
    gt_mat = loadmat(GT_PATH)
    gt = next(a for key, a in gt_mat.items() if not key.startswith("__") and a.shape == (145, 145))
    cube_mat = loadmat(HSI_PATH)
    cube = next(a for key, a in cube_mat.items() if not key.startswith("__") and a.shape == (145, 145, 200))
    cube = cube.astype(np.float64)
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        split_blob = archive.read("comparison_4methods_20fffe805617/split_indices.npz")
    with np.load(io.BytesIO(split_blob), allow_pickle=False) as data:
        np.testing.assert_array_equal(gt, data["standard_gt"])
    assert np.count_nonzero(gt) == 10249 and set(np.unique(gt)) == set(range(17))
    print("Verified original HSI/GT hashes and all 10249 labeled centers.", flush=True)

    fixture = fixture_checks()
    variants = [
        ("lattice_m3", Lattice(3, (0, 1, 2))),
        ("lattice_m5", Lattice(5, (0, 0, 0, 1, 2))),
    ]
    summaries, counts_rows, context_rows = [], [], []
    for name, lattice in variants:
        owner, coordinates = partition_centers(gt > 0, lattice)
        unions = source_unions(gt.shape, coordinates, lattice)
        shared = {f"{a}_{b}": int(len(np.intersect1d(unions[a], unions[b])))
                  for a, b in ((0, 1), (0, 2), (1, 2))}
        assert all(v == 0 for v in shared.values())
        all_coords = np.concatenate(coordinates)
        assert len(np.unique(all_coords, axis=0)) == np.count_nonzero(gt)
        dense_unions = [np.flatnonzero(maximum_filter(((gt > 0) & (owner == s)).astype(np.uint8),
                                                    size=11, mode="constant") > 0)
                        for s in range(3)]
        dense_shared = {f"{a}_{b}": int(len(np.intersect1d(dense_unions[a], dense_unions[b])))
                        for a, b in ((0, 1), (0, 2), (1, 2))}
        minimum_distances = {}
        for a, b in ((0, 1), (0, 2), (1, 2)):
            distance = cKDTree(coordinates[a]).query(coordinates[b], p=np.inf)[0]
            minimum_distances[f"{a}_{b}"] = float(distance.min())
        assert all(d == 1 for d in minimum_distances.values())

        for split_id, coords in enumerate(coordinates):
            ids = source_indices(gt.shape, coords, lattice)
            eligible = ids >= 0
            source_owners = owner.ravel()[ids[eligible]]
            assert np.all(source_owners == split_id)
            assert np.all(ids[:, 5, 5] == coords[:, 0] * gt.shape[1] + coords[:, 1])
            interior = np.all((coords >= 5) & (coords < np.array(gt.shape) - 5), axis=1)
            if np.any(interior):
                np.testing.assert_array_equal(eligible[interior],
                                              np.broadcast_to(lattice.offset_mask(), eligible[interior].shape))
            for class_id in range(1, 17):
                selected = gt[coords[:, 0], coords[:, 1]] == class_id
                assert np.any(selected)
                n = eligible[selected].sum(axis=(1, 2))
                row = {"variant": name, "split": split_id, "class_id": class_id,
                       "class_name": CLASS_NAMES[class_id - 1], "centers": int(selected.sum())}
                counts_rows.append(row)
                context_rows.append({**row, "min_observed": int(n.min()),
                                     "median_observed": float(np.median(n)),
                                     "max_observed": int(n.max())})

        preprocessing = fit_center_pca(cube, coordinates[0], 40)
        phase = lattice.phase_map(gt.shape)
        train_perturbed = cube.copy()
        train_perturbed[owner != 0] = -98765.4321
        changed_fit = fit_center_pca(train_perturbed, coordinates[0], 40)
        for key in preprocessing:
            np.testing.assert_array_equal(preprocessing[key], changed_fit[key])
        invariance = {}
        for split_id, coords in enumerate(coordinates):
            original_digest = observation_digest(cube, coords, lattice, preprocessing)
            changed = cube.copy()
            changed[owner != split_id] = -54321.1234 + split_id
            forbidden_digest = observation_digest(changed, coords, lattice, preprocessing)
            assert original_digest == forbidden_digest
            # Positive control: alter an allowed center and require a changed input.
            changed[tuple(coords[0])] = 45678.9
            positive_digest = observation_digest(changed, coords, lattice, preprocessing)
            assert original_digest != positive_digest
            invariance[str(split_id)] = {
                "forbidden_spectra_changes_invariant": True,
                "allowed_spectrum_change_detected": True,
                "original_digest": original_digest,
            }
        # Rebuild the geometry from changed evaluation class IDs, keeping only
        # labeled/background eligibility unchanged. Feed the rebuilt coordinates
        # through the actual extractor, rather than reuse the original coordinates.
        original_sources = [source_indices(gt.shape, coords, lattice) for coords in coordinates]
        altered_gt = gt.copy()
        changed_ids = (gt > 0) & (owner != 0)
        altered_gt[changed_ids] = 1 + gt[changed_ids] % 16
        assert not np.array_equal(altered_gt, gt)
        rebuilt_owner, rebuilt_coordinates = partition_centers(altered_gt > 0, lattice)
        np.testing.assert_array_equal(owner, rebuilt_owner)
        for split_id, (coords, rebuilt, sources) in enumerate(zip(
                coordinates, rebuilt_coordinates, original_sources)):
            np.testing.assert_array_equal(coords, rebuilt)
            np.testing.assert_array_equal(sources, source_indices(altered_gt.shape, rebuilt, lattice))
            assert observation_digest(cube, rebuilt, lattice, preprocessing) == invariance[str(split_id)]["original_digest"]
        # Eligibility is a separate benchmark input: changing it SHOULD change
        # the selected centers. We do not claim invariance to that operation.
        changed_eligibility = altered_gt > 0
        changed_eligibility[tuple(coordinates[1][0])] = False
        _, eligibility_coordinates = partition_centers(changed_eligibility, lattice)
        assert len(eligibility_coordinates[1]) == len(coordinates[1]) - 1

        np.savez_compressed(OUT / f"{name}_geometry.npz", owner=owner, phase=phase,
                            offset_mask=lattice.offset_mask(),
                            coords_train=coordinates[0], coords_validation=coordinates[1],
                            coords_test=coordinates[2],
                            source_union_train=unions[0], source_union_validation=unions[1],
                            source_union_test=unions[2])
        np.savez_compressed(OUT / f"{name}_probe_pca.npz", **preprocessing)
        summary = {
            "variant": name, "modulus": lattice.modulus,
            "phase_to_split": list(lattice.phase_to_split),
            "centers": [int(len(coords)) for coords in coordinates],
            "classes_in_every_split": 16,
            "minimum_class_count": min(row["centers"] for row in counts_rows if row["variant"] == name),
            "interior_observed_pixels": int(lattice.offset_mask().sum()),
            "original_patch_pixels": 121,
            "shared_raw_sources": shared,
            "dense_same_centers_shared_sources": dense_shared,
            "source_union_sizes": [int(len(u)) for u in unions],
            "minimum_cross_split_center_chebyshev": minimum_distances,
            "interior_masks_identical": True,
            "every_center_retained": True,
            "extractor_and_ownership_receive_no_class_ids": True,
            "heldout_class_id_changes_leave_geometry_and_inputs_unchanged": True,
            "eligibility_change_negative_control_passed": True,
            "train_center_PCA_invariant_to_heldout_spectra": True,
            "counterfactual_input_checks": invariance,
            "geometry_only_probe": True,
            "PCA_is_probe_not_reused_B_preprocessing": True,
            "geographic_independence": False,
            "classification_evaluated": False,
            "novelty_verified": False,
        }
        summaries.append(summary)
        print(json.dumps({k: summary[k] for k in ("variant", "centers", "interior_observed_pixels",
                                                 "shared_raw_sources", "minimum_class_count")}), flush=True)

    write_csv("class_counts.csv", counts_rows)
    write_csv("context_counts.csv", context_rows)
    certificate = {
        "status": "CPU construction checks passed; classification and novelty unverified",
        "source_hsi": str(HSI_PATH), "source_hsi_sha256": sha(HSI_PATH),
        "source_gt": str(GT_PATH), "source_gt_sha256": sha(GT_PATH),
        "gt_matches_previous_audit": True, "gt_array_sha256": array_hash(gt),
        "fixture_checks": fixture, "variants": summaries,
        "training_performed": False, "test_performance_evaluated": False,
        "unmodified_dense_patch_protocol": False,
        "implementation_sha256": {p.name: sha(p) for p in
                                  (ROOT / "spatial_leakage_core.py", ROOT / "audit_real_data.py")},
        "runtime": {"numpy": np.__version__},
        "artifacts": {name: sha(OUT / name) for name in
                      ("class_counts.csv", "context_counts.csv", "lattice_m3_geometry.npz",
                       "lattice_m5_geometry.npz", "lattice_m3_probe_pca.npz", "lattice_m5_probe_pca.npz")},
    }
    write_json("certificate.json", certificate)
    print("CPU certificate complete. No classifier training or test scoring.", flush=True)




### Code — `audit_real_data.py` · phần 2/2: Imports, cấu hình và script body

**Mục đích:** Oracle tọa độ độc lập, all16 class counts, biên ảnh và counterfactual spectra/nhãn.

**Đầu vào:** Hai MAT + comparison ZIP cũ. **Đầu ra:** cpu_certificate, geometry NPZ và probe PCA.

**Trạng thái:** CPU certificate thật đã có. **Giới hạn/cần review:** PCA prototype riêng; không dùng lại preprocessing B hoặc chấm classification.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma audit_real_data.py 2/2 0:1 9a6a7ffeed1af83ced837054949e0b2fdf0e568fe3d4c2e20c229ec0a9707cc3
if __name__ == "__main__":
    main()


## 04 — Cận giữ context của bài toán ownership cố định

Cận M3 40,6327 và construction40,4625 là số nguồn quan sát trung bình có xét biên. Cận giữ fixedfolds, fixedcenters, window11×11, singleowner và center retention; không tối ưu accuracy.

### Code — `context_retention_bound.py` · phần 1/1: demands, brute_demands, solve, main

**Mục đích:** Tính và đạt cận số raw sources khi cố định tâm/fold/cửa sổ và giữ tâm.

**Đầu vào:** GT eligibility, phase map và fold groups. **Đầu ra:** Cận ownership và attaining-map kiểm độc lập.

**Trạng thái:** Đã kiểm CPU + oracle. **Giới hạn/cần review:** Không là cận accuracy/HSI nói chung; đổi giả thuyết sẽ đổi bài toán.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma context_retention_bound.py 1/1 0:1 ff39a65ce3bca7a542690b22162578daa798b565406e3a6c353e01b84abe20b3
"""Exact geometry bound for fixed centers, fixed folds and direct 11x11 sources.

No HSI spectra, classifier, GPU or fold selection are involved.
Each raw-coordinate spectrum may be observed by only one split; same-split
reuse is permitted. Missing/outside-image positions do not count as sources.
"""
from pathlib import Path
import hashlib
import json

import numpy as np
from scipy.io import loadmat
from scipy.ndimage import convolve


def demands(center_masks, patch_size):
    kernel = np.ones((patch_size, patch_size), dtype=np.int32)
    return np.stack([
        convolve(mask.astype(np.int32), kernel, mode="constant", cval=0)
        for mask in center_masks
    ])


def brute_demands(center_masks, patch_size):
    result = np.zeros((len(center_masks), *center_masks[0].shape), dtype=np.int32)
    radius = patch_size // 2
    height, width = center_masks[0].shape
    for split, mask in enumerate(center_masks):
        for r, c in np.argwhere(mask):
            for qr in range(max(0, r-radius), min(height, r+radius+1)):
                for qc in range(max(0, c-radius), min(width, c+radius+1)):
                    result[split, qr, qc] += 1
    return result


def solve(eligible, phase, groups, patch_size=11):
    owner = np.asarray(groups)[phase]
    center_masks = [eligible & (owner == split) for split in range(3)]
    demand = demands(center_masks, patch_size)
    optimum_owner = demand.argmax(axis=0)
    # Keeping a labeled center as an input forces its spectrum to its own fold.
    optimum_owner[eligible] = owner[eligible]
    best_contribution = np.take_along_axis(demand, optimum_owner[None], axis=0)
    best_total = int(best_contribution.sum())
    # Independently count the total actually retained by this attaining map.
    retained_maps = demands([optimum_owner == s for s in range(3)], patch_size)
    actual_total = sum(int(retained_maps[s][center_masks[s]].sum()) for s in range(3))
    assert actual_total == best_total
    phase_demand = demands([eligible & (phase == p) for p in range(len(groups))], patch_size)
    current_total = int(np.take_along_axis(phase_demand, phase[None], axis=0).sum())
    count = int(eligible.sum())
    return {
        "centers": count,
        "phase_to_split": groups,
        "dense_mean_in_image_sources": float(demand.sum() / count),
        "current_fixed_phase_mean_sources": current_total / count,
        "relaxed_bound_mean_sources": float(demand.max(axis=0).sum() / count),
        "center_preserving_max_total_sources": best_total,
        "center_preserving_max_mean_sources": best_total / count,
        "attaining_map_total_checked": actual_total,
        "per_split_attaining_mean_sources": [
            float(retained_maps[s][center_masks[s]].mean()) for s in range(3)
        ],
        "mask_shape_consistency_required": False,
        "per_class_minimum_context_required": False,
    }


def main():
    root = Path(__file__).resolve().parent
    certificate = json.loads((root / "cpu_certificate/certificate.json").read_text(encoding="utf-8"))
    gt_path = Path(certificate["source_gt"])
    assert hashlib.sha256(gt_path.read_bytes()).hexdigest() == certificate["source_gt_sha256"]
    contents = loadmat(gt_path)
    gt = next(value for key, value in contents.items()
              if not key.startswith("__") and isinstance(value, np.ndarray)
              and value.shape == (145, 145))
    eligible = gt > 0
    assert eligible.sum() == 10249
    # An independent physical-coordinate loop verifies the convolution counts.
    rr, cc = np.indices((13, 14))
    fixture_eligible = ((rr * 7 + cc * 11) % 5) != 0
    fixture_owner = (rr + cc) % 3
    fixture_masks = [fixture_eligible & (fixture_owner == s) for s in range(3)]
    assert np.array_equal(demands(fixture_masks, 11), brute_demands(fixture_masks, 11))
    fixture_result = solve(fixture_eligible, fixture_owner, [0, 1, 2])
    rr, cc = np.indices(gt.shape)
    result = {
        "scope": "Fixed labeled centers/folds; direct in-image 11x11 coordinate sources; one owner per raw spectrum",
        "proof": "Let d_s(q) count split-s centers whose window contains q. A sole owner o(q) retains d_o(q). Thus sum_q max_s d_s(q) is an upper bound. Requiring every center spectrum fixes its owner; all remaining q independently choose the argmax, attaining the constrained maximum.",
        "gt_sha256": certificate["source_gt_sha256"],
        "independent_fixture_checked": True,
        "fixture_attaining_map_checked": fixture_result["attaining_map_total_checked"],
        "variants": [solve(eligible, (rr+cc) % m, groups)
                     for m, groups in [(3, [0, 1, 2]), (5, [0, 0, 0, 1, 2])]],
        "limits": "Not a universal HSI bound. Changing center folds, dropping centers, enlarging windows or predicting context changes the problem. No accuracy or information-quality claim; optimized masks need not match across samples/folds.",
    }
    destination = root / "context_retention_bound.json"
    destination.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()


## 05 — Geometry và frozen-model sensitivity của B

B final test4721; shared occurrence khoảng82,0243%. K5 shell matching còn2962mẫu/14lớp, thiếu9/16. Hai maskseeds173/947 và modelseed42 đã chạy; không chứng minh causal inflation. Core, bootstrap và main phải chạy chung namespace.

### Code — `impact_probe_feasibility.py` · phần 1/1: literal_assignment, probe_geometry, main

**Mục đích:** Khảo sát nguồn shared/nonshared và capacity control cùng Chebyshev shell.

**Đầu vào:** GT và masks B final/development. **Đầu ra:** Geometry gate trước inference.

**Trạng thái:** CPU geometry đã chạy. **Giới hạn/cần review:** Capacity không đủ cân bằng content/hướng; không đo accuracy.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma impact_probe_feasibility.py 1/1 0:1 3e13db21b44ed90250b14b7faf40e19b70a2cf1a2f1b61b4e6827154c8321a3a
"""CPU geometry gate for shared/nonshared perturbations of fixed B models.

Reconstruct the known B masks without executing notebook code. No spectra,
model, predictions, training or test classification metric are read/computed.
"""
from pathlib import Path
import ast
import base64
import hashlib
import io
import json
import zlib

import numpy as np
from scipy.io import loadmat
from scipy.ndimage import maximum_filter


ROOT = Path(__file__).resolve().parent
NOTEBOOK_SOURCE = ROOT.parent / "v2_code.py"


def literal_assignment(tree, name):
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(
            isinstance(target, ast.Name) and target.id == name for target in node.targets
        ):
            return ast.literal_eval(node.value)
    raise ValueError(name)


def probe_geometry(train, evaluation, gt, radius=5):
    height, width = gt.shape
    seen = maximum_filter(train.astype(np.uint8), size=2*radius+1,
                          mode="constant", cval=0).astype(bool)
    # Independent rectangular coordinate union, including clipped image edges.
    oracle = np.zeros_like(train)
    for r, c in np.argwhere(train):
        oracle[max(0,r-radius):min(height,r+radius+1),
               max(0,c-radius):min(width,c+radius+1)] = True
    assert np.array_equal(seen, oracle)
    coords = np.argwhere(evaluation)
    dr, dc = np.mgrid[-radius:radius+1, -radius:radius+1]
    rr, cc = coords[:,0,None,None]+dr, coords[:,1,None,None]+dc
    valid = (rr>=0) & (rr<height) & (cc>=0) & (cc<width)
    shared = valid & seen[np.clip(rr,0,height-1), np.clip(cc,0,width-1)]
    is_center = (dr==0) & (dc==0)
    nonshared_context = valid & ~shared & ~is_center
    shared_context = shared & ~is_center
    nonshared_count = nonshared_context.sum(axis=(1,2))
    shell = np.maximum(np.abs(dr), np.abs(dc))
    capacity = np.zeros(len(coords), dtype=np.int32)
    for distance in range(1,radius+1):
        a = (shared_context & (shell==distance)).sum(axis=(1,2))
        b = (nonshared_context & (shell==distance)).sum(axis=(1,2))
        capacity += np.minimum(a,b)
    labels = gt[coords[:,0],coords[:,1]]
    return {
        "train_centers": int(train.sum()),
        "evaluation_centers": int(evaluation.sum()),
        "unique_train_source_coordinates": int(seen.sum()),
        "evaluation_patches_sharing_any_actual_source": int(shared.any(axis=(1,2)).sum()),
        "fraction_actual_evaluation_source_occurrences_shared": float(shared.sum()/valid.sum()),
        "evaluation_centers_seen_as_train_context": int(shared[:,radius,radius].sum()),
        "patches_all_actual_sources_shared": int((shared | ~valid).all(axis=(1,2)).sum()),
        "nonshared_context_quantiles": np.quantile(nonshared_count,[0,.25,.5,.75,.9,.99,1]).tolist(),
        "nonshared_context_at_least_k": {str(k):int((nonshared_count>=k).sum())
                                        for k in [1,5,10,20,30]},
        "same_chebyshev_shell_pair_capacity_at_least_k": {
            str(k):int((capacity>=k).sum()) for k in [1,5,10,20,30]
        },
        "per_class": [{
            "class_id": cid,
            "evaluation_samples": int((labels==cid).sum()),
            "all_actual_sources_shared": int(((labels==cid) & (nonshared_count==0)).sum()),
            "pair_capacity_at_least_5": int(((labels==cid) & (capacity>=5)).sum()),
            "pair_capacity_at_least_30": int(((labels==cid) & (capacity>=30)).sum()),
        } for cid in range(1,17)],
        "independent_coordinate_union_checked": True,
        "center_excluded_from_both_perturbation_groups": True,
        "constant_padding_excluded_from_source_counts": True,
        "capacity_is_necessary_not_sufficient_for_full_geometry_matching": True,
    }


def main():
    certificate = json.loads((ROOT/"cpu_certificate/certificate.json").read_text(encoding="utf-8"))
    gt_path = Path(certificate["source_gt"])
    assert hashlib.sha256(gt_path.read_bytes()).hexdigest() == certificate["source_gt_sha256"]
    gt = loadmat(gt_path)["indian_pines_gt"]
    source_text = NOTEBOOK_SOURCE.read_text(encoding="utf-8")
    tree = ast.parse(source_text)
    config = literal_assignment(tree,"CAU_HINH")
    encoded = literal_assignment(tree,"DIP_B64")
    dip = loadmat(io.BytesIO(zlib.decompress(base64.b64decode(encoded))))["indianpines_disjoint_dset"]
    mapping = np.array([0,2,3,5,6,8,10,11,12,14,1,4,7,9,13,15,16])
    author = mapping[dip]
    train = (author>0) & (gt>0) & (author==gt)
    test = (gt>0) & ~train
    assert train.sum()==5528 and test.sum()==4721
    dev_train, dev_val = train.copy(), np.zeros_like(train)
    rng = np.random.default_rng(config["development_seed"])
    for cid in range(1,17):
        ids = np.flatnonzero((train & (gt==cid)).ravel())
        count = min(len(ids)-1,max(2,int(round(len(ids)*config["development_val_fraction"]))))
        chosen = rng.permutation(ids)[:count]
        rr, cc = np.unravel_index(chosen,gt.shape,order="C")
        dev_train[rr,cc], dev_val[rr,cc] = False, True
    assert dev_train.sum()==4422 and dev_val.sum()==1106
    result = {
        "purpose": "Feasibility of paired shared/nonshared input perturbations; no classification or causal effect measured",
        "source_gt_sha256": certificate["source_gt_sha256"],
        "notebook_source_sha256": hashlib.sha256(NOTEBOOK_SOURCE.read_bytes()).hexdigest(),
        "patch_size": config["patch_size"],
        "padding": "constant; outside-image positions are not raw sources",
        "source_definition": "A full raw-coordinate spectrum is seen if included in any real training-center patch",
        "fixed_final_B_test": probe_geometry(train,test,gt),
        "fixed_development_B_validation": probe_geometry(dev_train,dev_val,gt),
        "limits": "Geometry availability is not matched content, masking validity, model use, accuracy inflation, geographic independence or sample equivalence. Eligible subsets can be unrepresentative, especially of minority classes.",
    }
    (ROOT/"impact_probe_feasibility.json").write_text(json.dumps(result,indent=2),encoding="utf-8")
    print(json.dumps(result,indent=2))


if __name__=="__main__":
    main()


### Code — `impact_probe_core.py` · phần 1/2: Hình học nguồn và mask cặp

**Mục đích:** Đánh dấu nguồn từng patch test đã từng xuất hiện trong train; chọn 5 vị trí shared và 5 nonshared cùng histogram Chebyshev-shell, giữ tâm.

**Đầu vào:** coords_train, coords_test, ảnh 145x145. **Đầu ra:** Geometry, eligible indices, masks deterministic.

**Trạng thái:** Đã nhận kết quả seed42, 2.962 mẫu/14 lớp. **Giới hạn/cần review:** Matched shell không cân bằng mọi hướng/adjacency/texture; có ghi rõ hạn chế này?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma impact_probe_core.py 1/2 0:3 bbbc717b9f54a1cbb587fb0ee75d4dad3f48b1616909d185974b35943607e0c3
"""Geometry and scoring for a preregistered frozen-model sensitivity probe.

This measures perturbation sensitivity, not causal accuracy inflation.
"""
import hashlib
import numpy as np
from scipy.ndimage import maximum_filter


def impact_array_hash(array):
    array = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str(array.shape).encode())
    digest.update(array.dtype.str.encode())
    digest.update(array.tobytes())
    return digest.hexdigest()


def impact_geometry(image_shape, train_coords, test_coords, patch_size=11):
    train_coords, test_coords = np.asarray(train_coords), np.asarray(test_coords)
    for coords in (train_coords,test_coords):
        if coords.ndim!=2 or coords.shape[1]!=2 or not np.issubdtype(coords.dtype,np.integer):
            raise ValueError("Coordinates must be integer Nx2")
        if np.any(coords<0) or np.any(coords>=np.asarray(image_shape)):
            raise ValueError("Coordinate outside image")
    train = np.zeros(image_shape,dtype=bool)
    train[tuple(train_coords.T)] = True
    if train[tuple(test_coords.T)].any():
        raise ValueError("Training and evaluation centers must be different")
    if patch_size%2!=1:
        raise ValueError("Odd patch size required")
    radius = patch_size//2
    seen = maximum_filter(train.astype(np.uint8),size=patch_size,
                          mode="constant",cval=0).astype(bool)
    dr,dc = np.mgrid[-radius:radius+1,-radius:radius+1]
    rr,cc = test_coords[:,0,None,None]+dr,test_coords[:,1,None,None]+dc
    height,width = image_shape
    valid = (rr>=0)&(rr<height)&(cc>=0)&(cc<width)
    shared = valid & seen[np.clip(rr,0,height-1),np.clip(cc,0,width-1)]
    center = (dr==0)&(dc==0)
    shell = np.maximum(np.abs(dr),np.abs(dc))
    a,b = shared & ~center,valid & ~shared & ~center
    capacity = sum(np.minimum((a&(shell==r)).sum((1,2)),(b&(shell==r)).sum((1,2)))
                   for r in range(1,radius+1))
    return {"valid":valid,"shared":shared,"shared_context":a,"nonshared_context":b,
            "shell":shell,"capacity":capacity,"dr":dr,"dc":dc,
            "shared_fraction":shared.sum((1,2))/valid.sum((1,2)),
            "center_shared":shared[:,radius,radius],
            "all_sources_shared":(shared|~valid).all((1,2))}


def impact_paired_masks(geometry, test_coords, k=5, mask_seeds=(173,947)):
    if k<1 or not isinstance(k,int) or not mask_seeds or len(set(mask_seeds))!=len(mask_seeds):
        raise ValueError("Positive integer k and distinct mask seeds required")
    eligible = np.flatnonzero(geometry["capacity"]>=k)
    if len(eligible)==0:
        raise ValueError("No samples have enough matched shared/nonshared positions")
    size = geometry["shell"].shape[0]
    shell_flat = geometry["shell"].ravel()
    masks = np.zeros((len(mask_seeds),2,len(eligible),size,size),dtype=bool)
    for rep,seed in enumerate(mask_seeds):
        for j,original in enumerate(eligible):
            r,c = map(int,test_coords[original])
            rng = np.random.default_rng(np.random.SeedSequence([int(seed),r,c]))
            shared = geometry["shared_context"][original].ravel()
            nonshared = geometry["nonshared_context"][original].ravel()
            pools = []
            tickets = []
            for distance in range(1,size//2+1):
                aa = np.flatnonzero(shared&(shell_flat==distance))
                bb = np.flatnonzero(nonshared&(shell_flat==distance))
                pools.append((aa,bb))
                tickets.extend([distance]*min(len(aa),len(bb)))
            chosen = rng.choice(np.asarray(tickets),size=k,replace=False)
            for distance,(aa,bb) in enumerate(pools,start=1):
                count = int((chosen==distance).sum())
                if count:
                    masks[rep,0,j].flat[rng.choice(aa,size=count,replace=False)] = True
                    masks[rep,1,j].flat[rng.choice(bb,size=count,replace=False)] = True
    impact_check_masks(geometry,eligible,masks,k)
    return eligible,masks


def impact_check_masks(geometry,eligible,masks,k):
    size = geometry["shell"].shape[0]
    if masks.dtype!=np.bool_ or masks.shape[1:]!=(2,len(eligible),size,size):
        raise ValueError("Invalid mask shape/dtype")
    if not np.all(masks.sum(axis=(-2,-1))==k) or masks[...,size//2,size//2].any():
        raise ValueError("Mask budget or preserved center violated")
    for arm,key in enumerate(("shared_context","nonshared_context")):
        if np.any(masks[:,arm] & ~geometry[key][eligible][None]):
            raise ValueError("Mask selects padding/center or wrong source category")
    for distance in range(1,size//2+1):
        hist = (masks & (geometry["shell"]==distance)).sum(axis=(-2,-1))
        if not np.array_equal(hist[:,0],hist[:,1]):
            raise ValueError("Paired masks have unequal radial histograms")




### Code — `impact_probe_core.py` · phần 2/2: NLL, OA và phân tích theo lớp

**Mục đích:** Tính NLL từng mẫu và độ đo trên các lớp thực sự có mẫu; xuất descriptors và association.

**Đầu vào:** Nhãn và xác suất đủ 16 output. **Đầu ra:** Metrics/per-sample/association rows.

**Trạng thái:** Đã nhận kết quả seed42, 2.962 mẫu/14 lớp. **Giới hạn/cần review:** Mean NLL có bị vài mẫu cực trị hoặc một lớp chi phối?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma impact_probe_core.py 2/2 0:1 3327772793e6698298bd23704309364b897dae8e1757b2fa0b27e1476771dec3
def impact_per_sample(y,probabilities):
    y,probabilities = np.asarray(y),np.asarray(probabilities,dtype=np.float64)
    if probabilities.shape!=(len(y),16) or not np.isfinite(probabilities).all():
        raise ValueError("Finite Nx16 probabilities required")
    if (probabilities<0).any() or not np.allclose(probabilities.sum(axis=1),1,atol=1e-4):
        raise ValueError("Invalid probability distribution")
    true_probability = probabilities[np.arange(len(y)),y]
    prediction = probabilities.argmax(axis=1)
    return {"prediction":prediction,"true_probability":true_probability,
            "nll":-np.log(np.clip(true_probability,1e-12,1)),"correct":prediction==y}


def impact_metrics(y,probabilities):
    y = np.asarray(y,dtype=np.int64)
    values = impact_per_sample(y,probabilities)
    confusion = np.bincount(y*16+values["prediction"],minlength=256).reshape(16,16)
    support = confusion.sum(axis=1)
    predicted = confusion.sum(axis=0)
    true_positive = np.diag(confusion)
    present = np.flatnonzero(support)
    recall = np.divide(true_positive,support,out=np.zeros(16,dtype=float),where=support>0)
    f1 = np.divide(2*true_positive,support+predicted,out=np.zeros(16,dtype=float),where=(support+predicted)>0)
    return {"samples":len(y),"classes_present":len(present),
            "class_ids_present":(present+1).tolist(),"OA":float(values["correct"].mean()),
            "AA_present_classes":float(recall[present].mean()),
            "Macro_F1_present_classes":float(f1[present].mean()),
            "mean_NLL":float(values["nll"].mean()),
            "balanced_mean_NLL_present_classes":float(np.mean([values["nll"][y==cid].mean() for cid in present]))}


def impact_geometry_descriptors(mask,geometry):
    # Exact shell equality does not imply equal angles, adjacency or content.
    n = mask.sum(axis=(-2,-1))
    dr,dc = geometry["dr"],geometry["dc"]
    return {"mean_dr":(mask*dr).sum((-2,-1))/n,
            "mean_dc":(mask*dc).sum((-2,-1))/n,
            "mean_squared_euclidean_radius":(mask*(dr*dr+dc*dc)).sum((-2,-1))/n,
            "adjacent_4_edges":(mask[:,:-1,:]&mask[:,1:,:]).sum((1,2))+
                               (mask[:,:,:-1]&mask[:,:,1:]).sum((1,2))}


def impact_association_rows(y,probabilities,geometry):
    values = impact_per_sample(y,probabilities)
    edges = (0,.25,.5,.75,1.0000001)
    rows = []
    for cid in range(16):
        for lo,hi in zip(edges[:-1],edges[1:]):
            indices = (y==cid)&(geometry["shared_fraction"]>=lo)&(geometry["shared_fraction"]<hi)
            if indices.any():
                rows.append({"class_id":cid+1,"shared_fraction_lo":lo,"shared_fraction_hi":min(hi,1),
                             "samples":int(indices.sum()),"OA":float(values["correct"][indices].mean()),
                             "mean_NLL":float(values["nll"][indices].mean()),
                             "mean_shared_fraction":float(geometry["shared_fraction"][indices].mean())})
    return rows


### Code — `impact_probe_bootstrap_fragment.py` · phần 1/4: Nguồn B và chữ ký checkpoint

**Mục đích:** Khóa run B đã test xong; kiểm SHA của model/cache trước khi đọc. Không train.

**Đầu vào:** Thư mục Output B, method, seed. **Đầu ra:** Hàm hash, hằng số run, kiểm done/cache.

**Trạng thái:** Đã nhận kết quả seed42, 2.962 mẫu/14 lớp. **Giới hạn/cần review:** Có đủ artifact được pin, hay đang dùng nhầm Output chỉ có báo cáo?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma impact_probe_bootstrap_fragment.py 1/4 0:1 f9d47782449c1f155cc21b8fb273a3fecb4e1cdf84f3c1c730abb75434dd3fde
"""Definitions for an inference-only cell; no import, IO or TensorFlow operation here.

The parent cell calls impact_load_source first and impact_load_classifier only
after its experiment specification has been fixed. This never fits preprocessing
or a model, never opens an old notebook, and never downloads a dataset.
"""


def _impact_constants():
    return {
        "run_id": "50e988fae260",
        "implementation": "5e15f8214bf273bb889187b96c80407a8000e5e1a3ef30734bcfa86ced275a6f",
        "train_counts": [21,753,426,138,209,376,16,228,10,469,1390,311,125,720,287,49],
        "test_counts": [25,675,404,99,274,354,12,250,10,503,1065,282,80,545,99,44],
        # Final markers already observed when the original B test was frozen.
        "model_done_sha256": {
            "c_real_only_matched/42": "d6cf124b1fbf6b23d2ddf66332ccba30f7af35556bd0354e26501bcd60ecf148",
            "c_real_only_matched/43": "c007c942f8dd4ceb27fb899ba99434fd6afd472067ee6e29f4859fbb4d714513",
            "c_real_only_matched/44": "5ff7a40de110972e8a4c066b8be3e6530073e6ab94691c04d53bfbd3ac7a4718",
            "hypergamo/42": "4c97642445aba40746ac1340b90386912ebb9a5561f60c368adcd42ea17e74d0",
            "hypergamo/43": "6ac0270062cd3efdc5398d4e0fdf844255c30c4996eeb717ef8fbdee6424b562",
            "hypergamo/44": "3c2247144a88f6745a73507a791ca6ff773dc48ddfe440b0897e1ae14de9e361",
        },
    }


def _impact_sha_file(path):
    import hashlib
    from pathlib import Path
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def _impact_sha_array(value):
    import hashlib
    import numpy as np
    array = np.ascontiguousarray(value)
    digest = hashlib.sha256()
    digest.update(str(array.shape).encode())
    digest.update(array.dtype.str.encode())
    digest.update(array.tobytes())
    return digest.hexdigest()


def _impact_json(path):
    import json
    from pathlib import Path
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _impact_checked_done(folder, method, seed, required, pin_model=False):
    from pathlib import Path
    constants = _impact_constants()
    folder = Path(folder)
    marker = folder / "done.json"
    document = _impact_json(marker)
    if (document.get("complete") is not True
            or document.get("run_id") != constants["run_id"]
            or document.get("implementation") != constants["implementation"]
            or document.get("method") != method
            or document.get("seed") != int(seed)):
        raise RuntimeError(f"Done marker chưa hoàn tất hoặc sai baseline: {marker}")
    actual_marker_hash = _impact_sha_file(marker)
    if pin_model:
        expected = constants["model_done_sha256"].get(f"{method}/{int(seed)}")
        if expected is None or expected != actual_marker_hash:
            raise RuntimeError(f"Model không phải checkpoint final B đã đóng băng: {marker}")
    hashes = document.get("artifact_hashes", {})
    verified = {"done.json": actual_marker_hash}
    for relative in required:
        path = folder / relative
        expected = hashes.get(relative)
        if (not path.is_file() or not isinstance(expected, str) or len(expected) != 64
                or _impact_sha_file(path) != expected):
            raise RuntimeError(f"Artifact thiếu hoặc không khớp hash: {path}")
        verified[relative] = expected
    return document, verified


def _impact_verified_cache(path, expected_contract):
    from pathlib import Path
    path = Path(path)
    sidecar = path.with_suffix(path.suffix + ".json")
    document = _impact_json(sidecar)
    if (document.get("contract") != expected_contract
            or document.get("sha256") != _impact_sha_file(path)):
        raise RuntimeError(f"Cache/contract/hash không khớp: {path}")
    return {"sha256": document["sha256"], "sidecar_sha256": _impact_sha_file(sidecar)}


### Code — `impact_probe_bootstrap_fragment.py` · phần 2/4: Tìm và kiểm dữ liệu B

**Mục đích:** Đọc contract, tọa độ, scaler/PCA và dự đoán test đã lưu; chỉ nguồn đúng mới được nạp.

**Đầu vào:** Input đã lưu KLTN_HyperGAMO_B_50e988fae260. **Đầu ra:** Nguồn đã qua kiểm checksum.

**Trạng thái:** Đã nhận kết quả seed42, 2.962 mẫu/14 lớp. **Giới hạn/cần review:** Class order, tọa độ, nhãn và model có cùng run không?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma impact_probe_bootstrap_fragment.py 2/4 2:3 7ad33e18a25e6d10f20e91e1b825bc18c69c1c6e51c0ac6af4c22a4ff9791049


def _impact_validate_source(path, methods, seeds, verify_bytes=True):
    import hashlib
    import json
    from pathlib import Path
    import numpy as np
    path = Path(path)
    constants = _impact_constants()
    spec = _impact_json(path / "protocol_spec.json")
    identity = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:12]
    config = spec.get("scientific_config", {})
    if (identity != constants["run_id"]
            or spec.get("implementation_sha256") != constants["implementation"]
            or config.get("seeds") != [42,43,44] or config.get("so_lop") != 16
            or spec.get("patch_size") != 11 or spec.get("pca_components") != 40
            or spec.get("b_train_counts") != constants["train_counts"]
            or spec.get("b_test_counts") != constants["test_counts"]):
        raise RuntimeError(f"Nguồn không đúng protocol B 50e988fae260: {path}")
    manifest = _impact_json(path / "manifest.json")
    spec_hash = _impact_sha_file(path / "protocol_spec.json")
    if (manifest.get("run_id") != constants["run_id"]
            or manifest.get("protocol_spec_sha256") != spec_hash):
        raise RuntimeError(f"Manifest/protocol không khớp: {path}")
    required_data = ["B_master_split.npz", "B_master_split.npz.json",
                     "final_data/test_4721_LOCKED.npz", "final_data/test_4721_LOCKED.npz.json",
                     "final_data/preprocessing_final.joblib", "final_data/preprocessing_final.joblib.json",
                     "final_test_freeze.json"]
    missing = [relative for relative in required_data if not (path / relative).is_file()]
    if missing:
        raise RuntimeError("Nguồn thiếu final test cache hoặc provenance: " + ", ".join(missing)
                           + ". Thêm Notebook Outputs của version đã test xong vào Kaggle Input; "
                           "không chạy Run All và không train lại để sửa lỗi này.")
    freeze = _impact_json(path / "final_test_freeze.json")
    if (freeze.get("run_id") != constants["run_id"]
            or freeze.get("implementation") != constants["implementation"]):
        raise RuntimeError(f"Final test freeze không khớp: {path}")
    registry = {item["method"]: item for item in freeze.get("registry", [])}
    expected_order = (np.argsort(np.asarray(constants["train_counts"]), kind="mergesort") + 1).tolist()
    fingerprint = {"protocol_spec_sha256": spec_hash,
                   "final_test_freeze_sha256": _impact_sha_file(path / "final_test_freeze.json"),
                   "models": {}, "data": {}}
    for relative in ["B_master_split.npz", "final_data/test_4721_LOCKED.npz",
                     "final_data/preprocessing_final.joblib"]:
        sidecar = path / (relative + ".json")
        meta = _impact_json(sidecar)
        expected_hash = meta.get("sha256")
        contract = meta.get("contract", {})
        if (not isinstance(expected_hash, str) or len(expected_hash) != 64
                or contract.get("run_id") != constants["run_id"]
                or contract.get("implementation") != constants["implementation"]):
            raise RuntimeError(f"Provenance sai baseline: {sidecar}")
        if verify_bytes and _impact_sha_file(path / relative) != expected_hash:
            raise RuntimeError(f"Nội dung đã đổi: {path / relative}")
        fingerprint["data"][relative] = expected_hash
    if manifest.get("master_split_sha256") != fingerprint["data"]["B_master_split.npz"]:
        raise RuntimeError("Manifest/master split hash không khớp.")
    for method in methods:
        expected_file = {"c_real_only_matched": "classifier.weights.h5", "hypergamo": "C.weights.h5"}.get(method)
        if expected_file is None:
            raise ValueError(f"Method chưa được đặc tả trong phép khảo sát: {method}")
        item = registry.get(method, {})
        if (item.get("weights_file") != expected_file
                or item.get("class_order_file") != "class_order.json"):
            raise RuntimeError(f"Method không nằm đúng final registry: {method}")
        for seed in seeds:
            seed = int(seed)
            key = f"{method}/{seed}"
            folder = path / "models" / method / f"seed_{seed}"
            # Model bytes are small; always verify them, also during source discovery.
            _, verified = _impact_checked_done(folder, method, seed,
                                               [expected_file, "class_order.json"], pin_model=True)
            if freeze.get("models", {}).get(key) != verified["done.json"]:
                raise RuntimeError(f"Model không khớp final test freeze: {folder}")
            if _impact_json(folder / "class_order.json").get("original_class_ids") != expected_order:
                raise RuntimeError(f"Class order không đúng baseline: {folder}")
            fingerprint["models"][key] = verified
    return spec, fingerprint


def _impact_find_source(explicit_path, methods, seeds):
    from pathlib import Path
    constants = _impact_constants()
    if explicit_path:
        path = Path(explicit_path)
        _impact_validate_source(path, methods, seeds, verify_bytes=False)
        return path
    name = "KLTN_HyperGAMO_B_" + constants["run_id"]
    working = Path("/kaggle/working") / name
    candidates = {working} if working.is_dir() else set()
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        candidates.update(path.parent for path in input_root.rglob("protocol_spec.json")
                          if path.parent.name == name)
    accepted, rejected = [], []
    for path in sorted(candidates):
        try:
            _, fingerprint = _impact_validate_source(path, methods, seeds, verify_bytes=True)
            accepted.append((path, fingerprint))
        except (OSError, ValueError, KeyError, TypeError, RuntimeError) as error:
            rejected.append(f"{path}: {error}")
    if not accepted:
        raise RuntimeError("Không tìm thấy đủ final Output B 50e988fae260. Trong Kaggle, "
                           "Add Input > Notebook Outputs > minhlongg/notebook708cde343b, "
                           "chọn version đã test xong và có final_data cùng models. "
                           "Hoặc điền IMPACT_SOURCE_RUN_PATH trỏ đúng thư mục run. Không cần train lại.\n"
                           + "\n".join(rejected))
    if any(fingerprint != accepted[0][1] for _, fingerprint in accepted[1:]):
        raise RuntimeError("Có nhiều Output khác nội dung. Điền IMPACT_SOURCE_RUN_PATH để chọn nguồn: "
                           + ", ".join(str(path) for path, _ in accepted))
    # Prefer a completed reference-prediction bundle among byte-identical models/caches.
    def reference_count(path):
        return sum((path / "test_results" / method / f"seed_{seed}" / "predictions.npz").is_file()
                   for method in methods for seed in seeds)
    accepted.sort(key=lambda item: (-reference_count(item[0]), item[0] != working, str(item[0])))
    return accepted[0][0]




### Code — `impact_probe_bootstrap_fragment.py` · phần 3/4: Khôi phục patch và mean-fill B

**Mục đích:** Tái tạo X_test bằng preprocessing đã fit; không fit lại PCA/scaler.

**Đầu vào:** Nguồn B đã xác minh. **Đầu ra:** X_test, y_test, coords, fill_vector, reference_probabilities.

**Trạng thái:** Đã nhận kết quả seed42, 2.962 mẫu/14 lớp. **Giới hạn/cần review:** Mean-fill có dùng đúng mean train, scale cuối và 40 PCA components?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma impact_probe_bootstrap_fragment.py 3/4 0:1 718f2330ff4ed0eb910b790d023a25a5e8dcdb26b40f5ca395232bb3c6372bdb
def impact_load_source(explicit_path=None, methods=("c_real_only_matched", "hypergamo"), seeds=(42,)):
    from pathlib import Path
    import numpy as np
    import joblib
    constants = _impact_constants()
    methods, seeds = tuple(methods), tuple(int(seed) for seed in seeds)
    if not methods or not seeds or len(set(methods)) != len(methods) or len(set(seeds)) != len(seeds):
        raise ValueError("Danh sách methods/seeds phải không rỗng, không trùng.")
    source = _impact_find_source(explicit_path, methods, seeds)
    spec, fingerprint = _impact_validate_source(source, methods, seeds, verify_bytes=True)
    common = {"run_id": constants["run_id"], "implementation": constants["implementation"]}
    split_file = source / "B_master_split.npz"
    _impact_verified_cache(split_file, dict(common, kind="master_split"))
    with np.load(split_file, allow_pickle=False) as bundle:
        gt = bundle["standard_gt"].copy()
        coords_train, coords_test = bundle["coords_train"].copy(), bundle["coords_test"].copy()
        split, class_ids = bundle["split"].copy(), bundle["class_ids"].copy()
    if (gt.shape != (145,145) or split.shape != gt.shape or not np.issubdtype(gt.dtype, np.integer)
            or coords_train.shape != (5528,2) or coords_test.shape != (4721,2)):
        raise RuntimeError("Master geometry sai shape/type.")
    np.testing.assert_array_equal(class_ids, np.arange(1,17))
    for coords, expected_split in [(coords_train,0), (coords_test,2)]:
        if (not np.issubdtype(coords.dtype, np.integer) or np.any(coords < 0)
                or np.any(coords >= 145) or len(np.unique(coords,axis=0)) != len(coords)):
            raise RuntimeError("Tọa độ cache sai hoặc trùng.")
        np.testing.assert_array_equal(coords, np.argwhere(split == expected_split))
        labels = gt[coords[:,0],coords[:,1]]
        if np.any((labels < 1) | (labels > 16)):
            raise RuntimeError("Nhãn tâm không thuộc 16 lớp standard GT.")
    np.testing.assert_array_equal(np.bincount(gt[coords_train[:,0],coords_train[:,1]],minlength=17)[1:], constants["train_counts"])
    np.testing.assert_array_equal(np.bincount(gt[coords_test[:,0],coords_test[:,1]],minlength=17)[1:], constants["test_counts"])
    prep_file = source / "final_data" / "preprocessing_final.joblib"
    prep_hash = fingerprint["data"]["final_data/preprocessing_final.joblib"]
    expected_prep = dict(common, fit_coords_sha256=_impact_sha_array(coords_train),
                         pca_components=40, pca_solver=spec["scientific_config"]["pca_solver"])
    _impact_verified_cache(prep_file, expected_prep)
    prep = joblib.load(prep_file)
    if (prep.get("fit_count") != 5528 or prep.get("fit_coords_sha256") != _impact_sha_array(coords_train)
            or getattr(prep["pca"], "n_samples_", None) != 5528
            or tuple(prep["pca"].components_.shape) != (40,200)):
        raise RuntimeError("Preprocessing không fit trên đúng 5528 tâm train.")
    fill_vector = np.asarray(prep["scaler"].transform(np.zeros((1,40),dtype=np.float64))[0],dtype=np.float32)
    if fill_vector.shape != (40,) or not np.isfinite(fill_vector).all():
        raise RuntimeError("Fill vector không hợp lệ.")
    test_file = source / "final_data" / "test_4721_LOCKED.npz"
    expected_test = dict(common, kind="locked_test", coords=_impact_sha_array(coords_test),
                         preprocessing_sha256=prep_hash)
    _impact_verified_cache(test_file, expected_test)
    with np.load(test_file, allow_pickle=False) as bundle:
        X_test, y_test = bundle["X_test"].copy(), bundle["y_test"].copy()
        np.testing.assert_array_equal(bundle["coords_test"],coords_test)
    if X_test.shape != (4721,11,11,40,1) or X_test.dtype != np.dtype("float32") or not np.isfinite(X_test).all():
        raise RuntimeError("Test cache phải là float32 [4721,11,11,40,1], không NaN/Inf.")
    if y_test.shape != (4721,) or not np.issubdtype(y_test.dtype,np.integer):
        raise RuntimeError("Nhãn test cache sai shape/type.")
    np.testing.assert_array_equal(y_test,gt[coords_test[:,0],coords_test[:,1]]-1)
    references, reference_metrics = {}, {}
    fingerprint["reference_predictions"] = {}
    for method in methods:
        for seed in seeds:
            folder = source / "test_results" / method / f"seed_{seed}"
            prediction_file = folder / "predictions.npz"
            if not prediction_file.is_file():
                fingerprint["reference_predictions"][f"{method}/{seed}"] = {"available": False}
                continue
            _, verified = _impact_checked_done(folder,method,seed,
                                               ["predictions.npz","metrics.json","per_class.csv"])
            with np.load(prediction_file,allow_pickle=False) as bundle:
                np.testing.assert_array_equal(bundle["coords"],coords_test)
                np.testing.assert_array_equal(bundle["y_true"],y_test)
                probabilities = bundle["probabilities"].copy()
                prediction = bundle["y_pred"].copy()
            if (probabilities.shape != (4721,16) or not np.isfinite(probabilities).all()
                    or np.any(probabilities < 0) or np.any(probabilities > 1)):
                raise RuntimeError(f"Reference probabilities sai: {prediction_file}")
            np.testing.assert_allclose(probabilities.sum(axis=1),1.0,rtol=1e-5,atol=1e-5)
            np.testing.assert_array_equal(probabilities.argmax(axis=1),prediction)
            references[(method,seed)] = probabilities.astype(np.float32,copy=False)
            reference_metrics[(method,seed)] = _impact_json(folder / "metrics.json")
            fingerprint["reference_predictions"][f"{method}/{seed}"] = dict(available=True,**verified)
    return {"source": Path(source), "spec": spec, "X_test": X_test,
            "y_test": y_test.astype(np.int64,copy=False), "coords_test": coords_test,
            "coords_train": coords_train, "gt": gt, "fingerprint": fingerprint,
            "fill_vector": fill_vector, "fill_rule": "train_mean_spectrum_transformed_by_saved_final_PCA_scaler",
            "reference_probabilities": references, "reference_metrics": reference_metrics}


### Code — `impact_probe_bootstrap_fragment.py` · phần 4/4: TensorFlow/Keras và weights classifier

**Mục đích:** Khởi tạo runtime khi cần, dựng classifier và restore layers từ checkpoint B.

**Đầu vào:** TensorFlow/tf_keras tương thích, checkpoint C. **Đầu ra:** Classifier chỉ inference.

**Trạng thái:** Đã nhận kết quả seed42, 2.962 mẫu/14 lớp. **Giới hạn/cần review:** Memory growth các GPU có cùng chính sách? Inference dùng training=False?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma impact_probe_bootstrap_fragment.py 4/4 2:1 40d8f0a49f786208fa851b01517420d01cc011f4c83df6ea62e84fa211efa7c5


def impact_keras():
    """Lazy runtime setup. Default inference device is CPU; GPUs use equal policy."""
    if hasattr(impact_keras,"_cached"):
        return impact_keras._cached
    import tensorflow as tf
    devices = tf.config.get_visible_devices("GPU")
    for device in devices:
        try:
            tf.config.experimental.set_memory_growth(device,True)
        except (RuntimeError,ValueError) as error:
            flags = [tf.config.experimental.get_memory_growth(gpu) for gpu in devices]
            if len(set(flags)) > 1:
                raise RuntimeError("GPU memory growth khác nhau trong kernel hiện tại. "
                                   "Restart session rồi chạy riêng cell khảo sát; không train lại.") from error
            break
    flags = [tf.config.experimental.get_memory_growth(gpu) for gpu in devices]
    if len(set(flags)) > 1:
        raise RuntimeError("GPU memory growth chưa đồng nhất; restart session rồi chạy riêng cell.")
    try:
        import tf_keras as keras
    except ImportError:
        keras = tf.keras
    keras.mixed_precision.set_global_policy("float32")
    keras.backend.set_image_data_format("channels_last")
    impact_keras._cached = (tf,keras,{gpu.name:flag for gpu,flag in zip(devices,flags)})
    return impact_keras._cached


def _impact_h5_weight_groups(path):
    import h5py
    import numpy as np
    groups = []
    with h5py.File(path,"r") as handle:
        root = handle["model_weights"] if "model_weights" in handle else handle
        if "layer_names" in root.attrs:
            for name in root.attrs["layer_names"]:
                name = name.decode("utf-8") if isinstance(name,bytes) else str(name)
                layer = root[name]
                names = layer.attrs.get("weight_names",[])
                arrays = [np.asarray(layer[item.decode("utf-8") if isinstance(item,bytes) else str(item)]) for item in names]
                if arrays:
                    groups.append((layer.name,arrays))
        else:
            def collect(name,item):
                if not isinstance(item,h5py.Group) or name.rsplit("/",1)[-1] != "vars":
                    return
                keys = list(item.keys())
                if not keys:
                    return
                if (not all(key.isdecimal() and isinstance(item[key],h5py.Dataset) for key in keys)
                        or sorted(map(int,keys)) != list(range(len(keys)))):
                    raise RuntimeError(f"HDF5 vars không đúng weights format: {item.name}")
                groups.append((item.name,[np.asarray(item[key]) for key in sorted(keys,key=int)]))
            # Keras 3 can also save optimizer/vars. Only model layer variables
            # are restored for inference; the classifier has no own vars.
            layer_root = root["layers"] if "layers" in root else root
            layer_root.visititems(collect)
    if not groups or any(not np.isfinite(array).all() for _,arrays in groups for array in arrays):
        raise RuntimeError(f"Weights rỗng hoặc NaN/Inf: {path}")
    return groups


def _impact_restore_weight_layers(model,path):
    groups = _impact_h5_weight_groups(path)
    consumed = set()
    for layer in model.layers:
        target = layer.get_weights()
        if not target:
            continue
        signature = [tuple(array.shape) for array in target]
        matches = [index for index,(_,arrays) in enumerate(groups)
                   if index not in consumed and [tuple(array.shape) for array in arrays] == signature]
        if len(matches) != 1:
            raise RuntimeError(f"Không ánh xạ weights duy nhất cho {layer.name}: {signature}; {path}")
        selected = matches[0]
        layer.set_weights(groups[selected][1])
        consumed.add(selected)
    if len(consumed) != len(groups):
        raise RuntimeError(f"Có weights không được khôi phục: {path}")


def _impact_build_classifier(keras):
    L = keras.layers
    inp = L.Input(shape=(11,11,40,1))
    x = L.Conv3D(8,(1,1,7),activation="relu",padding="same")(inp)
    x = L.Conv3D(16,(3,3,5),activation="relu",padding="same")(x)
    x = L.Conv3D(32,(5,5,7),activation="relu",padding="same")(x)
    shape = x.shape
    x = L.Reshape((shape[1],shape[2],shape[3]*shape[4]))(x)
    x = L.Conv2D(64,(3,3),activation="relu",padding="same")(x)
    x = L.GlobalAveragePooling2D()(x)
    x = L.Dropout(0.5)(x)
    output = L.Dense(16,activation="softmax",kernel_initializer="he_normal")(x)
    return keras.Model(inp,output,name="classifier_3d_hypergamo")


def impact_load_classifier(source,method,seed):
    from pathlib import Path
    import numpy as np
    seed = int(seed)
    file_name = {"c_real_only_matched":"classifier.weights.h5","hypergamo":"C.weights.h5"}.get(method)
    if file_name is None:
        raise ValueError(method)
    folder = Path(source) / "models" / method / f"seed_{seed}"
    _,verified = _impact_checked_done(folder,method,seed,[file_name,"class_order.json"],pin_model=True)
    class_order = _impact_json(folder / "class_order.json")["original_class_ids"]
    np.testing.assert_array_equal(np.sort(class_order),np.arange(1,17))
    tf,keras,_ = impact_keras()
    # The main cell may explicitly request inference GPU; the default is CPU.
    device = globals().get("IMPACT_DEVICE","/CPU:0")
    if device not in ("/CPU:0","/GPU:0"):
        raise ValueError("IMPACT_DEVICE chỉ nhận /CPU:0 hoặc /GPU:0.")
    if device == "/GPU:0" and not tf.config.list_physical_devices("GPU"):
        raise RuntimeError("Đã chọn inference GPU nhưng session không có GPU; dùng /CPU:0.")
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    with tf.device(device):
        model = _impact_build_classifier(keras)
        # Shape-unique mapping supports both legacy tf-keras and Keras 3 HDF5.
        _impact_restore_weight_layers(model,folder / file_name)
        model.trainable = False
    model.impact_weights_fingerprint = verified
    model.impact_device = device
    return model,[int(value) for value in class_order]


### Code — `impact_probe_main_fragment.py` · phần 1/1: Điều phối phép thử độ nhạy frozen-model

**Mục đích:** Tái lập baseline trước perturbation; chạy original/shared-mask/nonshared-mask, kiểm weights không thay đổi, lưu cache/report. Một hàm lớn được giữ nguyên để không thay logic frozen.

**Đầu vào:** Các định nghĩa impact_* và cấu hình IMPACT_*. **Đầu ra:** paired_summary.csv, condition_summary.csv, manifest.json; ZIP nhỏ.

**Trạng thái:** Đã nhận kết quả seed42, 2.962 mẫu/14 lớp. **Giới hạn/cần review:** Có pass baseline prediction tolerance trước mask? Chỉ gọi đây là sensitivity trên 2962 mẫu/14 lớp, không gọi là số điểm accuracy do leakage?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma impact_probe_main_fragment.py 1/1 0:1 abe5d03887e1f5c66210a3b13db2a0351e49e783a9f8ab6ab84d0ab96e69339a
def impact_run_probe():
    from pathlib import Path
    import json
    import hashlib
    import time
    import zipfile
    import pandas as pd

    if IMPACT_K!=5 or not isinstance(IMPACT_EVAL_BATCH,int) or IMPACT_EVAL_BATCH<1:
        raise ValueError("Phép probe này khóa K=5 và cần batch inference nguyên dương.")
    config = {"methods":list(IMPACT_METHODS),"model_seeds":list(IMPACT_MODEL_SEEDS),
              "k":IMPACT_K,"mask_seeds":list(IMPACT_MASK_SEEDS),"batch":IMPACT_EVAL_BATCH,
              "fill":"saved_train_PCA_mean_transformed_by_saved_final_scaler",
              "match":"same Chebyshev-shell histogram; same center; no padding",
              "primary":"balanced mean paired (NLL_shared - NLL_nonshared) over evaluable classes",
              "interpretation":"frozen-model sensitivity, not causal inflation or independent confirmation"}
    t0 = time.time()
    data = impact_load_source(IMPACT_SOURCE_RUN_PATH,tuple(IMPACT_METHODS),tuple(IMPACT_MODEL_SEEDS))
    missing_reference = [(method,int(seed)) for method in IMPACT_METHODS for seed in IMPACT_MODEL_SEEDS
                         if (method,int(seed)) not in data.get("reference_probabilities",{})]
    if missing_reference:
        raise RuntimeError("Output nguồn chưa có predictions.npz của test B: "+str(missing_reference)+
                           ". Chọn Notebook Outputs của version đã test xong, không Run All hoặc train lại.")
    X,y,coords = data["X_test"],data["y_test"],data["coords_test"]
    geometry = impact_geometry(data["gt"].shape,data["coords_train"],coords)
    eligible,masks = impact_paired_masks(geometry,coords,IMPACT_K,tuple(IMPACT_MASK_SEEDS))
    per_class = [{"class_id":cid+1,"test_samples":int((y==cid).sum()),
                  "eligible_samples":int((y[eligible]==cid).sum())} for cid in range(16)]
    if len(eligible)!=2962 or [r["class_id"] for r in per_class if not r["eligible_samples"]]!=[9,16]:
        raise RuntimeError("Geometry differs from verified B/K5 scope; stop before model inference.")
    fill = np.asarray(data["fill_vector"],dtype=np.float32).reshape(40,1)
    tf_runtime,keras_runtime,growth = impact_keras()
    runtime = {"tensorflow":getattr(tf_runtime,"__version__","unknown"),
               "keras":getattr(keras_runtime,"__version__","unknown"),
               "numpy":np.__version__,"device":IMPACT_DEVICE,"gpu_memory_growth":growth}
    identity = {"source_fingerprint":data["fingerprint"],"config":config,
                "implementation":IMPACT_IMPLEMENTATION_SHA256,
                "runtime":runtime,
                "masks_sha256":impact_array_hash(masks),"fill_sha256":impact_array_hash(fill)}
    encoded = json.dumps(identity,sort_keys=True,allow_nan=False).encode()
    identity_hash = hashlib.sha256(encoded).hexdigest()
    out = Path(IMPACT_OUTPUT_ROOT)/("leakage_sensitivity_"+identity_hash[:12])
    out.mkdir(parents=True,exist_ok=True)

    def atomic_json(path,value):
        path = Path(path)
        temp = path.with_name(path.name+".tmp")
        temp.write_text(json.dumps(value,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False),encoding="utf-8")
        temp.replace(path)

    def sha_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as stream:
            for block in iter(lambda:stream.read(1024*1024),b""):
                digest.update(block)
        return digest.hexdigest()

    def atomic_npz(path,**arrays):
        temp = path.with_name(path.name+".tmp")
        with temp.open("wb") as stream:
            np.savez_compressed(stream,**arrays)
        temp.replace(path)

    spec_path = out/"probe_spec.json"
    if spec_path.is_file() and json.loads(spec_path.read_text(encoding="utf-8"))!=identity:
        raise RuntimeError("Output identity collision or changed probe specification; stop.")
    atomic_json(out/"manifest.json",{"complete":False,"probe_id":identity_hash[:12],
                                    "source_run_id":"50e988fae260","status":"inference/reproduction gates pending"})
    atomic_json(spec_path,identity)
    pd.DataFrame(per_class).to_csv(out/"geometry_by_class.csv",index=False)
    atomic_npz(out/"paired_masks.npz",eligible_indices=eligible,coords=coords[eligible],
               y_true=y[eligible],masks=masks,fill_vector=fill,
               mask_seeds=np.asarray(IMPACT_MASK_SEEDS))
    geometry_summary = {"test_samples":len(y),"eligible_samples":len(eligible),"eligible_class_count":14,
                        "not_evaluable_class_ids":[9,16],
                        "all_source_shared_patches":int(geometry["all_sources_shared"].sum()),
                        "centers_seen_as_train_context":int(geometry["center_shared"].sum()),
                        "shared_occurrence_fraction":float(geometry["shared"].sum()/geometry["valid"].sum()),
                        "same_shell_only":True,"per_class":per_class}
    atomic_json(out/"geometry_summary.json",geometry_summary)
    print("Đã khóa geometry: 2.962 mẫu, 14 lớp; lớp 9/16 chưa có nonshared control.")
    print("Nguồn checkpoint:",data["source"],"| báo cáo:",out)
    summaries,comparisons,comparisons_by_class = [],[],[]

    def model_state_hash(model):
        digest = hashlib.sha256()
        for weight in model.get_weights():
            digest.update(impact_array_hash(weight).encode())
        return digest.hexdigest()

    def predict_condition(model,order,method,seed,name,indices,mask=None):
        path = out/(method+"_seed_"+str(seed)+"_"+name+".npz")
        sidecar = path.with_suffix(".npz.json")
        signature = {"probe_identity":identity_hash,"method":method,"model_seed":int(seed),
                     "condition":name,"indices":impact_array_hash(indices),
                     "mask":None if mask is None else impact_array_hash(mask)}
        if path.is_file() and sidecar.is_file():
            cached = json.loads(sidecar.read_text(encoding="utf-8"))
            if cached.get("signature")!=signature or cached.get("sha256")!=sha_file(path):
                raise RuntimeError("Cache inference không khớp: "+str(path))
            with np.load(path,allow_pickle=False) as saved:
                probabilities,l2 = saved["probabilities"],saved["perturbation_l2"]
                np.testing.assert_array_equal(saved["y_true"],y[indices])
                np.testing.assert_array_equal(saved["coords"],coords[indices])
            impact_per_sample(y[indices],probabilities)
            if l2.shape!=(len(indices),) or not np.isfinite(l2).all() or (l2<0).any():
                raise RuntimeError("Invalid cached perturbation magnitude")
            print("Dùng lại inference đã hoàn tất:",method,seed,name)
            return probabilities,l2
        probabilities = np.empty((len(indices),16),dtype=np.float32)
        l2 = np.zeros(len(indices),dtype=np.float64)
        started,last = time.time(),time.time()
        for start in range(0,len(indices),IMPACT_EVAL_BATCH):
            stop = min(start+IMPACT_EVAL_BATCH,len(indices))
            batch = X[indices[start:stop]].copy()
            if mask is not None:
                chosen = mask[start:stop]
                difference = batch[chosen].astype(np.float64)-fill
                l2[start:stop] = np.sqrt(np.square(difference).reshape(stop-start,IMPACT_K,-1).sum((1,2)))
                batch[chosen] = fill
            tf_runtime,_,_ = impact_keras()
            with tf_runtime.device(model.impact_device):
                raw = model(batch,training=False)
            raw = np.asarray(raw.numpy() if hasattr(raw,"numpy") else raw,dtype=np.float32)
            standardized = np.empty_like(raw)
            standardized[:,order] = raw
            impact_per_sample(y[indices[start:stop]],standardized)
            probabilities[start:stop] = standardized
            if start==0 or stop==len(indices) or time.time()-last>15:
                print(method,"seed",seed,name,stop,"/",len(indices),";",round(time.time()-started,1),"s")
                last = time.time()
        atomic_npz(path,probabilities=probabilities,perturbation_l2=l2,
                   y_true=y[indices],coords=coords[indices])
        atomic_json(sidecar,{"signature":signature,"sha256":sha_file(path)})
        return probabilities,l2

    for method in IMPACT_METHODS:
        for seed in IMPACT_MODEL_SEEDS:
            model,original_class_ids = impact_load_classifier(data["source"],method,int(seed))
            order = np.asarray(original_class_ids,dtype=int)-1
            state_before = model_state_hash(model)
            baseline,_ = predict_condition(model,order,method,seed,"original",np.arange(len(y)))
            key = (method,int(seed))
            reference = data.get("reference_probabilities",{}).get(key)
            if reference is None:
                raise RuntimeError("Cần predictions.npz đã lưu của test B cho baseline reproduction gate: "+str(key))
            reference = np.asarray(reference)
            max_difference = float(np.max(np.abs(baseline-reference)))
            classification_equal = bool(np.array_equal(baseline.argmax(1),reference.argmax(1)))
            reproduction = {"max_abs_probability_difference":max_difference,
                            "all_predictions_identical":classification_equal,"tolerance":1e-4,
                            "passed":classification_equal and max_difference<=1e-4}
            atomic_json(out/(method+"_seed_"+str(seed)+"_reproduction.json"),reproduction)
            if not reproduction["passed"]:
                raise RuntimeError("Dự đoán nguyên bản không tái lập trong tolerance; dừng trước perturbation: "+str(key))
            pd.DataFrame(impact_association_rows(y,baseline,geometry)).to_csv(
                out/(method+"_seed_"+str(seed)+"_association_within_class.csv"),index=False)
            original = baseline[eligible]
            original_values = impact_per_sample(y[eligible],original)
            baseline_metrics = impact_metrics(y,baseline)
            atomic_json(out/(method+"_seed_"+str(seed)+"_baseline_all16.json"),baseline_metrics)
            subset_metrics = impact_metrics(y[eligible],original)
            summaries.append({"method":method,"model_seed":seed,"condition":"original_same_subset",
                              "mask_seed":None,**subset_metrics})
            for rep,mask_seed in enumerate(IMPACT_MASK_SEEDS):
                probabilities_by_arm,values_by_arm,l2_by_arm = [],[],[]
                for arm,category in enumerate(("shared","nonshared")):
                    current,l2 = predict_condition(model,order,method,seed,
                                                  category+"_maskseed_"+str(mask_seed),eligible,masks[rep,arm])
                    current_values = impact_per_sample(y[eligible],current)
                    desc = impact_geometry_descriptors(masks[rep,arm],geometry)
                    table = {"test_index":eligible,"row":coords[eligible,0],"col":coords[eligible,1],
                             "class_id":y[eligible]+1,"shared_fraction":geometry["shared_fraction"][eligible],
                             "original_prediction":original_values["prediction"]+1,
                             "perturbed_prediction":current_values["prediction"]+1,
                             "original_true_probability":original_values["true_probability"],
                             "perturbed_true_probability":current_values["true_probability"],
                             "delta_NLL":current_values["nll"]-original_values["nll"],
                             "delta_true_probability":current_values["true_probability"]-original_values["true_probability"],
                             "prediction_flipped":current_values["prediction"]!=original_values["prediction"],
                             "perturbation_l2":l2,**desc}
                    pd.DataFrame(table).to_csv(out/(method+"_seed_"+str(seed)+"_"+category+
                                                  "_maskseed_"+str(mask_seed)+"_per_sample.csv"),index=False)
                    metrics = impact_metrics(y[eligible],current)
                    summaries.append({"method":method,"model_seed":seed,"condition":category,
                                      "mask_seed":mask_seed,**metrics,
                                      "OA_drop_percentage_points":100*(subset_metrics["OA"]-metrics["OA"]),
                                      "mean_delta_NLL":float(table["delta_NLL"].mean()),
                                      "prediction_flip_fraction":float(table["prediction_flipped"].mean()),
                                      "mean_perturbation_l2":float(l2.mean())})
                    probabilities_by_arm.append(current)
                    values_by_arm.append(current_values)
                    l2_by_arm.append(l2)
                paired = values_by_arm[0]["nll"]-values_by_arm[1]["nll"]
                present = np.unique(y[eligible])
                comparison = {"method":method,"model_seed":seed,"mask_seed":mask_seed,
                              "samples":len(eligible),"evaluable_classes":len(present),
                              "balanced_excess_delta_NLL":float(np.mean([paired[y[eligible]==cid].mean() for cid in present])),
                              "sample_mean_excess_delta_NLL":float(paired.mean()),
                              "fraction_shared_perturbation_higher_NLL":float((paired>0).mean()),
                              "mean_l2_shared":float(l2_by_arm[0].mean()),
                              "mean_l2_nonshared":float(l2_by_arm[1].mean())}
                comparisons.append(comparison)
                for cid in present:
                    selected = y[eligible]==cid
                    comparisons_by_class.append({"method":method,"model_seed":seed,"mask_seed":mask_seed,
                                                 "class_id":int(cid)+1,"samples":int(selected.sum()),
                                                 "mean_excess_delta_NLL":float(paired[selected].mean()),
                                                 "mean_NLL_shared":float(values_by_arm[0]["nll"][selected].mean()),
                                                 "mean_NLL_nonshared":float(values_by_arm[1]["nll"][selected].mean()),
                                                 "OA_shared":float(values_by_arm[0]["correct"][selected].mean()),
                                                 "OA_nonshared":float(values_by_arm[1]["correct"][selected].mean()),
                                                 "mean_l2_shared":float(l2_by_arm[0][selected].mean()),
                                                 "mean_l2_nonshared":float(l2_by_arm[1][selected].mean())})
                pd.DataFrame({"test_index":eligible,"class_id":y[eligible]+1,"paired_excess_delta_NLL":paired}).to_csv(
                    out/(method+"_seed_"+str(seed)+"_paired_maskseed_"+str(mask_seed)+".csv"),index=False)
            if model_state_hash(model)!=state_before:
                raise RuntimeError("Model state changed during inference; sensitivity results invalid.")
            del model
    pd.DataFrame(summaries).to_csv(out/"condition_summary.csv",index=False)
    pd.DataFrame(comparisons).to_csv(out/"paired_summary.csv",index=False)
    pd.DataFrame(comparisons_by_class).to_csv(out/"paired_by_class.csv",index=False)
    explanation = """B LEAKAGE SENSITIVITY — EXPLORATORY, CHECKPOINTS FIXED

Primary endpoint: mean over 14 evaluable classes of the paired NLL difference
(shared perturbation minus nonshared perturbation), for each model/mask seed.
Positive means greater sensitivity to replacing the shared group in this probe.
It is not the number of accuracy points inflated by training-source overlap.

Geometry eligibility is fixed before predictions. Class 9/Oats and
16/Stone-Steel-Towers have no nonshared control; no conclusion about their
relative shared/nonshared sensitivity is possible. All subset metrics use the
same 2962 samples and report Macro-F1 only over the 14 present classes.

Both arms preserve the center and padding, replace exactly 5 spectra and match
the Chebyshev-shell histogram. Angles, adjacency, content and L2 amplitudes need
not match. Those discrepancies and per-sample magnitudes are exported.
PCA/scaler and model weights stay fixed. Fill is saved train mean in PCA space,
then transformed by the saved scaler. Model inference uses training=False.

Within-class exposure/error tables are descriptive associations, not causal
estimates. Two mask seeds measure perturbation variation, not independent
training runs or scenes. No pixel-i.i.d. confidence intervals are claimed.
Indian Pines and B test have already been inspected during this project;
these results are exploratory, not independent confirmation.
Balanced means give equal weight to the 14 classes. Some eligible classes have
only a few samples; inspect paired_by_class.csv and geometry_by_class.csv.

No model training, G/D updates, hyperparameter search or baseline retraining.
Completed inference conditions are verified and reused when rerunning the
same cell with identical inputs/config/code. An interrupted condition restarts.
"""
    (out/"INTERPRETATION.txt").write_text(explanation,encoding="utf-8")
    files = {str(path.relative_to(out)):sha_file(path) for path in sorted(out.rglob("*"))
             if path.is_file() and path.name!="manifest.json" and not path.name.endswith(".tmp")}
    atomic_json(out/"manifest.json",{"complete":True,"source_run_id":"50e988fae260",
                                    "probe_id":identity_hash[:12],"config":config,
                                    "elapsed_seconds":time.time()-t0,"artifact_hashes":files,
                                    "classification_inflation_measured":False})
    archive = out.with_suffix(".zip")
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(out.rglob("*")):
            if path.is_file() and not path.name.endswith(".tmp"):
                bundle.write(path,str(Path(out.name)/path.relative_to(out)))
    print("HOÀN TẤT — inference-only, không train. Báo cáo:",out)
    print("ZIP nhỏ:",archive,"| gửi paired_summary.csv, condition_summary.csv và manifest.json.")
    print("Kết quả đo độ nhạy trên 14 lớp; chưa đo số điểm accuracy do leakage.")
    try:
        from IPython.display import display
        display(pd.DataFrame(comparisons))
    except ImportError:
        print(pd.DataFrame(comparisons).to_string(index=False))
    return str(out)


## 06 — Suite fresh HyperGAMO: can thiệp nguồn lúc train

Release4c08e7e92312; suite25d35efe55cc. 3regions×5seeds×2arms=30jobs; fixedfinal8000Cupdates/job (=4000loops). Train đầy16lớp; testpooled1109, exposed751,zero358 có12lớp thật (vắng1/7/9/16), classifier vẫn16outputs. Benchmark đã xác minh400Cupdates mỗi arm region1/seed42 và GPU smoke; chưa nhận fulltraining/test. REPORT.zip chỉ có metadata, không phải checkpoint backup.

### Code — `gan_exposure_data.py` · phần 1/7: Nguồn raw và IO có checksum

**Mục đích:** Khóa đúng hai MAT Indian Pines và provenance; ghi dữ liệu atomically để không đọc artifact dở.

**Đầu vào:** HSI/GT MAT theo SHA đã khóa. **Đầu ra:** Raw cube/GT đã kiểm.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Có đúng raw SHA và shape 145x145x200/145x145?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_data.py 1/7 0:3 3fb69f6330226c25e912bc665a6a04bcf426017f18d03399b3f600cb3b361fc6
"""Verified train-only observations for the registered HyperGAMO intervention.

No work is performed on import. This module can be embedded in a Kaggle cell.
All context labels are excluded from training input construction. Test patches
are constructed exclusively by the explicit exposure_load_test entry point.
"""
from pathlib import Path
import hashlib
import json
import os
import uuid
import numpy as np


EXPOSURE_RAW_SHA256 = {
    "hsi": "ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939",
    "gt": "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c",
}
EXPOSURE_SELECTION_SHA256 = "c91ea95a6d91df3751b3b13643fb4545234b5e7565960e6b255cef4a887c912c"
EXPOSURE_REGIONS = {
    1: {"box": [15, 39, 15, 39], "train_n": 9373, "test_n": 437,
        "affected_n": 285, "control_shared_raw_n": 596,
        "train_counts": [46,1047,685,155,473,724,28,478,20,936,2455,476,205,1265,297,83]},
    2: {"box": [15, 39, 105, 129], "train_n": 9554, "test_n": 330,
        "affected_n": 260, "control_shared_raw_n": 515,
        "train_counts": [46,1404,830,237,483,730,28,332,20,960,2252,593,205,955,386,93]},
    3: {"box": [105, 129, 45, 69], "train_n": 9582, "test_n": 342,
        "affected_n": 206, "control_shared_raw_n": 396,
        "train_counts": [46,1428,767,237,395,495,28,478,20,972,2230,593,149,1265,386,93]},
}
EXPOSURE_CLASS_NAMES = ["Alfalfa", "Corn-notill", "Corn-mintill", "Corn",
    "Grass-pasture", "Grass-trees", "Grass-pasture-mowed", "Hay-windrowed",
    "Oats", "Soybean-notill", "Soybean-mintill", "Soybean-clean", "Wheat",
    "Woods", "Buildings-Grass-Trees-Drives", "Stone-Steel-Towers"]


def exposure_sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def exposure_sha_array(value):
    array = np.ascontiguousarray(value)
    digest = hashlib.sha256()
    digest.update(str(array.dtype).encode("ascii"))
    digest.update(json.dumps(list(array.shape), separators=(",", ":")).encode("ascii"))
    digest.update(array.tobytes())
    return digest.hexdigest()


def _exposure_json_bytes(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=True,
                      separators=(",", ":"), allow_nan=False).encode("utf-8")


def _exposure_atomic_bytes(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        with temp.open("wb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


def _exposure_atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        with temp.open("wb") as stream:
            np.savez_compressed(stream, **arrays)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


def exposure_discover_raw(explicit_hsi=None, explicit_gt=None,
                          search_roots=("/kaggle/input",)):
    """Resolve the exact verified MAT bytes, including harmless duplicate copies."""
    chosen = {}
    for kind, explicit in (("hsi", explicit_hsi), ("gt", explicit_gt)):
        if explicit is not None:
            path = Path(explicit).expanduser().resolve()
            if not path.is_file() or exposure_sha_file(path) != EXPOSURE_RAW_SHA256[kind]:
                raise ValueError("Wrong verified Indian Pines %s MAT: %s" % (kind, path))
            chosen[kind] = path
    needed = set(EXPOSURE_RAW_SHA256) - set(chosen)
    if needed:
        candidates = set()
        for root in search_roots:
            root = Path(root).expanduser()
            if root.is_file() and root.suffix.lower() == ".mat":
                candidates.add(root.resolve())
            elif root.is_dir():
                candidates.update(p.resolve() for p in root.rglob("*")
                                  if p.is_file() and p.suffix.lower() == ".mat"
                                  and "pines" in p.name.lower())
        for path in sorted(candidates, key=lambda p: (len(str(p)), str(p))):
            digest = exposure_sha_file(path)
            for kind in sorted(needed):
                if digest == EXPOSURE_RAW_SHA256[kind]:
                    chosen[kind] = path
            needed -= set(chosen)
            if not needed:
                break
    if needed:
        raise FileNotFoundError("Exact verified MAT missing: %s. Attach original "
                                "indian_pines_corrected.mat and indian_pines_corrected_gt.mat; "
                                "no substitute labels or transformed cube." % ", ".join(sorted(needed)))
    return {"hsi_path": str(chosen["hsi"]), "gt_path": str(chosen["gt"]),
            "hsi_sha256": EXPOSURE_RAW_SHA256["hsi"],
            "gt_sha256": EXPOSURE_RAW_SHA256["gt"]}


def _exposure_load_raw(raw_paths):
    from scipy.io import loadmat
    arrays = {}
    for kind, shape in (("hsi", (145,145,200)), ("gt", (145,145))):
        path = Path(raw_paths[kind + "_path"])
        if exposure_sha_file(path) != EXPOSURE_RAW_SHA256[kind]:
            raise ValueError("Raw MAT hash changed: " + str(path))
        document = loadmat(path)
        candidates = [a for key, a in document.items() if not key.startswith("__")
                      and isinstance(a, np.ndarray) and a.shape == shape
                      and np.issubdtype(a.dtype, np.number)]
        if len(candidates) != 1 or not np.isfinite(candidates[0]).all():
            raise ValueError("Ambiguous/nonfinite MAT input " + kind)
        arrays[kind] = candidates[0]
    cube = arrays["hsi"].astype(np.float64)
    gt = arrays["gt"]
    if np.any(gt != np.floor(gt)) or set(np.unique(gt)) != set(range(17)):
        raise ValueError("GT must contain exactly integer classes0..16")
    gt = gt.astype(np.int16)
    if np.count_nonzero(gt) != 10249:
        raise ValueError("GT must contain exactly10249 labeled centers")
    return cube, gt




### Code — `gan_exposure_data.py` · phần 2/7: Hai nhánh observation cùng tâm train

**Mục đích:** U_test là hợp nguồn patch test. S là context train thuộc U_test; R là phản xạ 180° của S. Chỉ giữ tâm đủ điều kiện; blocked bỏ S, control bỏ R.

**Đầu vào:** GT và vùng hình học đã đăng ký. **Đầu ra:** coords_train, labels16, S/R, source unions, exposed counts.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Blocked union giao U_test có bằng 0? R có wholly nonshared, inbounds, không đụng tâm? Hai mask có cùng số pixel và shell histogram?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_data.py 2/7 0:3 3507fc97eb0bf978107b4538578defde1e3923fdafb512f76404a9a6a7b7f3bd
def _exposure_sources(shape, coords):
    offsets = np.arange(-5, 6, dtype=np.int32)
    rr = coords[:, 0, None, None] + offsets[None, :, None]
    cc = coords[:, 1, None, None] + offsets[None, None, :]
    valid = (rr >= 0) & (rr < shape[0]) & (cc >= 0) & (cc < shape[1])
    ids = np.broadcast_to(rr * shape[1] + cc, valid.shape).copy()
    ids[~valid] = -1
    return ids.astype(np.int32), valid


def exposure_build_geometry(gt, region_id, enforce_registered=True):
    """Labels are used only for eligible centers and their own training targets."""
    region_id = int(region_id)
    if region_id not in EXPOSURE_REGIONS or gt.shape != (145,145):
        raise ValueError("Unknown registered region or GTshape")
    expected = EXPOSURE_REGIONS[region_id]
    r0, r1, c0, c1 = expected["box"]
    coords = np.argwhere(gt > 0).astype(np.int32)
    in_box = ((coords[:,0] >= r0) & (coords[:,0] <= r1)
              & (coords[:,1] >= c0) & (coords[:,1] <= c1))
    test_coords = coords[in_box]
    test_ids, test_valid = _exposure_sources(gt.shape, test_coords)
    u_test = np.zeros(gt.size, dtype=bool)
    u_test[test_ids[test_valid]] = True
    train_candidates = coords[~u_test[coords[:,0] * gt.shape[1] + coords[:,1]]]
    ids, valid = _exposure_sources(gt.shape, train_candidates)
    s_mask = valid & u_test[np.maximum(ids, 0)]
    r_mask = s_mask[:, ::-1, ::-1].copy()
    eligible = ~(r_mask & (~valid | s_mask)).any(axis=(1,2))
    train_coords = train_candidates[eligible]
    train_ids, train_valid = ids[eligible], valid[eligible]
    s_mask, r_mask = s_mask[eligible], r_mask[eligible]
    if s_mask[:,5,5].any() or r_mask[:,5,5].any():
        raise AssertionError("Intervention removes a train center")
    blocked_keep = train_valid & ~s_mask
    control_keep = train_valid & ~r_mask
    blocked_union = np.zeros(gt.size, bool)
    control_union = np.zeros(gt.size, bool)
    blocked_union[train_ids[blocked_keep]] = True
    control_union[train_ids[control_keep]] = True
    exposure_counts = (test_valid & control_union[np.maximum(test_ids, 0)]).sum(axis=(1,2))
    y_train = gt[train_coords[:,0], train_coords[:,1]].astype(np.int64) - 1
    counts = np.bincount(y_train, minlength=16).astype(np.int64)
    if np.any(counts < 20) or (blocked_union & u_test).any():
        raise AssertionError("Class floor or blocked source-disjoint gate failed")
    np.testing.assert_array_equal(blocked_keep.sum(axis=(1,2)), control_keep.sum(axis=(1,2)))
    offsets = np.arange(-5,6)
    shell = np.maximum(np.abs(offsets[:,None]), np.abs(offsets[None,:]))
    for radius in range(6):
        np.testing.assert_array_equal((s_mask & (shell == radius)).sum(axis=(1,2)),
                                      (r_mask & (shell == radius)).sum(axis=(1,2)))
    geometry = {"coords_train": train_coords, "y_train": y_train,
        "counts": counts, "class_order": np.lexsort((np.arange(16),counts)).astype(np.int64),
        "coords_test": test_coords, "test_source_union": u_test.reshape(gt.shape),
        "s_mask": s_mask, "r_mask": r_mask, "train_source_ids": train_ids,
        "train_valid": train_valid, "test_source_ids": test_ids, "test_valid": test_valid,
        "blocked_source_union": blocked_union.reshape(gt.shape),
        "control_source_union": control_union.reshape(gt.shape),
        "exposure_counts": exposure_counts.astype(np.int64),
        "shared_source_counts_train": s_mask.sum(axis=(1,2)).astype(np.int64),
        "touched_train": s_mask.any(axis=(1,2))}
    if enforce_registered:
        assert len(train_coords) == expected["train_n"]
        assert len(test_coords) == expected["test_n"]
        assert int((exposure_counts > 0).sum()) == expected["affected_n"]
        assert int((control_union & u_test).sum()) == expected["control_shared_raw_n"]
        np.testing.assert_array_equal(counts, expected["train_counts"])
    return geometry




### Code — `gan_exposure_data.py` · phần 3/7: Oracle hình học độc lập

**Mục đích:** Đối chiếu source IDs bằng vòng tọa độ trực tiếp độc lập với bộ dựng mask.

**Đầu vào:** GT, geometry. **Đầu ra:** Chứng nhận giao nguồn/mask đúng.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Oracle có gọi lại chính hàm cần kiểm không?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_data.py 3/7 0:2 87976f39f92d53b036f43a37edefc64671ae587e840e1c8d8f9f51371021d88d
def exposure_physical_oracle(gt, region_id, geometry):
    """Independent nested-coordinate oracle; never calls source-ID extractor."""
    height, width = gt.shape
    r0,r1,c0,c1 = EXPOSURE_REGIONS[int(region_id)]["box"]
    test_centers = [(rr,cc) for rr in range(height) for cc in range(width)
                    if gt[rr,cc] > 0 and r0 <= rr <= r1 and c0 <= cc <= c1]
    test_sources = set()
    for row,col in test_centers:
        test_sources.update((rr,cc) for rr in range(max(0,row-5),min(height,row+6))
                            for cc in range(max(0,col-5),min(width,col+6)))
    train_centers, oracle_s, oracle_r, counts_train = [], [], [], []
    blocked_sources, control_sources = set(), set()
    for row in range(height):
        for col in range(width):
            if gt[row,col] <= 0 or (row,col) in test_sources:
                continue
            patch = {(rr,cc) for rr in range(max(0,row-5),min(height,row+6))
                     for cc in range(max(0,col-5),min(width,col+6))}
            removed_s = patch & test_sources
            removed_r = {(2*row-rr, 2*col-cc) for rr,cc in removed_s}
            if any(rr < 0 or rr >= height or cc < 0 or cc >= width
                   for rr,cc in removed_r) or removed_r & test_sources:
                continue
            if (row,col) in removed_s | removed_r:
                raise AssertionError("Oracle found removed center")
            train_centers.append((row,col))
            sm = np.zeros((11,11),bool)
            rm = np.zeros((11,11),bool)
            for rr,cc in removed_s:
                sm[rr-row+5,cc-col+5] = True
            for rr,cc in removed_r:
                rm[rr-row+5,cc-col+5] = True
            oracle_s.append(sm)
            oracle_r.append(rm)
            counts_train.append(len(removed_s))
            blocked_sources.update(patch - removed_s)
            control_sources.update(patch - removed_r)
    np.testing.assert_array_equal(np.asarray(train_centers), geometry["coords_train"])
    np.testing.assert_array_equal(np.asarray(test_centers), geometry["coords_test"])
    np.testing.assert_array_equal(np.asarray(oracle_s), geometry["s_mask"])
    np.testing.assert_array_equal(np.asarray(oracle_r), geometry["r_mask"])
    np.testing.assert_array_equal(counts_train, geometry["shared_source_counts_train"])
    for source_set,key in ((test_sources,"test_source_union"),
                           (blocked_sources,"blocked_source_union"),
                           (control_sources,"control_source_union")):
        mask = np.zeros(gt.shape,bool)
        for rr,cc in source_set:
            mask[rr,cc] = True
        np.testing.assert_array_equal(mask,geometry[key])
    oracle_e = [sum((rr,cc) in control_sources
                    for rr in range(max(0,row-5),min(height,row+6))
                    for cc in range(max(0,col-5),min(width,col+6)))
                for row,col in test_centers]
    np.testing.assert_array_equal(oracle_e,geometry["exposure_counts"])
    assert not blocked_sources & test_sources
    return {"independent_physical_oracle": True,
            "blocked_shared_raw_sources": 0,
            "control_shared_raw_sources": len(control_sources & test_sources),
            "train_centers": len(train_centers), "test_centers": len(test_centers),
            "affected_test_centers": int(np.count_nonzero(oracle_e))}



### Code — `gan_exposure_data.py` · phần 4/7: PCA/scaler và patch observation

**Mục đích:** Fit PCA40 full + MinMax(-1,1) chỉ spectra tại tâm train. Hai nhánh dùng cùng transform; context bị bỏ dùng scaled train-center mean, padding ngoài ảnh dùng 0.

**Đầu vào:** Cube, coords_train, geometry, arm. **Đầu ra:** Preprocessing và train patches (N,11,11,40,1).

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Không fit trên toàn cảnh/test hoặc fill từ test; padding khác masked fill?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_data.py 4/7 1:2 116dc5bdd91dd110b03907e033403130cf538786996fc43b10cdd2d62a956b43

def exposure_fit_preprocessing(cube, coords_train):
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import MinMaxScaler
    centers = cube[coords_train[:,0],coords_train[:,1]].astype(np.float64)
    pca = PCA(n_components=40, svd_solver="full", whiten=False)
    pca.fit(centers)
    projected = pca.transform(centers)
    scaler = MinMaxScaler(feature_range=(-1,1), clip=False).fit(projected)
    return {"pca_mean": pca.mean_, "pca_components": pca.components_,
            "pca_explained_variance": pca.explained_variance_,
            "scaler_scale": scaler.scale_, "scaler_min": scaler.min_,
            "scaler_data_min": scaler.data_min_, "scaler_data_max": scaler.data_max_,
            "fit_coords_sha256": np.asarray(exposure_sha_array(coords_train)),
            "fit_samples": np.asarray(len(coords_train),np.int64),
            # PCA(raw train-center mean)=0; this is its train-fitted scaled
            # value. Image-boundary padding remains0, as in baseline B.
            "missing_scaled_fill": scaler.min_.copy()}


def exposure_transform_spectra(spectra, preprocessing):
    projected = (np.asarray(spectra,np.float64) - preprocessing["pca_mean"]) @ preprocessing["pca_components"].T
    return projected * preprocessing["scaler_scale"] + preprocessing["scaler_min"]


def exposure_extract_train(cube, geometry, preprocessing, arm, indices=None):
    if arm not in ("blocked","reuse_control"):
        raise ValueError("Unknown arm: " + str(arm))
    indices = np.arange(len(geometry["coords_train"])) if indices is None else np.asarray(indices,np.int64)
    ids = geometry["train_source_ids"][indices]
    removed = geometry["s_mask"] if arm == "blocked" else geometry["r_mask"]
    allowed = geometry["train_valid"][indices] & ~removed[indices]
    if arm == "blocked" and geometry["test_source_union"].ravel()[ids[allowed]].any():
        raise AssertionError("Blocked extractor attempted to read test source")
    # Read and transform only allowed physical coordinates, rather than build a
    # globally transformed cube which could accidentally enter training later.
    unique_ids, inverse = np.unique(ids[allowed], return_inverse=True)
    transformed = exposure_transform_spectra(cube.reshape(-1,cube.shape[2])[unique_ids],preprocessing).astype(np.float32)
    output = np.zeros((len(indices),11,11,40,1),np.float32)
    valid_removed = geometry["train_valid"][indices] & removed[indices]
    output[...,0][valid_removed] = preprocessing["missing_scaled_fill"].astype(np.float32)
    output[...,0][allowed] = transformed[inverse]
    if not np.isfinite(output).all():
        raise ValueError("Nonfinite training observations")
    return output



### Code — `gan_exposure_data.py` · phần 5/7: Kiểm counterfactual và cache train

**Mục đích:** Thay spectra trong U_test để kiểm blocked, preprocessing và supports bất biến; lưu mmap và provenance selection.

**Đầu vào:** Cube/GT/preprocessing/masks. **Đầu ra:** Counterfactual certificate và artifacts.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Đổi toàn bộ U_test có làm blocked train X đổi không?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_data.py 5/7 1:3 ffe982358cbaf8de67a181f94765d860ad8d7c126b0490c7f7866caaf60a734b

def exposure_counterfactual_audit(cube, gt, region_id, geometry, preprocessing):
    """Numerical invariance of fit/support; positive control detects reuse input."""
    changed = cube.copy()
    changed[geometry["test_source_union"]] += 12345.6789
    rebuilt = exposure_fit_preprocessing(changed, geometry["coords_train"])
    for key in preprocessing:
        np.testing.assert_array_equal(preprocessing[key],rebuilt[key])
    # Digest every support batch in both arms. No classification is performed.
    digests = {arm:[hashlib.sha256(),hashlib.sha256()] for arm in ("blocked","reuse_control")}
    for start in range(0,len(geometry["coords_train"]),128):
        indices = np.arange(start,min(start+128,len(geometry["coords_train"])))
        for arm in digests:
            before = exposure_extract_train(cube,geometry,preprocessing,arm,indices)
            after = exposure_extract_train(changed,geometry,rebuilt,arm,indices)
            digests[arm][0].update(before.tobytes())
            digests[arm][1].update(after.tobytes())
            if arm == "blocked":
                np.testing.assert_array_equal(before,after)
    hashes = {arm:[h.hexdigest() for h in pair] for arm,pair in digests.items()}
    assert hashes["blocked"][0] == hashes["blocked"][1]
    assert hashes["reuse_control"][0] != hashes["reuse_control"][1]
    changed_gt = gt.copy()
    heldout = geometry["test_source_union"] & (gt > 0)
    changed_gt[heldout] = 1 + changed_gt[heldout] % 16
    assert not np.array_equal(gt,changed_gt)
    changed_geometry = exposure_build_geometry(changed_gt,region_id,enforce_registered=False)
    for key in geometry:
        np.testing.assert_array_equal(geometry[key],changed_geometry[key])
    return {"preprocessing_invariant_to_test_source_spectra": True,
            "all_blocked_train_supports_invariant": True,
            "reuse_control_inputs_change": True,
            "heldout_class_ids_without_eligibility_change_train_invariant": True,
            "train_tensor_digests_original_changed": hashes,
            "original_raw_inputs_unmodified": True}


def _exposure_write_train_npy(path, cube, geometry, preprocessing, arm):
    path = Path(path)
    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        output = np.lib.format.open_memmap(temp,mode="w+",dtype=np.float32,
                    shape=(len(geometry["coords_train"]),11,11,40,1))
        for start in range(0,len(output),128):
            indices = np.arange(start,min(start+128,len(output)))
            output[start:start+len(indices)] = exposure_extract_train(cube,geometry,preprocessing,arm,indices)
        output.flush()
        del output
        os.replace(temp,path)
    finally:
        if temp.exists():
            temp.unlink()


def _exposure_selection_bytes(scientific_config):
    text = scientific_config.get("selection_manifest_text", globals().get("EXPOSURE_SELECTION_MANIFEST_TEXT"))
    if text is not None:
        data = text.encode("utf-8") if isinstance(text,str) else bytes(text)
    elif scientific_config.get("selection_manifest_path"):
        data = Path(scientific_config["selection_manifest_path"]).read_bytes()
    else:
        raise ValueError("Embed the original region-selection manifest before prepare")
    if hashlib.sha256(data).hexdigest() != EXPOSURE_SELECTION_SHA256:
        raise ValueError("Region-selection provenance hash differs from registered grid")
    selection = json.loads(data)
    if selection["GT_sha256"] != EXPOSURE_RAW_SHA256["gt"] or selection["candidate_grid_starts"] != [15,45,75,105]:
        raise ValueError("Selection provenance does not match registered GT/grid")
    return data


def _exposure_verify_artifacts(root, manifest):
    for relative,digest in manifest["artifacts"].items():
        path = root / relative
        if not path.is_file() or exposure_sha_file(path) != digest:
            raise ValueError("Prepared data artifact missing/corrupt: " + relative)




### Code — `gan_exposure_data.py` · phần 6/7: Chuẩn bị ba vùng và nạp train

**Mục đích:** Tạo cache train của cả hai arms; không xuất X_test/y_test tại prepare. Cache reuse cần contract/artifact hashes đúng.

**Đầu vào:** Raw source và scientific config. **Đầu ra:** data_manifest.json, region manifests, train readonly mmap.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Test tensor/labels có bị load trước giai đoạn test? Pair invariants có cùng coords/labels/preprocess/order?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_data.py 6/7 0:2 bcaacd22e12b80a756cac839448eeec48d7f4fc8a09310fbcffb78b346d26241
def exposure_prepare(raw_paths, output_root, scientific_config):
    """Create immutable-by-contract data artifacts, safely reusing hash-verified output.

    Scientific configuration may contain additional training fields; data-critical
    settings below are fixed. Metadata contains no test labels or test tensors.
    """
    import sklearn
    root = Path(output_root).expanduser().resolve()
    root.mkdir(parents=True,exist_ok=True)
    config = dict(scientific_config)
    if int(config.get("patch_size",11)) != 11 or int(config.get("pca_components",40)) != 40 or config.get("pca_solver","full") != "full":
        raise ValueError("Registered preprocessing requires fullPCA40 and patch11")
    if config.get("pca_whiten",False) or config.get("scaler_clip",False):
        raise ValueError("Registered preprocessing disables whitening/scaler clipping")
    if tuple(config.get("scaler_feature_range",(-1,1))) != (-1,1):
        raise ValueError("Registered baseline B scaler uses MinMax(-1,1)")
    selection_bytes = _exposure_selection_bytes(config)
    for key in ("selection_manifest_text","selection_manifest_path"):
        config.pop(key,None)
    signature = {"raw_sha256": EXPOSURE_RAW_SHA256,
        "selection_manifest_sha256": EXPOSURE_SELECTION_SHA256,
        "scientific_config": config, "data_contract": "train-exposure-v2-B-scaler-mean-fill",
        "patch_size":11, "pca_components":40, "pca_solver":"full",
        "scaler":"MinMax(-1,1), no clipping", "image_padding_scaled":0,
        "missing_valid_context":"scaled raw train-center mean: scaler_min",
        "sklearn_version":sklearn.__version__, "numpy_version":np.__version__}
    signature_hash = hashlib.sha256(_exposure_json_bytes(signature)).hexdigest()
    manifest_path = root / "data_manifest.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest["signature_sha256"] != signature_hash:
            raise ValueError("Data directory already contains different registered configuration")
        _exposure_load_raw(raw_paths)  # Verify current mounted raw bytes too.
        _exposure_verify_artifacts(root,manifest)
        # This locator can change when Kaggle remounts identical raw bytes under
        # a new slug. It is intentionally outside immutable scientific identity.
        _exposure_atomic_bytes(root / "raw_locator.json",_exposure_json_bytes(raw_paths))
        return {"root":str(root), "manifest":manifest, "regions":manifest["regions"]}
    cube,gt = _exposure_load_raw(raw_paths)
    _exposure_atomic_bytes(root / "region_selection.json",selection_bytes)
    regions = {}
    artifacts = {"region_selection.json":EXPOSURE_SELECTION_SHA256}
    for region_id in sorted(EXPOSURE_REGIONS):
        folder = root / ("region_%d" % region_id)
        folder.mkdir(exist_ok=True)
        geometry = exposure_build_geometry(gt,region_id)
        oracle = exposure_physical_oracle(gt,region_id,geometry)
        preprocessing = exposure_fit_preprocessing(cube,geometry["coords_train"])
        counterfactual = exposure_counterfactual_audit(cube,gt,region_id,geometry,preprocessing)
        _exposure_atomic_npz(folder / "geometry.npz",**geometry)
        _exposure_atomic_npz(folder / "preprocessing.npz",**preprocessing)
        for arm in ("blocked","reuse_control"):
            _exposure_write_train_npy(folder / ("X_train_%s.npy" % arm),cube,geometry,preprocessing,arm)
        file_hashes = {p.name:exposure_sha_file(p) for p in sorted(folder.iterdir()) if p.is_file() and p.name != "region_manifest.json" and not p.name.endswith(".tmp")}
        metadata = {"region_id":region_id,"box_inclusive_zero_based":EXPOSURE_REGIONS[region_id]["box"],
            "train_n":len(geometry["coords_train"]),"test_n":len(geometry["coords_test"]),
            "affected_test_n":int((geometry["exposure_counts"] > 0).sum()),
            "train_class_counts":geometry["counts"].tolist(),
            "class_order_zero_based":geometry["class_order"].tolist(),
            "touched_train_n":int(geometry["touched_train"].sum()),
            "shared_train_source_occurrences_per_pass":int(geometry["shared_source_counts_train"].sum()),
            "train_coords_sha256":exposure_sha_array(geometry["coords_train"]),
            "train_labels_sha256":exposure_sha_array(geometry["y_train"]),
            "train_class_counts_sha256":exposure_sha_array(geometry["counts"]),
            "class_order_sha256":exposure_sha_array(geometry["class_order"]),
            "mask_s_sha256":exposure_sha_array(geometry["s_mask"]),
            "mask_r_sha256":exposure_sha_array(geometry["r_mask"]),
            "preprocessing_sha256":file_hashes["preprocessing.npz"],
            "test_exposure_counts_sha256":exposure_sha_array(geometry["exposure_counts"]),
            "fit_scope":"same train-center spectra for both arms; outside U_test",
            "raw_paths":dict(raw_paths),"missing_scaled_fill":preprocessing["missing_scaled_fill"].tolist(),
            "missing_scaled_fill_sha256":exposure_sha_array(preprocessing["missing_scaled_fill"]),
            "image_padding_scaled":0,"scaler_feature_range":[-1,1],
            "oracle":oracle,"counterfactual":counterfactual,
            "artifact_sha256":dict(file_hashes),"training_performed":False,
            "test_tensor_created":False,"test_predictions_created":False}
        _exposure_atomic_bytes(folder / "region_manifest.json",_exposure_json_bytes(metadata))
        file_hashes["region_manifest.json"] = exposure_sha_file(folder / "region_manifest.json")
        artifacts.update({"region_%d/%s" % (region_id,name):digest for name,digest in file_hashes.items()})
        regions[str(region_id)] = metadata
        print("Prepared region%d: train%d, test geometry%d, blockedrawintersection0; no test tensor." % (region_id,metadata["train_n"],metadata["test_n"]),flush=True)
    manifest = {"signature":signature,"signature_sha256":signature_hash,
        "raw_paths":dict(raw_paths),"regions":regions,"artifacts":artifacts,
        "read_only_by_contract":True,"test_tensor_created":False,
        "test_predictions_created":False,"training_performed":False}
    _exposure_atomic_bytes(manifest_path,_exposure_json_bytes(manifest))
    _exposure_atomic_bytes(root / "raw_locator.json",_exposure_json_bytes(raw_paths))
    return {"root":str(root),"manifest":manifest,"regions":regions}


def _exposure_read_region(root,region_id):
    root = Path(root).expanduser().resolve()
    region_id = int(region_id)
    if region_id not in EXPOSURE_REGIONS:
        raise ValueError("Region must be1,2,3")
    manifest = json.loads((root / "data_manifest.json").read_text(encoding="utf-8"))
    folder = root / ("region_%d" % region_id)
    metadata = manifest["regions"][str(region_id)]
    for name in ("region_manifest.json","geometry.npz","preprocessing.npz"):
        expected = manifest["artifacts"]["region_%d/%s" % (region_id,name)]
        if exposure_sha_file(folder / name) != expected:
            raise ValueError("Region artifact corrupt: " + name)
    with np.load(folder / "geometry.npz",allow_pickle=False) as source:
        geometry = {key:source[key] for key in source.files}
    return root,folder,manifest,metadata,geometry


def exposure_load_region(root, region_id, arm):
    if arm not in ("blocked","reuse_control"):
        raise ValueError("Unknown arm")
    _,folder,manifest,metadata,geometry = _exposure_read_region(root,region_id)
    filename = "X_train_%s.npy" % arm
    expected = manifest["artifacts"]["region_%d/%s" % (int(region_id),filename)]
    if exposure_sha_file(folder / filename) != expected:
        raise ValueError("Training input hash changed")
    x_train = np.load(folder / filename,mmap_mode="r",allow_pickle=False)
    if x_train.shape != (metadata["train_n"],11,11,40,1) or x_train.dtype != np.float32:
        raise ValueError("Training input tensor contract differs")
    common = {key:metadata[key] for key in ("train_coords_sha256","train_labels_sha256",
              "train_class_counts_sha256","class_order_sha256",
              "preprocessing_sha256","test_exposure_counts_sha256")}
    return {"X_train":x_train,"y_train":geometry["y_train"],
        "coords_train":geometry["coords_train"],"counts":geometry["counts"],
        "class_order":geometry["class_order"],"touched_train":geometry["touched_train"],
        "shared_source_counts_train":geometry["shared_source_counts_train"],
        "metadata":dict(metadata,arm=arm,input_sha256=expected,pair_invariants=common)}



### Code — `gan_exposure_data.py` · phần 7/7: Mở test ở giai đoạn cuối

**Mục đích:** Tạo patch full11x11x40 cho cả hai nhánh bằng preprocessing train đã lưu; đây là cửa duy nhất nạp test features/targets của suite.

**Đầu vào:** Suite đã đủ test gates. **Đầu ra:** Identical full X_test/y_test/coords/exposure.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Hai arms chấm đúng cùng full tensor, không dùng hai mask test khác nhau?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_data.py 7/7 1:1 3955efa6d82900d3a3a139708ff2c86661290df55c87320b50a9f12c9908cd9f

def exposure_load_test(root, region_id):
    """Explicit test-stage call; returns the identical full tensor for both arms.

    The orchestrator, not this data-only module, enforces suite-complete gates.
    Test targets are materialized here, rather than exported into train artifacts.
    The raw GT was already read for geometry/count checks during preparation.
    """
    root,folder,manifest,metadata,geometry = _exposure_read_region(root,region_id)
    locator_path = root / "raw_locator.json"
    raw_paths = (json.loads(locator_path.read_text(encoding="utf-8"))
                 if locator_path.is_file() else manifest["raw_paths"])
    if not all(Path(raw_paths[kind + "_path"]).is_file() for kind in ("hsi","gt")):
        raw_paths = exposure_discover_raw(search_roots=("/kaggle/input",))
    cube,gt = _exposure_load_raw(raw_paths)
    with np.load(folder / "preprocessing.npz",allow_pickle=False) as source:
        preprocessing = {key:source[key] for key in source.files}
    ids = geometry["test_source_ids"]
    valid = geometry["test_valid"]
    unique_ids,inverse = np.unique(ids[valid],return_inverse=True)
    observed = exposure_transform_spectra(cube.reshape(-1,200)[unique_ids],preprocessing).astype(np.float32)
    x_test = np.zeros((len(ids),11,11,40,1),np.float32)
    x_test[...,0][valid] = observed[inverse]
    if not np.isfinite(x_test).all():
        raise ValueError("Nonfinite full test observations")
    coords = geometry["coords_test"]
    y_test = gt[coords[:,0],coords[:,1]].astype(np.int64) - 1
    return {"X_test":x_test,"y_test":y_test,"coords_test":coords,
        "exposure_counts":geometry["exposure_counts"],
        "metadata":dict(metadata,full_test_tensor_sha256=exposure_sha_array(x_test),
                        test_labels_sha256=exposure_sha_array(y_test),test_tensor_created=True,
                        same_full_test_tensor_for_both_arms=True)}


### Code — `gan_exposure_training.py` · phần 1/9: Hợp đồng train và runtime

**Mục đích:** Chuẩn hóa config/data; pin runtime, GPU model, CUDA/cuDNN và float32. Không import TensorFlow trước chọn worker GPU.

**Đầu vào:** tf/keras do worker cấp; config/data. **Đầu ra:** Validated runtime/config.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Hai nhánh có cùng hardware/runtime và hparams?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 1/9 0:3 7eb7055a0cf50444f14dfca90f01b0467be54360206d309403af71b43cb3fae4
"""HyperGAMO training for a paired source-exposure intervention.

TensorFlow and tf_keras are supplied by the caller, after GPU bootstrap.  This
module never imports TensorFlow, fits preprocessing, or evaluates test data.
The architecture, losses, batches and 34-call shared-Adam schedule follow B.
Explicit variable lists and checkpointed dropout RNGs are a new implementation;
the generated masks are not a replay of the historical B TensorFlow trajectory.
"""
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import time
import uuid

import numpy as np

SCHEMA_VERSION = 1
TRAINING_SCHEMA = "exposure_manual_tape_shared_adam_tracked_dropout_v1"
SHAPE = (11, 11, 40, 1)


def _json_hash(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False,
                         separators=(",", ":"), allow_nan=False).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def _atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    temporary.write_text(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                   indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def _read_json(path):
    value = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError("JSON record must be an object")
    return value


def _file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _array_hash(value):
    array = np.asarray(value)
    digest = hashlib.sha256()
    digest.update(str(array.shape).encode("ascii"))
    digest.update(array.dtype.str.encode("ascii"))
    # Avoid making a second copy of the complete train tensor.
    for start in range(0, len(array) if array.ndim else 1, 128):
        block = array[start:start + 128] if array.ndim else array
        digest.update(np.ascontiguousarray(block).tobytes())
    return digest.hexdigest()


def _config(cfg):
    values = {
        "target_c_updates": int(cfg.get("target_c_updates", 8000)),
        "batch": int(cfg.get("batch", cfg.get("batch_size", 100))),
        "latent": int(cfg.get("latent", cfg.get("latent_dim", 100))),
        "learning_rate": float(cfg.get("learning_rate", 2e-4)),
        "beta1": float(cfg.get("beta1", 0.5)),
        "gen_points": int(cfg.get("gen_points", 7)),
        "schema": TRAINING_SCHEMA,
        "training_implementation_sha256": cfg.get("training_implementation_sha256"),
    }
    if (values["target_c_updates"] != 8000 or values["batch"] != 100
            or values["latent"] != 100 or values["gen_points"] != 7
            or values["learning_rate"] != 2e-4 or values["beta1"] != 0.5):
        raise ValueError("The registered experiment fixes 8000 C updates, batch100, "
                         "latent100, 7 points/head, Adam LR2e-4 beta1=.5")
    return values


def _validate_data(data):
    X = data["X_train"]
    y = np.asarray(data["y_train"])
    if (np.asarray(X).dtype != np.float32 or np.asarray(X).shape != (len(y), *SHAPE)
            or y.ndim != 1 or y.dtype.kind not in "iu" or len(y) < 100
            or np.any(y < 0) or np.any(y > 15)):
        raise ValueError("Train arrays must be finite float32 patches and 0-based labels")
    for start in range(0, len(X), 256):
        if not np.isfinite(X[start:start + 256]).all():
            raise ValueError("Train patches contain NaN/Inf")
    counts = np.bincount(y.astype(np.int64), minlength=16)
    if np.any(counts == 0):
        raise ValueError("Every training layout must retain all 16 classes")
    order = np.argsort(counts, kind="stable")
    if np.any(counts[order[-1]] - counts[order[:-1]] <= 0):
        raise ValueError("The registered 15-head schedule needs one strict majority class")
    if "counts" in data and not np.array_equal(counts, data["counts"]):
        raise ValueError("Data class counts do not match labels")
    if "class_order" in data and not np.array_equal(order, data["class_order"]):
        raise ValueError("Class order must be sorted by (count, original class id)")
    coords = np.asarray(data.get("coords_train", data.get("coords")))
    if coords.shape != (len(y), 2) or coords.dtype.kind not in "iu":
        raise ValueError("The ordered train coordinates are required")
    touched = np.asarray(data.get("touched_train", np.zeros(len(y), dtype=bool)), dtype=bool)
    shared = np.asarray(data.get("shared_source_counts_train", np.zeros(len(y), dtype=np.int64)),
                        dtype=np.int64)
    if (touched.shape != (len(y),) or shared.shape != (len(y),)
            or np.any(shared < 0) or np.any(shared > 120)
            or not np.array_equal(touched, shared > 0)):
        raise ValueError("Source-exposure flags must agree with per-patch source counts")
    inverse = np.empty(16, dtype=np.int64)
    inverse[order] = np.arange(16)
    return X, y.astype(np.int64), coords, counts, order, inverse, touched, shared


def _runtime(tf, keras, require_gpu=True):
    devices = tf.config.get_visible_devices("GPU")
    if require_gpu and not devices:
        raise RuntimeError("This experiment requires GPU; enable GPU before running this cell")
    if not hasattr(keras.optimizers, "legacy") or not hasattr(keras.optimizers.legacy, "Adam"):
        raise RuntimeError("tf_keras legacy Adam is required; Keras3 is unsupported")
    keras.mixed_precision.set_global_policy("float32")
    keras.backend.set_image_data_format("channels_last")
    # Failure is a gate, rather than quietly falling back to nondeterministic replay.
    tf.config.experimental.enable_op_determinism()
    hardware = []
    for device in devices:
        details = tf.config.experimental.get_device_details(device)
        hardware.append({"device_name": str(details.get("device_name", "unknown")),
                         "compute_capability": [int(x) for x in details.get("compute_capability", ())]})
    build = tf.sysconfig.get_build_info()
    return {"tensorflow": tf.__version__, "tf_keras": keras.__version__,
            "gpu_names": [device.name for device in devices], "gpu_hardware": hardware,
            "cuda_version": str(build.get("cuda_version", "unknown")),
            "cudnn_version": str(build.get("cudnn_version", "unknown")),
            "float_policy": "float32", "deterministic_ops_requested": True}




### Code — `gan_exposure_training.py` · phần 2/9: Dựng C/D/G và 15 convex heads

**Mục đích:** C/D Conv3D classifier/discriminator, G cho 784 latent features, 15 heads softmax trên supports từng lớp; Dropout RNG trackable.

**Đầu vào:** Train observations, class order, seed. **Đầu ra:** Models, heads, shared Adam.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Supports có chỉ lấy train observations đúng arm? Đây là implementation fresh, không khẳng định byte-identical historical B.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 2/9 0:3 ea2579283c2a350f7ed8ca9fcc9093b8ac03ae4aef963de71723ac6541072820
def _classifier(tf, keras, seed):
    """B classifier, with an explicit trackable dropout stream."""
    L = keras.layers

    class TrackedDropout(L.Layer):
        def __init__(self, rate, rng_seed, **kwargs):
            super().__init__(**kwargs)
            self.rate = float(rate)
            self.rng = tf.random.Generator.from_seed(int(rng_seed))

        def call(self, inputs, training=None):
            def dropped():
                uniform = self.rng.uniform(tf.shape(inputs), dtype=inputs.dtype)
                return inputs * tf.cast(uniform >= self.rate, inputs.dtype) / (1.0 - self.rate)
            if training is None:
                training = False
            if isinstance(training, (bool, np.bool_)):
                return dropped() if training else tf.identity(inputs)
            return tf.cond(tf.cast(training, tf.bool), dropped, lambda: tf.identity(inputs))

    inputs = L.Input(shape=SHAPE)
    x = L.Conv3D(8, (1, 1, 7), activation="relu", padding="same")(inputs)
    x = L.Conv3D(16, (3, 3, 5), activation="relu", padding="same")(x)
    x = L.Conv3D(32, (5, 5, 7), activation="relu", padding="same")(x)
    s = x.shape
    x = L.Reshape((s[1], s[2], s[3] * s[4]))(x)
    x = L.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = L.GlobalAveragePooling2D()(x)
    dropout = TrackedDropout(0.5, seed + 101, name="paired_dropout_C")
    x = dropout(x)
    output = L.Dense(16, activation="softmax", kernel_initializer="he_normal")(x)
    return keras.Model(inputs, output, name="classifier_3d_hypergamo"), dropout


def _build_models(tf, keras, data, cfg, seed):
    X, y_original, _, counts_original, order, inverse, _, _ = _validate_data(data)
    keras.backend.clear_session()
    keras.utils.set_random_seed(int(seed))
    L = keras.layers
    C, c_dropout = _classifier(tf, keras, seed)
    inputs = L.Input(shape=SHAPE)
    x = L.Conv3D(8, (1, 1, 7), activation="relu", padding="same")(inputs)
    x = L.Conv3D(16, (3, 3, 5), activation="relu", padding="same")(x)
    x = L.Conv3D(32, (5, 5, 7), activation="relu", padding="same")(x)
    s = x.shape
    x = L.Reshape((s[1], s[2], s[3] * s[4]))(x)
    x = L.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = L.GlobalAveragePooling2D()(x)
    label = L.Input(shape=(16,))
    x = L.Concatenate()([x, label])
    # Use exactly the same layer implementation, independent stream for D.
    d_dropout = c_dropout.__class__(0.5, seed + 202, name="paired_dropout_D")
    x = d_dropout(x)
    output = L.Dense(1, activation="sigmoid", kernel_initializer="he_normal")(x)
    D = keras.Model([inputs, label], output, name="D")

    noise = L.Input(shape=(cfg["latent"],))
    condition = L.Input(shape=(16,))
    x = L.Concatenate()([noise, condition])
    x = L.Dense(7 * 7 * 256, use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Reshape((7, 7, 256))(x)
    x = L.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    G = keras.Model([noise, condition], L.Flatten()(x), name="G")
    y = inverse[y_original]
    heads = []
    for head_index in range(15):
        support = np.ascontiguousarray(X[y == head_index].reshape(np.sum(y == head_index), -1))
        constant = tf.constant(support, dtype=tf.float32)
        features = L.Input(shape=(784,))
        weights = L.Dense(len(support), activation="softmax", name="support_weights")(features)
        # Bind each support now; do not capture the final loop constant by reference.
        flat = L.Lambda(lambda w, support_constant=constant: tf.matmul(w, support_constant),
                        name="convex_sum")(weights)
        heads.append(keras.Model(features, L.Reshape(SHAPE)(flat), name=f"head_{head_index:02d}"))
    optimizer = keras.optimizers.legacy.Adam(learning_rate=cfg["learning_rate"], beta_1=cfg["beta1"])
    variable_groups = {"C": tuple(C.trainable_variables), "D": tuple(D.trainable_variables),
                       "G": tuple(G.trainable_variables),
                       "heads": [tuple(head.trainable_variables) for head in heads]}
    all_variables = list(variable_groups["C"] + variable_groups["D"] + variable_groups["G"])
    for group in variable_groups["heads"]:
        all_variables.extend(group)
    if len({id(variable) for variable in all_variables}) != len(all_variables):
        raise RuntimeError("Trainable component variable groups unexpectedly overlap")
    if not hasattr(optimizer, "_create_all_weights"):
        raise RuntimeError("Legacy Adam cannot allocate all slots before checkpoint restore")
    optimizer._create_all_weights(all_variables)
    result = {"C": C, "D": D, "G": G, "heads": heads, "optimizer": optimizer,
              "dropout_C": c_dropout, "dropout_D": d_dropout, "variables": variable_groups,
              "order": order, "inverse": inverse, "y": y,
              "counts": counts_original[order], "cfg": cfg}
    result["updates"] = _update_functions(tf, result)
    return result




### Code — `gan_exposure_training.py` · phần 3/9: Update routes và trạng thái mạng

**Mục đích:** Capture đúng danh sách variables cho C/D/GC/GD; hash trainable weights/optimizer/Dropout để kiểm freeze và finite.

**Đầu vào:** Models. **Đầu ra:** Update functions, component/optimizer hashes.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** GC/GD chỉ cập nhật G+head active, C/D frozen weights? Inference BN/RNG bất biến?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 3/9 0:3 a38941410552a96a36b3ee2ec8d5e863c2bb5d4c43b348fa4ac4ddaa2178730d
def _update_functions(tf, models):
    """Capture the exact allowed variables, independent of Keras trainable flags."""
    C, D, G, heads = (models[key] for key in ("C", "D", "G", "heads"))
    groups, optimizer = models["variables"], models["optimizer"]

    def apply(tape, loss, variables):
        gradients = tape.gradient(loss, variables)
        if any(gradient is None for gradient in gradients):
            raise RuntimeError("An active update variable is disconnected from its loss")
        checked = [tf.debugging.check_numerics(gradient, "Nonfinite gradient") for gradient in gradients]
        optimizer.apply_gradients(zip(checked, variables))
        return tf.debugging.check_numerics(loss, "Nonfinite training loss")

    @tf.function(reduce_retracing=True)
    def c_step(X, labels):
        variables = groups["C"]
        with tf.GradientTape(watch_accessed_variables=False) as tape:
            tape.watch(variables)
            probability = C(X, training=True)
            loss = tf.reduce_mean(tf.square(labels - probability))
        return apply(tape, loss, variables)

    @tf.function(reduce_retracing=True)
    def d_step(X, labels, targets):
        variables = groups["D"]
        with tf.GradientTape(watch_accessed_variables=False) as tape:
            tape.watch(variables)
            probability = D([X, labels], training=True)
            loss = tf.reduce_mean(tf.square(targets - probability))
        return apply(tape, loss, variables)

    gc, gd = [], []
    for index in range(15):
        head = heads[index]
        variables = groups["G"] + groups["heads"][index]

        def make_gc(active_head, active_variables):
            @tf.function(reduce_retracing=True)
            def step(noise, labels):
                with tf.GradientTape(watch_accessed_variables=False) as tape:
                    tape.watch(active_variables)
                    fake = active_head(G([noise, labels], training=True), training=True)
                    # Frozen C weights; its dropout is still in training phase,
                    # as in the original compiled composite's global learning phase.
                    probability = C(fake, training=True)
                    loss = tf.reduce_mean(tf.square((1.0 - labels) - probability))
                return apply(tape, loss, active_variables)
            return step

        def make_gd(active_head, active_variables):
            @tf.function(reduce_retracing=True)
            def step(noise, labels):
                with tf.GradientTape(watch_accessed_variables=False) as tape:
                    tape.watch(active_variables)
                    fake = active_head(G([noise, labels], training=True), training=True)
                    probability = D([fake, labels], training=True)
                    loss = tf.reduce_mean(tf.square(1.0 - probability))
                return apply(tape, loss, active_variables)
            return step

        gc.append(make_gc(head, variables))
        gd.append(make_gd(head, variables))
    return {"C": c_step, "D": d_step, "GC": gc, "GD": gd}


def _components(models):
    return [(key, models[key]) for key in ("C", "D", "G")] + [
        (f"head_{index:02d}", head) for index, head in enumerate(models["heads"])]


def _component_hashes(models):
    result = {}
    # A runtime may expose Generator.state through Layer.weights.  It is a
    # sampling stream, not a model parameter; audit it separately below.
    rng_variables = {id(models[key].rng.state) for key in ("dropout_C", "dropout_D")}
    for name, model in _components(models):
        digest = hashlib.sha256()
        for variable in model.weights:
            if id(variable) in rng_variables:
                continue
            digest.update(_array_hash(variable.numpy()).encode("ascii"))
        result[name] = digest.hexdigest()
    return result


def _optimizer_hash(models):
    digest = hashlib.sha256()
    for value in models["optimizer"].get_weights():
        digest.update(_array_hash(value).encode("ascii"))
    return digest.hexdigest()


def _trainable_hashes(models):
    result = {}
    for name, component in _components(models):
        digest = hashlib.sha256()
        for variable in component.trainable_variables:
            digest.update(_array_hash(variable.numpy()).encode("ascii"))
        result[name] = digest.hexdigest()
    return result


def _check_finite(models):
    for name, component in _components(models):
        for variable in component.weights:
            if not np.isfinite(variable.numpy()).all():
                raise FloatingPointError(f"Nonfinite model variable: {name}/{variable.name}")
    for value in models["optimizer"].get_weights():
        if not np.isfinite(value).all():
            raise FloatingPointError("Nonfinite shared Adam state")


def _dropout_state(models):
    return {key: getattr(models[key], "rng").state.numpy().tolist()
            for key in ("dropout_C", "dropout_D")}




### Code — `gan_exposure_training.py` · phần 4/9: Trace exposure độc lập với arm

**Mục đích:** Trace lưu RNG/permutation/cursor và sinh batch real, fake classes theo deficit, noise/smoothing; hai arms dùng cùng trace.

**Đầu vào:** Seed, region, counts/classes. **Đầu ra:** Serializable ExposureTrace.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Fake sinh online theo deficit; không được mô tả chỉ tạo Nmax-Nc một lần. Trace/resume có khớp?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 4/9 0:2 46582edd776e9a011f732806d388ea3a7a0a0cf450b7be3e67d44e46184cdfbf
class ExposureTrace:
    """Arm-independent sampling with a serializable fixed-order batch cursor."""
    def __init__(self, samples, sorted_counts, seed, region_id, state=None):
        self.samples = int(samples)
        self.counts = np.asarray(sorted_counts, dtype=np.int64)
        deficits = self.counts[-1] - self.counts[:-1]
        self.cdf = np.cumsum(deficits / deficits.sum())
        self.cdf[-1] = 1.0
        self.rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(region_id), 314159]))
        self.real_order = self.rng.permutation(self.samples)
        self.loop = 0
        self.trace_sha256 = hashlib.sha256(b"exposure_trace_v1").hexdigest()
        self.real_counts = np.zeros(16, dtype=np.int64)
        self.fake_counts = np.zeros(16, dtype=np.int64)
        self.touched_real_presentations = 0
        self.potential_shared_raw_source_occurrences = 0
        if state is not None:
            self.restore(state)

    def restore(self, state):
        self.rng.bit_generator.state = state["rng_state"]
        self.real_order = np.asarray(state["real_order"], dtype=np.int64)
        self.loop = int(state["loop"])
        self.trace_sha256 = state["trace_sha256"]
        self.real_counts = np.asarray(state["real_counts"], dtype=np.int64)
        self.fake_counts = np.asarray(state["fake_counts"], dtype=np.int64)
        self.touched_real_presentations = int(state["touched_real_presentations"])
        self.potential_shared_raw_source_occurrences = int(state["potential_shared_raw_source_occurrences"])
        if (self.loop < 0 or not np.array_equal(np.sort(self.real_order), np.arange(self.samples))
                or self.real_counts.shape != (16,) or self.fake_counts.shape != (16,)
                or np.any(self.real_counts < 0) or np.any(self.fake_counts < 0)
                or self.fake_counts[-1] != 0 or self.touched_real_presentations < 0
                or self.potential_shared_raw_source_occurrences < 0):
            raise RuntimeError("Recovery sampling state is invalid")
        expected = self.expected_presentations()
        if self.real_counts.sum() != expected or self.fake_counts.sum() != expected:
            raise RuntimeError("Recovery presentation counts do not match completed loops")

    def expected_presentations(self):
        batches = int(math.ceil(self.samples / 100))
        cycles, partial = divmod(self.loop, batches)
        return cycles * self.samples + min(partial * 100, self.samples)

    def next(self):
        batches = int(math.ceil(self.samples / 100))
        batch_index = self.loop % batches
        indices = self.real_order[batch_index * 100:min((batch_index + 1) * 100, self.samples)]
        size = len(indices)
        record = {"indices": indices,
                  "valid": (1.0 - self.rng.uniform(0, .1, (size, 1))).astype(np.float32)}
        record["fake_ids"] = np.searchsorted(self.cdf, self.rng.random(size), side="right")
        record["noise"] = self.rng.normal(size=(size, 100)).astype(np.float32)
        record["invalid"] = self.rng.uniform(0, .1, (size, 1)).astype(np.float32)
        record["head_noise"] = [self.rng.normal(size=(7, 100)).astype(np.float32) for _ in range(15)]
        return record

    def commit(self, record, y_sorted, touched, shared, dropout_states=None):
        self.real_counts += np.bincount(y_sorted[record["indices"]], minlength=16)
        self.fake_counts += np.bincount(record["fake_ids"], minlength=16)
        self.touched_real_presentations += int(touched[record["indices"]].sum())
        self.potential_shared_raw_source_occurrences += int(shared[record["indices"]].sum())
        digest = hashlib.sha256(bytes.fromhex(self.trace_sha256))
        digest.update(str(self.loop).encode("ascii"))
        for key in ("indices", "valid", "fake_ids", "noise", "invalid"):
            digest.update(key.encode("ascii"))
            digest.update(_array_hash(record[key]).encode("ascii"))
        for noise in record["head_noise"]:
            digest.update(_array_hash(noise).encode("ascii"))
        if dropout_states is not None:
            digest.update(_json_hash(dropout_states).encode("ascii"))
        self.trace_sha256 = digest.hexdigest()
        self.loop += 1

    def state(self):
        return {"rng_state": self.rng.bit_generator.state, "real_order": self.real_order.tolist(),
                "loop": self.loop, "trace_sha256": self.trace_sha256,
                "real_counts": self.real_counts.tolist(), "fake_counts": self.fake_counts.tolist(),
                "touched_real_presentations": self.touched_real_presentations,
                "potential_shared_raw_source_occurrences": self.potential_shared_raw_source_occurrences}



### Code — `gan_exposure_training.py` · phần 5/9: Một vòng GAN và lưu/restore

**Mục đích:** Một loop: 2 C updates; đủ 15 head routes; 34 calls shared Adam. Checkpoint chứa C/D/G/heads/Adam/Dropout và trace, commit atomic.

**Đầu vào:** Models, X_train, Trace. **Đầu ra:** Weights/state/history/checkpoint.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** 8000 C updates =4000 loops =136000 Adam calls; checkpoint có đủ optimizer/RNG, không chỉ .h5?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 5/9 1:2 0819ee64e77d956c16257f2c7e2adcf9316d3839dc1e4306f35dffef4dfb9e20

def _one_loop(tf, models, X, trace, touched, shared):
    record = trace.next()
    indices = record["indices"]
    labels = np.eye(16, dtype=np.float32)[models["y"][indices]]
    step = models["updates"]
    losses = []
    losses.append(float(step["C"](X[indices], labels).numpy()))
    losses.append(float(step["D"](X[indices], labels, record["valid"]).numpy()))
    fake_ids = record["fake_ids"]
    fake_labels = np.eye(16, dtype=np.float32)[fake_ids]
    # Inference generation does not update BN state or consume dropout streams.
    features = models["G"]([record["noise"], fake_labels], training=False)
    fake = np.empty((len(indices), *SHAPE), dtype=np.float32)
    for head_index, head in enumerate(models["heads"]):
        selected = np.flatnonzero(fake_ids == head_index)
        if len(selected):
            fake[selected] = head(tf.gather(features, selected), training=False).numpy()
    if not np.isfinite(fake).all():
        raise FloatingPointError("Generated training patches are nonfinite")
    losses.append(float(step["C"](fake, fake_labels).numpy()))
    losses.append(float(step["D"](fake, fake_labels, record["invalid"]).numpy()))
    gc, gd = [], []
    for head_index in range(15):
        condition = np.zeros((7, 16), dtype=np.float32)
        condition[:, head_index] = 1
        noise = record["head_noise"][head_index]
        gc.append(float(step["GC"][head_index](noise, condition).numpy()))
        gd.append(float(step["GD"][head_index](noise, condition).numpy()))
    values = np.asarray([(losses[0] + losses[2]) / 2,
                         (losses[1] + losses[3]) / 2, np.mean(gc), np.mean(gd)])
    if not np.isfinite(values).all():
        raise FloatingPointError("Training loss is nonfinite")
    trace.commit(record, models["y"], touched, shared, _dropout_state(models))
    clock = int(models["optimizer"].iterations.numpy())
    if clock != 34 * trace.loop:
        raise RuntimeError(f"Expected 34 shared Adam calls per loop; got clock {clock}")
    return values


def _checkpoint(tf, models):
    objects = {key: models[key] for key in ("C", "D", "G")}
    objects.update({f"head_{index:02d}": head for index, head in enumerate(models["heads"])})
    objects["optimizer"] = models["optimizer"]
    objects["dropout_rng_C"] = models["dropout_C"].rng
    objects["dropout_rng_D"] = models["dropout_D"].rng
    return tf.train.Checkpoint(**objects)


def _confined(jobdir, relative):
    relative = Path(relative)
    if relative.is_absolute() or ".." in relative.parts:
        raise RuntimeError("Recovery references a path outside its job directory")
    result = (Path(jobdir) / relative).resolve()
    result.relative_to(Path(jobdir).resolve())
    return result


def _restore(jobdir, checkpoint, state, contract=None):
    if contract is not None and (state.get("contract") != contract
                                 or state.get("contract_sha256") != _json_hash(contract)):
        raise RuntimeError("Saved training contract does not match this job")
    record = state.get("checkpoint", {})
    prefix = _confined(jobdir, record.get("prefix", ""))
    pieces = record.get("files", {})
    expected = {path.resolve() for path in prefix.parent.glob(prefix.name + ".*")
                if path.name == prefix.name + ".index" or path.name.startswith(prefix.name + ".data-")}
    if (not pieces or not any(Path(relative).name == prefix.name + ".index" for relative in pieces)
            or not any(Path(relative).name.startswith(prefix.name + ".data-") for relative in pieces)
            or expected != {_confined(jobdir, relative) for relative in pieces}):
        raise RuntimeError("Checkpoint index/data pieces do not match the committed manifest")
    for relative, digest in pieces.items():
        path = _confined(jobdir, relative)
        if not path.is_file() or _file_hash(path) != digest:
            raise RuntimeError(f"Checkpoint piece is missing or changed: {path}")
    # .write/.read deliberately exclude save_counter; assert_consumed catches
    # missing slots, head weights, BN state or explicit dropout RNGs.
    checkpoint.read(str(prefix)).assert_consumed()


def _save(jobdir, checkpoint, models, state):
    save_started = time.perf_counter()
    _check_finite(models)
    directory = Path(jobdir) / "recovery" / "checkpoints"
    directory.mkdir(parents=True, exist_ok=True)
    prefix = directory / f"u{state['updates']:06d}_{uuid.uuid4().hex}"
    checkpoint.write(str(prefix))
    pieces = sorted(path for path in directory.glob(prefix.name + ".*")
                    if path.name == prefix.name + ".index" or path.name.startswith(prefix.name + ".data-"))
    if not any(path.suffix == ".index" for path in pieces) or not any(".data-" in path.name for path in pieces):
        raise RuntimeError("TensorFlow did not write complete checkpoint pieces")
    payload = dict(state)
    payload["checkpoint"] = {"prefix": prefix.relative_to(jobdir).as_posix(),
                              "files": {path.relative_to(jobdir).as_posix(): _file_hash(path)
                                        for path in pieces}}
    state_path = Path(jobdir) / "recovery" / "state.json"
    previous = _read_json(state_path).get("checkpoint", {}) if state_path.is_file() else None
    if "elapsed_seconds" in payload:
        payload["elapsed_seconds"] = float(payload["elapsed_seconds"]) + time.perf_counter() - save_started
    _atomic_json(state_path, payload)
    # Only prune after publishing the complete new checkpoint.  Keep its
    # immediate predecessor too; an interrupted write cannot invalidate resume.
    keep = {str(prefix.resolve())}
    if previous and previous.get("prefix"):
        keep.add(str(_confined(jobdir, previous["prefix"])))
    for index in directory.glob("u*.index"):
        old_prefix = index.with_suffix("").resolve()
        if str(old_prefix) not in keep:
            for piece in directory.glob(old_prefix.name + ".*"):
                if piece.name == old_prefix.name + ".index" or piece.name.startswith(old_prefix.name + ".data-"):
                    piece.unlink()
    return payload


def _history_csv(jobdir, history):
    if not history:
        return
    path = Path(jobdir) / "training_history.csv"
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    temporary.replace(path)



### Code — `gan_exposure_training.py` · phần 6/9: Pin khởi tạo và dữ liệu của cặp

**Mục đích:** Pin atomic common initialization và invariants để hai arms không lệch seed/data/runtime/architecture ngoài intervention.

**Đầu vào:** Job contract và initialization hashes. **Đầu ra:** Immutable pair pin.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Pair pin có phân biệt điều phải giống với train-X cần khác?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 6/9 1:3 cbc72114eca32f2d936746014df99219c8ee256e44a342c09ae8adb54d3d3842

def _data_contract(data):
    X, y, coords, counts, order, _, touched, shared = _validate_data(data)
    return {"train_shape": list(X.shape), "train_array_sha256": _array_hash(X),
            "train_labels_sha256": _array_hash(y), "train_coords_sha256": _array_hash(coords),
            "counts": counts.tolist(), "class_order": order.tolist(),
            "touched_train_sha256": _array_hash(touched),
            "shared_source_counts_sha256": _array_hash(shared)}


def _pair_data_invariants(data, data_contract):
    """Common provenance; arm-dependent train values are deliberately excluded."""
    metadata = data.get("metadata")
    if not isinstance(metadata, dict):
        raise RuntimeError("Prepared region metadata is required before pinning paired initialization")
    common = metadata.get("pair_invariants")
    required = ("train_coords_sha256", "train_labels_sha256", "train_class_counts_sha256",
                "class_order_sha256", "preprocessing_sha256", "test_exposure_counts_sha256")
    if not isinstance(common, dict) or set(common) != set(required):
        raise RuntimeError("Prepared metadata lacks the registered common pair invariants")
    for key in required:
        value = common[key]
        if (not isinstance(value, str) or len(value) != 64
                or any(character not in "0123456789abcdef" for character in value)
                or metadata.get(key) != value):
            raise RuntimeError(f"Invalid or inconsistent prepared pair invariant: {key}")
    return {
        "actual_train_coords_sha256": data_contract["train_coords_sha256"],
        "actual_train_labels_sha256": data_contract["train_labels_sha256"],
        "train_shape": data_contract["train_shape"],
        "counts": data_contract["counts"], "class_order": data_contract["class_order"],
        "touched_train_sha256": data_contract["touched_train_sha256"],
        "shared_source_counts_sha256": data_contract["shared_source_counts_sha256"],
        "prepared_pair_invariants": {key: common[key] for key in required},
    }


def _pair_pin_record(initialization, contract):
    if (initialization.get("combined_sha256") != _json_hash(initialization.get("component_sha256"))
            or initialization.get("class_order") != contract["data"]["class_order"]
            or initialization.get("seed") != contract["seed"]
            or initialization.get("region_id") != contract["region_id"]
            or not isinstance(initialization.get("dropout_initial_state"), dict)
            or set(initialization["dropout_initial_state"]) != {"dropout_C", "dropout_D"}):
        raise RuntimeError("Initialization is incomplete or inconsistent with the paired job")
    if not isinstance(contract.get("pair_data_invariants"), dict):
        raise RuntimeError("Training contract does not contain common pair provenance")
    return {"schema_version": SCHEMA_VERSION,
            "purpose": "first arm establishes immutable paired initialization; counterpart verifies before study updates",
            "region_id": contract["region_id"], "seed": contract["seed"],
            "training": contract["training"], "runtime": contract["runtime"],
            "pair_data_invariants": contract["pair_data_invariants"],
            "initial_weights_sha256": initialization["combined_sha256"],
            "initial_component_sha256": initialization["component_sha256"],
            "initial_dropout_state": initialization["dropout_initial_state"],
            "initial_dropout_state_sha256": _json_hash(initialization["dropout_initial_state"]),
            "class_order": initialization["class_order"],
            "readiness_barrier": False}


def _pair_pin_path(jobdir, contract):
    # Both arm directories are children of this suite's jobs directory.
    return (Path(jobdir).resolve().parent / "pair_initialization"
            / f"region{int(contract['region_id']):02d}_seed{int(contract['seed']):02d}.json")


def _verify_pair_pin(jobdir, initialization, contract):
    expected = _pair_pin_record(initialization, contract)
    path = _pair_pin_path(jobdir, contract)
    if not path.is_file():
        raise RuntimeError("Paired initialization pin is missing; no study updates are authorized")
    try:
        saved = _read_json(path)
    except (OSError, ValueError) as error:
        raise RuntimeError(f"Paired initialization pin is unreadable: {path}") from error
    if saved != expected:
        raise RuntimeError("Paired initialization/provenance mismatch; stop before study updates")
    return {"path": str(path), "sha256": _file_hash(path), "record_sha256": _json_hash(saved)}


def _pin_pair_initialization(jobdir, initialization, contract):
    """Publish a complete immutable pin atomically, without an exists/write race.

    An exclusive lock protects initial publication.  A fully-written/fsynced
    temporary file is atomically renamed only while holding that lock and only
    if no pin exists.  Concurrent writers either establish the same pin or
    reject a differing winner; neither can replace it.  A crashed writer's
    stale lock fails closed after a bounded wait.  The first arm need not wait
    for its counterpart, but both check before study updates.
    """
    record = _pair_pin_record(initialization, contract)
    path = _pair_pin_path(jobdir, contract)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        return _verify_pair_pin(jobdir, initialization, contract)
    lock_path = path.with_name(path.name + ".lock")
    for _ in range(100):
        try:
            lock = lock_path.open("x", encoding="ascii")
        except FileExistsError:
            if path.exists():
                return _verify_pair_pin(jobdir, initialization, contract)
            time.sleep(0.05)
            continue
        temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
        try:
            with lock:
                lock.write(uuid.uuid4().hex + "\n")
                lock.flush()
                os.fsync(lock.fileno())
            if not path.exists():
                with temporary.open("x", encoding="utf-8", newline="\n") as stream:
                    stream.write(json.dumps(record, sort_keys=True, ensure_ascii=False,
                                            indent=2, allow_nan=False) + "\n")
                    stream.flush()
                    os.fsync(stream.fileno())
                # All legitimate publishers hold this same exclusive lock.
                # No existing immutable pin is ever replaced by this branch.
                temporary.replace(path)
            return _verify_pair_pin(jobdir, initialization, contract)
        finally:
            if temporary.exists():
                temporary.unlink()
            lock_path.unlink()
    raise RuntimeError("Paired initialization publication is locked or incomplete; "
                       "stop without updates and inspect the stale pin lock before retrying")




### Code — `gan_exposure_training.py` · phần 7/9: Train GPU và resume đủ ngân sách

**Mục đích:** Train fresh hoặc resume đến absolute stop/target; checkpoint định kỳ, không validation/test và không chọn best epoch.

**Đầu vào:** Validated data/runtime, jobdir, cfg, seed/region/arm. **Đầu ra:** recovery/state.json, history, done.json khi đủ8000.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Benchmark400 tính trong8000. Chỉ status.json không đủ; authoritative recovery/state phải hợp hash.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 7/9 0:3 58c8037a521623fb1591bbe4df5b099f652ac7bbf05bac835f46598f955b2461
def exposure_train_job(tf, keras, data, jobdir, cfg, seed, region_id, arm,
                       max_c_updates=None, save_every=250, wall_seconds=None):
    """Train/resume a job; max_c_updates is an absolute temporary stop.

    The scientific target stays 8000 in the contract.  A loop is indivisible:
    an interruption before commit replays that loop from the previous checkpoint.
    No test/validation calls occur here.  Return JSON-safe committed status.
    """
    normalized = _config(cfg)
    runtime = _runtime(tf, keras)
    if (arm not in ("blocked", "reuse_control") or int(region_id) not in (1, 2, 3)
            or int(seed) not in (42, 43, 44, 45, 46)):
        raise ValueError("Arm/region is not part of the registered experiment")
    stop = normalized["target_c_updates"] if max_c_updates is None else int(max_c_updates)
    if stop < 0 or stop > 8000 or stop % 2 or save_every <= 0 or save_every % 2:
        raise ValueError("Stop and save cadence must be even; stop is within the fixed 8000-update target")
    if wall_seconds is not None and (not np.isfinite(wall_seconds) or wall_seconds <= 0):
        raise ValueError("wall_seconds must be a positive finite duration")
    jobdir = Path(jobdir).resolve()
    jobdir.mkdir(parents=True, exist_ok=True)
    state_path = jobdir / "recovery" / "state.json"
    if state_path.exists():
        inspected = exposure_inspect_job(jobdir, cfg=cfg, data=data)
        current = inspected["c_updates"]
        # A complete marker/export is needed at target; a crash just before
        # publishing them falls through to restore-and-finalize below.
        if current >= stop and (current < 8000 or inspected["complete"]):
            return {"jobdir": str(jobdir), "complete": inspected["complete"],
                    "stop_reason": "complete" if inspected["complete"] else "already_at_requested_stop",
                    "classifier_updates": current, "optimizer_clock": inspected["optimizer_iterations"],
                    "initial_component_sha256": inspected["initial_weights_sha256"],
                    "trace_sha256": inspected["trace_sha256"], "state": inspected["state"],
                    "session_completed_loops": 0,
                    "final_component_sha256": (_read_json(jobdir / "done.json").get("final_component_sha256")
                                               if inspected["complete"] else None),
                    "test_evaluated": False}
    # Every GPU job must pass the integration gate before its first training
    # update.  A valid cached report makes this a cheap contract/hash check.
    smoke_dir = Path(cfg.get("gpu_smoke_dir", jobdir / "gpu_smoke"))
    exposure_gpu_smoke(tf, keras, data, cfg, smoke_dir)
    X, y, coords, counts, order, _, touched, shared = _validate_data(data)
    contract = {"schema_version": SCHEMA_VERSION, "training": normalized, "runtime": runtime,
                "seed": int(seed), "region_id": int(region_id), "arm": arm,
                "data": _data_contract(data), "selection": "fixed final budget; no validation/test selection",
                "schedule": {"real_batch_order": "one fixed permutation; tail retained; no epoch reshuffle",
                             "fake_class": "majority-minus-class deficits over the 15 minority classes",
                             "C_calls_per_loop": 2, "D_calls_per_loop": 2,
                             "G_vs_C_calls_per_loop": 15, "G_vs_D_calls_per_loop": 15,
                             "shared_Adam_calls_per_loop": 34},
                "resume": "full components, Adam slots/clock, explicit dropout generators, NumPy RNG/cursor"}
    contract["pair_data_invariants"] = _pair_data_invariants(data, contract["data"])
    contract_path = jobdir / "training_contract.json"
    if contract_path.is_file() and _read_json(contract_path) != contract:
        raise RuntimeError("Existing job directory belongs to a different contract; do not overwrite it")
    if not contract_path.exists():
        _atomic_json(contract_path, contract)
    models = _build_models(tf, keras, data, normalized, int(seed))
    initial_hashes = _component_hashes(models)
    initial_record = {"component_sha256": initial_hashes, "combined_sha256": _json_hash(initial_hashes),
                      "seed": int(seed), "region_id": int(region_id), "class_order": order.tolist(),
                      "fixed_supports_excluded": True, "dropout_initial_state": _dropout_state(models)}
    initial_path = jobdir / "initialization.json"
    if initial_path.is_file() and _read_json(initial_path) != initial_record:
        raise RuntimeError("Fresh initialization does not match the saved job")
    if not initial_path.exists():
        _atomic_json(initial_path, initial_record)
    pair_pin = _pin_pair_initialization(jobdir, initial_record, contract)
    checkpoint = _checkpoint(tf, models)
    trace = ExposureTrace(len(X), counts[order], seed, region_id)
    history, recent, recent_n, elapsed_base = [], np.zeros(4), 0, 0.0
    if state_path.exists():
        state = _read_json(state_path)
        _restore(jobdir, checkpoint, state, contract)
        trace.restore(state["trace"])
        history = list(state["history"])
        recent = np.asarray(state["recent"], dtype=np.float64)
        recent_n = int(state["recent_n"])
        elapsed_base = float(state["elapsed_seconds"])
        if (state["initial_component_sha256"] != initial_record["combined_sha256"]
                or state["updates"] != 2 * trace.loop
                or state["optimizer_clock"] != 34 * trace.loop
                or int(models["optimizer"].iterations.numpy()) != 34 * trace.loop
                or _dropout_state(models) != state["dropout_state"]
                or recent.shape != (4,) or not np.isfinite(recent).all()
                or recent_n < 0 or not np.isfinite(elapsed_base) or elapsed_base < 0):
            raise RuntimeError("Recovered optimizer/RNG/history counters are inconsistent")
    elif (jobdir / "done.json").exists():
        raise RuntimeError("A completion marker without recovery state is not a resumable job")
    if 2 * trace.loop > 8000:
        raise RuntimeError("Recovery exceeds the registered target")
    started = time.perf_counter()
    initial_loop = trace.loop

    def capture():
        return {"schema_version": SCHEMA_VERSION, "contract": contract, "contract_sha256": _json_hash(contract),
                "updates": 2 * trace.loop, "loop": trace.loop, "optimizer_clock": 34 * trace.loop,
                "initial_component_sha256": initial_record["combined_sha256"], "trace": trace.state(),
                "pair_initialization_sha256": pair_pin["sha256"],
                "dropout_state": _dropout_state(models), "history": history,
                "actual_shared_raw_source_occurrences_presented": (
                    trace.potential_shared_raw_source_occurrences if arm == "reuse_control" else 0),
                "recent": recent.tolist(), "recent_n": recent_n,
                "elapsed_seconds": elapsed_base + time.perf_counter() - started}

    if not state_path.exists():
        state = _save(jobdir, checkpoint, models, capture())
    stop_reason = "requested_stop"
    last_history_loop = trace.loop
    last_history_time = time.perf_counter()
    session_intervals = 0
    while 2 * trace.loop < stop:
        loop_start = time.perf_counter()
        recent += _one_loop(tf, models, X, trace, touched, shared)
        recent_n += 1
        updates = 2 * trace.loop
        timed_out = wall_seconds is not None and time.perf_counter() - started >= wall_seconds
        if updates % save_every == 0 or updates == stop or timed_out:
            seconds = time.perf_counter() - last_history_time
            completed_loops = trace.loop - last_history_loop
            row = {"classifier_updates": updates, "optimizer_clock": 34 * trace.loop,
                   "C_loss": float(recent[0] / recent_n), "D_loss": float(recent[1] / recent_n),
                   "G_vs_C_loss": float(recent[2] / recent_n), "G_vs_D_loss": float(recent[3] / recent_n),
                   "elapsed_seconds": elapsed_base + time.perf_counter() - started,
                   "interval_seconds": seconds, "interval_loops": completed_loops,
                   "warmup_interval": session_intervals == 0,
                   "seconds_per_loop": seconds / completed_loops,
                   "last_loop_seconds": time.perf_counter() - loop_start,
                   "touched_real_presentations": trace.touched_real_presentations,
                   "potential_shared_raw_source_occurrences": trace.potential_shared_raw_source_occurrences,
                   "actual_shared_raw_source_occurrences_presented": (
                       trace.potential_shared_raw_source_occurrences if arm == "reuse_control" else 0)}
            history.append(row)
            session_intervals += 1
            recent[:] = 0
            recent_n = 0
            state = _save(jobdir, checkpoint, models, capture())
            _history_csv(jobdir, history)
            print(f"region{region_id} seed{seed} {arm}: {updates}/8000 C updates; "
                  f"Adam={34 * trace.loop}; interval={seconds / completed_loops:.3f}s/loop", flush=True)
            last_history_loop, last_history_time = trace.loop, time.perf_counter()
        if timed_out:
            stop_reason = "wall_seconds"
            break
    # Even a no-op/earlier stop reads the last atomic checkpoint, never uncommitted state.
    state = _read_json(state_path)
    complete = int(state["updates"]) == 8000
    if complete:
        _check_finite(models)
        final_path = jobdir / "C_final.weights.h5"
        if not final_path.exists():
            temporary = jobdir / ("C_final." + uuid.uuid4().hex + ".weights.h5")
            models["C"].save_weights(str(temporary))
            temporary.replace(final_path)
        final_record = {"complete": True, "contract_sha256": _json_hash(contract),
                        "classifier_updates": 8000, "optimizer_clock": 136000,
                        "initial_component_sha256": initial_record["combined_sha256"],
                        "class_order": order.tolist(), "trace_sha256": state["trace"]["trace_sha256"],
                        "checkpoint": state["checkpoint"], "final_classifier_sha256": _file_hash(final_path),
                        "final_component_sha256": _component_hashes(models), "test_evaluated": False}
        done_path = jobdir / "done.json"
        if done_path.exists() and _read_json(done_path) != final_record:
            raise RuntimeError("Existing final completion record does not match the checkpoint")
        if not done_path.exists():
            _atomic_json(done_path, final_record)
        stop_reason = "complete"
    status = {"jobdir": str(jobdir), "complete": complete, "stop_reason": stop_reason,
              "classifier_updates": int(state["updates"]), "optimizer_clock": int(state["optimizer_clock"]),
              "initial_component_sha256": initial_record["combined_sha256"],
              "trace_sha256": state["trace"]["trace_sha256"], "state": state,
              "session_completed_loops": trace.loop - initial_loop,
              "final_component_sha256": _component_hashes(models) if complete else None,
              "test_evaluated": False}
    _atomic_json(jobdir / "status.json", {
        "schema_version": SCHEMA_VERSION, "complete": complete, "completed": complete,
        "c_updates": int(state["updates"]), "optimizer_iterations": int(state["optimizer_clock"]),
        "initial_weights_sha256": initial_record["combined_sha256"],
        "trace_sha256": state["trace"]["trace_sha256"], "stop_reason": stop_reason,
        "checkpoint": state["checkpoint"], "recovery_state": "recovery/state.json",
        "contract_sha256": _json_hash(contract), "test_evaluated": False})
    del models, checkpoint
    keras.backend.clear_session()
    return status




### Code — `gan_exposure_training.py` · phần 8/9: Kiểm completion và nạp C cuối

**Mục đích:** Kiểm committed checkpoint và restore classifier final; cung cấp logits và class order cho test đầy16 outputs.

**Đầu vào:** Job artifacts complete. **Đầu ra:** Verified status / C final logits.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Không test partial checkpoint; mapping sorted class order về IDs1..16 đúng?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 8/9 0:3 a311800d957fc7e73abc87f2b8ae4f10145475500376ecbd8036342bb793290a
def exposure_inspect_job(jobdir, cfg=None, data=None, require_complete=False):
    """Read and validate committed job state and piece hashes without TensorFlow."""
    jobdir = Path(jobdir).resolve()
    state_path = jobdir / "recovery" / "state.json"
    if not state_path.exists():
        if require_complete:
            raise RuntimeError("A required job has no committed checkpoint")
        return {"jobdir": str(jobdir), "exists": False, "complete": False,
                "completed": False, "c_updates": 0, "optimizer_iterations": 0}
    state = _read_json(state_path)
    contract = state.get("contract", {})
    if (state.get("schema_version") != SCHEMA_VERSION
            or state.get("contract_sha256") != _json_hash(contract)
            or _read_json(jobdir / "training_contract.json") != contract):
        raise RuntimeError("Committed state and job contract do not match")
    if cfg is not None and contract.get("training") != _config(cfg):
        raise RuntimeError("Saved job training configuration does not match")
    if data is not None and contract.get("data") != _data_contract(data):
        raise RuntimeError("Saved job training data does not match")
    updates, loop = int(state.get("updates", -1)), int(state.get("loop", -1))
    if not (0 <= updates <= 8000 and updates == 2 * loop
            and state.get("optimizer_clock") == 34 * loop
            and state.get("trace", {}).get("loop") == loop):
        raise RuntimeError("Committed update/optimizer/trace counters are inconsistent")
    data_contract = contract["data"]
    counts = np.asarray(data_contract["counts"], dtype=np.int64)
    order = np.asarray(data_contract["class_order"], dtype=np.int64)
    ExposureTrace(data_contract["train_shape"][0], counts[order], contract["seed"],
                  contract["region_id"], state=state["trace"])
    record = state.get("checkpoint", {})
    prefix = _confined(jobdir, record.get("prefix", ""))
    pieces = record.get("files", {})
    actual = {p.resolve() for p in prefix.parent.glob(prefix.name + ".*")
              if p.name == prefix.name + ".index" or p.name.startswith(prefix.name + ".data-")}
    if (not pieces or actual != {_confined(jobdir, relative) for relative in pieces}
            or not any(Path(relative).name == prefix.name + ".index" for relative in pieces)
            or not any(Path(relative).name.startswith(prefix.name + ".data-") for relative in pieces)):
        raise RuntimeError("Checkpoint pieces do not match the committed state")
    for relative, expected in pieces.items():
        path = _confined(jobdir, relative)
        if _file_hash(path) != expected:
            raise RuntimeError(f"Checkpoint hash does not match: {path}")
    initialization = _read_json(jobdir / "initialization.json")
    if (initialization.get("combined_sha256") != _json_hash(initialization.get("component_sha256"))
            or state.get("initial_component_sha256") != initialization.get("combined_sha256")):
        raise RuntimeError("Initialization record does not match checkpoint state")
    pair_pin = _verify_pair_pin(jobdir, initialization, contract)
    if state.get("pair_initialization_sha256") != pair_pin["sha256"]:
        raise RuntimeError("Paired initialization pin does not match committed checkpoint state")
    complete = updates == 8000 and (jobdir / "done.json").exists()
    if complete:
        done = _read_json(jobdir / "done.json")
        final_path = jobdir / "C_final.weights.h5"
        if (done.get("complete") is not True or done.get("checkpoint") != record
                or done.get("contract_sha256") != state["contract_sha256"]
                or done.get("trace_sha256") != state["trace"]["trace_sha256"]
                or not final_path.exists() or _file_hash(final_path) != done.get("final_classifier_sha256")):
            raise RuntimeError("Final classifier/manifest does not match committed checkpoint")
    if require_complete and not complete:
        raise RuntimeError("Job is not complete at the fixed 8000-update target")
    return {"jobdir": str(jobdir), "exists": True, "complete": complete, "completed": complete,
            "c_updates": updates, "optimizer_iterations": 34 * loop,
            "initial_weights_sha256": initialization["combined_sha256"],
            "initial_component_sha256": initialization["component_sha256"],
            "trace_sha256": state["trace"]["trace_sha256"], "checkpoint": record,
            "class_order": order.tolist(), "state": state, "test_evaluated": False}


def exposure_load_classifier(tf, keras, jobdir, cfg, data=None):
    """Restore only the committed final C; return logits in sorted class order.

    Return keys: C, logits (callable), order, state.  The caller maps columns via
    order before computing original-class metrics.  Stable NLL uses log_softmax
    of logits, rather than clipping float32 softmax probabilities.
    """
    normalized = _config(cfg)
    _runtime(tf, keras, require_gpu=False)
    jobdir = Path(jobdir).resolve()
    state = _read_json(jobdir / "recovery" / "state.json")
    done = _read_json(jobdir / "done.json")
    contract = state["contract"]
    if (contract["training"] != normalized or done.get("complete") is not True
            or state.get("updates") != 8000 or state.get("optimizer_clock") != 136000
            or done.get("contract_sha256") != _json_hash(contract)
            or done.get("checkpoint") != state.get("checkpoint")):
        raise RuntimeError("Only the completed registered final checkpoint can be evaluated")
    if data is not None and _data_contract(data) != contract["data"]:
        raise RuntimeError("Classifier data contract does not match this arm")
    final_path = jobdir / "C_final.weights.h5"
    if not final_path.is_file() or _file_hash(final_path) != done["final_classifier_sha256"]:
        raise RuntimeError("Exported final classifier is missing or changed")
    keras.backend.clear_session()
    keras.utils.set_random_seed(int(contract["seed"]))
    C, dropout = _classifier(tf, keras, int(contract["seed"]))
    C.load_weights(str(final_path))
    dense = C.layers[-1]
    feature_model = keras.Model(C.input, dense.input)

    def logits(inputs, training=False):
        if training:
            raise ValueError("The evaluation logits accessor is inference-only")
        features = feature_model(inputs, training=False)
        return tf.linalg.matmul(features, dense.kernel) + dense.bias

    probe = np.zeros((2, *SHAPE), dtype=np.float32)
    np.testing.assert_allclose(tf.nn.softmax(logits(probe)).numpy(), C(probe, training=False).numpy(),
                               rtol=1e-6, atol=1e-7)
    order = np.asarray(contract["data"]["class_order"], dtype=np.int64)
    if not np.array_equal(np.sort(order), np.arange(16)):
        raise RuntimeError("Saved class order is invalid")
    return {"C": C, "classifier": C, "logits": logits, "order": order,
            "class_order": order, "state": state}




### Code — `gan_exposure_training.py` · phần 9/9: GPU smoke bắt buộc

**Mục đích:** Kiểm update/freeze từng route, đủ15heads, finite, inference bất biến, exact checkpoint continuation và34Adamclock trước training nghiên cứu.

**Đầu vào:** Fresh runtime/models/data. **Đầu ra:** gpu_smoke.json và test recovery.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Cache smoke gắn đúng region/arm/runtime/input hash?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_training.py 9/9 0:1 8ec7d94310fe6ca755108d80af748864d75d434208ec64b27c77add1b12c16ab
def exposure_gpu_smoke(tf, keras, region_data, cfg, out):
    """Mandatory GPU integration gate: frozen routes and exact resume replay."""
    normalized = _config(cfg)
    runtime = _runtime(tf, keras)
    X, y, _, counts, order, _, touched, shared = _validate_data(region_data)
    out = Path(out).resolve()
    out.mkdir(parents=True, exist_ok=True)
    contract = {"training": normalized, "runtime": runtime, "counts": counts.tolist(),
                "class_order": order.tolist(), "data": _data_contract(region_data)}
    report_path = out / "gpu_smoke.json"
    if report_path.exists():
        report = _read_json(report_path)
        if report.get("passed") is True and report.get("contract") == contract:
            return report
        raise RuntimeError("Existing GPU smoke report belongs to another contract or did not pass")
    seed, region_id = 42, 1
    models = _build_models(tf, keras, region_data, normalized, seed)
    initial_hashes = _component_hashes(models)
    initial_trainable_hashes = _trainable_hashes(models)
    labels = np.eye(16, dtype=np.float32)[models["y"][:7]]
    noise = np.random.default_rng(991).normal(size=(7, 100)).astype(np.float32)
    freeze_checks = []

    def assert_route(route, allowed, action):
        before = _component_hashes(models)
        trainable_before = _trainable_hashes(models)
        rng_before = _dropout_state(models)
        clock = int(models["optimizer"].iterations.numpy())
        action()
        after = _component_hashes(models)
        trainable_after = _trainable_hashes(models)
        rng_after = _dropout_state(models)
        changed = {key for key in before if before[key] != after[key]}
        changed_trainables = {key for key in trainable_before
                              if trainable_before[key] != trainable_after[key]}
        if changed != set(allowed):
            raise RuntimeError(f"Frozen-variable smoke failed for {route}: changed={sorted(changed)}")
        if changed_trainables != set(allowed):
            raise RuntimeError(f"Trainable-variable smoke failed for {route}: {sorted(changed_trainables)}")
        if int(models["optimizer"].iterations.numpy()) != clock + 1:
            raise RuntimeError("One update route did not make exactly one shared Adam call")
        expected_rng = "dropout_C" if route == "C" or route.startswith("GC_") else "dropout_D"
        changed_rng = {key for key in rng_before if rng_before[key] != rng_after[key]}
        if changed_rng != {expected_rng}:
            raise RuntimeError(f"Dropout training-phase smoke failed for {route}: {changed_rng}")
        freeze_checks.append({"route": route, "changed_components": sorted(changed),
                              "changed_trainable_components": sorted(changed_trainables),
                              "allowed_components": list(allowed),
                              "advanced_dropout_stream": expected_rng, "passed": True})

    assert_route("C", ["C"], lambda: models["updates"]["C"](X[:7], labels).numpy())
    assert_route("D", ["D"], lambda: models["updates"]["D"](
        X[:7], labels, np.ones((7, 1), dtype=np.float32)).numpy())
    for head_index in (0, 14):
        condition = np.zeros((7, 16), dtype=np.float32)
        condition[:, head_index] = 1
        assert_route(f"GC_{head_index}", ["G", f"head_{head_index:02d}"],
                     lambda: models["updates"]["GC"][head_index](noise, condition).numpy())
        assert_route(f"GD_{head_index}", ["G", f"head_{head_index:02d}"],
                     lambda: models["updates"]["GD"][head_index](noise, condition).numpy())
    inference_before = {"weights": _component_hashes(models), "rng": _dropout_state(models),
                        "clock": int(models["optimizer"].iterations.numpy())}
    features = models["G"]([noise, labels], training=False)
    for head in models["heads"]:
        value = head(features, training=False).numpy()
        if value.shape != (7, *SHAPE) or not np.isfinite(value).all():
            raise RuntimeError("Inference generation smoke produced invalid patches")
    inference_after = {"weights": _component_hashes(models), "rng": _dropout_state(models),
                       "clock": int(models["optimizer"].iterations.numpy())}
    if inference_before != inference_after:
        raise RuntimeError("Detached generation changed model/BN/RNG/optimizer state")
    del models
    keras.backend.clear_session()
    # Fresh networks after route checks: smoke updates never enter a training job.
    models = _build_models(tf, keras, region_data, normalized, seed)
    if _component_hashes(models) != initial_hashes:
        raise RuntimeError("Fresh initialization is not deterministic")
    trace = ExposureTrace(len(X), counts[order], seed, region_id)
    first_start = time.perf_counter()
    _one_loop(tf, models, X, trace, touched, shared)
    after_full_loop = _trainable_hashes(models)
    if any(after_full_loop[key] == initial_trainable_hashes[key] for key in initial_trainable_hashes):
        raise RuntimeError("The complete loop did not update every component, including all 15 heads")
    first_seconds = time.perf_counter() - first_start
    checkpoint = _checkpoint(tf, models)
    smoke_state = {"updates": 2, "contract": contract, "contract_sha256": _json_hash(contract),
                   "trace": trace.state(), "dropout_state": _dropout_state(models)}
    state = _save(out, checkpoint, models, smoke_state)
    second_start = time.perf_counter()
    loss_uninterrupted = _one_loop(tf, models, X, trace, touched, shared)
    second_seconds = time.perf_counter() - second_start
    expected = {"components": _component_hashes(models), "optimizer": _optimizer_hash(models),
                "dropout": _dropout_state(models), "trace": trace.state()}
    del models, checkpoint
    keras.backend.clear_session()
    models = _build_models(tf, keras, region_data, normalized, seed)
    checkpoint = _checkpoint(tf, models)
    _restore(out, checkpoint, state, contract)
    trace = ExposureTrace(len(X), counts[order], seed, region_id, state=state["trace"])
    if _dropout_state(models) != state["dropout_state"]:
        raise RuntimeError("Checkpoint did not restore both dropout generator states")
    loss_resumed = _one_loop(tf, models, X, trace, touched, shared)
    actual = {"components": _component_hashes(models), "optimizer": _optimizer_hash(models),
              "dropout": _dropout_state(models), "trace": trace.state()}
    if expected != actual or not np.array_equal(loss_uninterrupted, loss_resumed):
        raise RuntimeError("GPU checkpoint continuation was not bitwise reproducible; do not train")
    report = {"passed": True, "contract": contract, "frozen_variable_routes": freeze_checks,
              "fresh_initialization_sha256": _json_hash(initial_hashes),
              "all_15_heads_exercised": True, "shared_Adam_calls_per_loop": 34,
              "resume_exact_component_and_optimizer_hashes": True,
              "resume_exact_NumPy_and_dropout_streams": True,
              "inference_generation_keeps_BN_RNG_optimizer_unchanged": True,
              "optimizer_clock_after_two_loops": int(models["optimizer"].iterations.numpy()),
              "first_loop_with_compilation_seconds": first_seconds,
              "second_loop_seconds": second_seconds,
              "smoke_updates_are_not_training_budget": True, "test_evaluated": False}
    _atomic_json(report_path, report)
    del models, checkpoint
    keras.backend.clear_session()
    return report


### Code — `gan_exposure_runner.py` · phần 1/5: Suite identity và restore Output

**Mục đích:** Release/spec hash tạo suite25d35efe55cc, đúng30jobs; chỉ restore Output đầy đủ có cùng contract, không tự merge snapshot khác.

**Đầu vào:** Release config, Working/Input hoặc explicit resume. **Đầu ra:** Suite root đã pin.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Không lấy REPORT.zip làm RESUME_RUN_PATH. Có checkpoints/arrays thật không?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_runner.py 1/5 0:3 315ba51f378216c3472e372add4c52ffe82ae59dc28049bf36694c5bbbe7aec0
"""Standalone Kaggle orchestration for the registered HyperGAMO exposure suite.

The release builder embeds this module and its two helpers in one notebook cell.
No training runs merely by importing this file. GPU workers use fresh processes.
"""
from pathlib import Path
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile


def _exposure_json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,
                                    separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def _exposure_file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _exposure_write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, sort_keys=True, ensure_ascii=False,
                              indent=2, allow_nan=False) + "\n", encoding="utf-8")
    tmp.replace(path)


def _exposure_read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _exposure_release_config():
    from gan_exposure_release import SCIENTIFIC_CONFIG, RELEASE_ID, COMPONENT_HASHES
    cfg = json.loads(json.dumps(SCIENTIFIC_CONFIG))
    cfg["implementation_release"] = RELEASE_ID
    cfg["component_sha256"] = COMPONENT_HASHES
    return cfg


def _exposure_expected_jobs():
    return [(region, seed, arm) for region in (1, 2, 3)
            for seed in range(42, 47) for arm in ("blocked", "reuse_control")]


def _exposure_job_id(region, seed, arm):
    if (region, seed, arm) not in _exposure_expected_jobs():
        raise ValueError("Job nằm ngoài ma trận 30 lượt đã khóa.")
    return f"region{region:02d}_seed{seed}_{arm}"


def _exposure_suite_spec(cfg):
    return {"schema": "gan-train-exposure-suite-v1", "scientific_config": cfg,
            "jobs": [{"region": r, "seed": s, "arm": a,
                      "job_id": _exposure_job_id(r, s, a)}
                     for r, s, a in _exposure_expected_jobs()],
            "scope": "HyperGAMO system; fixed full test; not isolated G or B inflation",
            "test_gate": "all30_complete_and_pairs_valid; no scoring during benchmark/train"}


def _exposure_validate_suite(root, cfg):
    p = Path(root) / "suite_manifest.json"
    doc = _exposure_read_json(p)
    expected = _exposure_suite_spec(cfg)
    if doc.get("spec") != expected or doc.get("suite_id") != _exposure_json_hash(expected)[:12]:
        raise RuntimeError(f"Suite/config/release không khớp: {p}")
    return doc


def _exposure_snapshot_fingerprint(root):
    # Different saved snapshots must be selected explicitly, not silently merged.
    rows = []
    for p in sorted((Path(root) / "jobs").glob("*/recovery/state.json")):
        rows.append((p.relative_to(root).as_posix(), _exposure_file_hash(p)))
    for p in sorted((Path(root) / "jobs").glob("*/done.json")):
        rows.append((p.relative_to(root).as_posix(), _exposure_file_hash(p)))
    return _exposure_json_hash(rows)


def _exposure_restore_suite(output_root, cfg, explicit=None):
    spec = _exposure_suite_spec(cfg)
    suite_id = _exposure_json_hash(spec)[:12]
    destination = Path(output_root) / f"KLTN_GAN_Exposure_{suite_id}"
    if destination.exists():
        _exposure_validate_suite(destination, cfg)
        if explicit and Path(explicit).resolve() != destination.resolve():
            raise RuntimeError("Working đã có suite. Dùng suite hiện tại hoặc chọn Output root khác "
                               "để khôi phục snapshot riêng; không ghi đè checkpoint.")
        return destination
    candidates = []
    if explicit:
        candidate = Path(explicit)
        _exposure_validate_suite(candidate, cfg)
        candidates = [candidate]
    elif Path("/kaggle/input").is_dir():
        for p in Path("/kaggle/input").rglob("suite_manifest.json"):
            if p.parent.name == destination.name:
                _exposure_validate_suite(p.parent, cfg)
                candidates.append(p.parent)
    if candidates:
        fingerprints = {_exposure_snapshot_fingerprint(p) for p in candidates}
        if len(fingerprints) != 1:
            raise RuntimeError("Có nhiều snapshot khác tiến độ. Điền RESUME_RUN_PATH đúng thư mục "
                               "suite muốn tiếp tục:\n" + "\n".join(map(str, candidates)))
        source = sorted(candidates, key=str)[0]
        print("Khôi phục Output đã lưu:", source, flush=True)
        shutil.copytree(source, destination)
        _exposure_validate_suite(destination, cfg)
    else:
        destination.mkdir(parents=True)
        _exposure_write_json(destination / "suite_manifest.json",
                             {"suite_id": suite_id, "spec": spec,
                              "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
    return destination




### Code — `gan_exposure_runner.py` · phần 2/5: GPU isolation và gates của cặp

**Mục đích:** Fresh subprocess mỗi GPU, CUDA_VISIBLE_DEVICES cô lập; kiểm pair invariants/update trace và data pin. Test gate yêu cầu30/30complete.

**Đầu vào:** GPU inventory và suite. **Đầu ra:** Verified worker runtime/gates.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Không import TF trong notebook kernel trước worker. Có chặn test trước load X_test?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_runner.py 2/5 0:2 662906f0291c1ca5ec09b440bb339bdd7b1f547fc93c56e229ba825d847ff290
def _exposure_gpu_inventory():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=True, timeout=20).stdout
        rows = []
        for line in out.strip().splitlines():
            index, name, memory = (part.strip() for part in line.split(",", 2))
            rows.append({"index": int(index), "name": name, "memory_mb": int(memory)})
        if rows:
            return rows
    except (OSError, ValueError, subprocess.SubprocessError):
        pass
    raise RuntimeError("Chưa có GPU. Kaggle Settings > Accelerator > GPU T4/P100, "
                       "rồi chạy riêng cell này. Không train bằng CPU.")


def _exposure_bootstrap_gpu():
    # Called inside a fresh worker with a single CUDA_VISIBLE_DEVICES value.
    os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    import tensorflow as tf
    try:
        import tf_keras as keras
    except ImportError as e:
        raise RuntimeError("Môi trường thiếu tf_keras. Cần tf_keras cùng minor version TensorFlow; "
                           "runner không tự đổi/install thư viện trong lượt train.") from e
    if tf.__version__.split(".")[:2] != keras.__version__.split(".")[:2]:
        raise RuntimeError(f"TensorFlow/tf_keras lệch version: {tf.__version__}/{keras.__version__}")
    devices = tf.config.list_physical_devices("GPU")
    if len(devices) != 1:
        raise RuntimeError(f"Worker phải nhìn đúng một GPU, hiện thấy {len(devices)}.")
    for gpu in devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.config.experimental.enable_op_determinism()
    print(f"Worker GPU: TF {tf.__version__}, tf_keras {keras.__version__}; "
          f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}", flush=True)
    return tf, keras


def _exposure_inspect_job(jobdir, cfg, data=None, require_complete=False):
    from gan_exposure_training import exposure_inspect_job
    record = exposure_inspect_job(jobdir, cfg, data=data, require_complete=require_complete)
    if record.get("exists"):
        contract = record["state"]["contract"]
        data_contract = contract["data"]
        record["train_coords_sha256"] = data_contract["train_coords_sha256"]
        record["train_labels_sha256"] = data_contract["train_labels_sha256"]
        record["common_config_sha256"] = _exposure_json_hash(
            {"release_config": cfg, "training": contract["training"], "runtime": contract["runtime"],
             "region_id": contract["region_id"], "seed": contract["seed"]})
        if data is not None:
            record["preprocessing_sha256"] = data["metadata"]["preprocessing_sha256"]
        history = record["state"].get("history", [])
        steady = [row for row in history if not row.get("warmup_interval", False)
                  and row.get("classifier_updates", 0) > 100
                  and row.get("interval_loops", 0) > 0]
        if steady:
            record["seconds_per_c_update"] = (
                sum(row["interval_seconds"] for row in steady)
                / (2 * sum(row["interval_loops"] for row in steady)))
    return record


def _exposure_pair_gate(root, cfg, region, seed, require_complete=False):
    from gan_exposure_data import exposure_load_region
    records = []
    for arm in ("blocked", "reuse_control"):
        data = exposure_load_region(Path(root) / "data", region, arm)
        jobdir = Path(root) / "jobs" / _exposure_job_id(region, seed, arm)
        records.append(_exposure_inspect_job(jobdir, cfg, data, require_complete))
    # The training helper defines pair-invariant metadata and active trace prefixes.
    left, right = records
    if not all(record.get("exists") for record in records):
        raise RuntimeError("Pair chưa có đủ hai checkpoint đã commit.")
    for key in ("initial_weights_sha256", "train_coords_sha256", "train_labels_sha256",
                "preprocessing_sha256", "class_order", "common_config_sha256"):
        if left.get(key) is None or left.get(key) != right.get(key):
            raise RuntimeError(f"Pair region {region}/seed {seed} không khớp {key}.")
    if left.get("c_updates") == right.get("c_updates"):
        if left.get("trace_sha256") is None or left.get("trace_sha256") != right.get("trace_sha256"):
            raise RuntimeError(f"Pair region {region}/seed {seed} khác trace cùng tiến độ.")
    return records


def _exposure_all_test_gate(root, cfg):
    _exposure_pin_data(root)
    rows = []
    for region in (1, 2, 3):
        for seed in range(42, 47):
            pair = _exposure_pair_gate(root, cfg, region, seed, require_complete=True)
            for arm, state in zip(("blocked", "reuse_control"), pair):
                if state.get("c_updates") != cfg["target_c_updates"]:
                    raise RuntimeError("Test chỉ mở khi mọi job đủ đúng ngân sách đã khóa.")
                rows.append({"region": region, "seed": seed, "arm": arm, **state})
    return rows


def _exposure_pin_data(root):
    path = Path(root) / "data" / "data_manifest.json"
    digest = _exposure_file_hash(path)
    pin = Path(root) / "data_certificate_pin.json"
    record = {"data_manifest_sha256": digest}
    if pin.exists():
        if _exposure_read_json(pin) != record:
            raise RuntimeError("Data manifest khác chứng nhận đã khóa cho suite.")
    else:
        _exposure_write_json(pin, record)
    return record



### Code — `gan_exposure_runner.py` · phần 3/5: Báo cáo nhỏ và điều phối workers

**Mục đích:** Tạo REPORT.zip JSON/CSV/MD; tính measured GPU-hours; queues cân bằng arm assignment để tránh confound GPU tốc độ.

**Đầu vào:** Job checkpoints/log/history, session budget. **Đầu ra:** Training summary/reportZIP/workerqueues.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** REPORT không phải backup weights. Wall limit và save_every có cho resume an toàn?

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_runner.py 3/5 1:3 3de6aff1ba988194bb038006c9fddce1705455f1e9b9a1ed5ad42e98beeb6b65

def _exposure_report_zip(root):
    root = Path(root)
    out = root.parent / (root.name + "_REPORT.zip")
    tmp = out.with_suffix(".zip.tmp")
    # Small reports, no model weights or train tensors.
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(root.rglob("*")):
            if (p.is_file() and p.suffix in {".json", ".csv", ".md"}
                    and "checkpoints" not in p.parts and "runner" not in p.parts):
                z.write(p, p.relative_to(root.parent).as_posix())
    tmp.replace(out)
    return out


def _exposure_training_summary(root, cfg):
    rows = []
    for region, seed, arm in _exposure_expected_jobs():
        jobdir = Path(root) / "jobs" / _exposure_job_id(region, seed, arm)
        try:
            state = _exposure_inspect_job(jobdir, cfg)
            rows.append({"region": region, "seed": seed, "arm": arm,
                         "c_updates": state.get("c_updates", 0),
                         "complete": state.get("completed", False),
                         "optimizer_iterations": state.get("optimizer_iterations", 0),
                         "seconds_per_c_update": state.get("seconds_per_c_update")})
        except FileNotFoundError:
            rows.append({"region": region, "seed": seed, "arm": arm,
                         "c_updates": 0, "complete": False, "optimizer_iterations": 0,
                         "seconds_per_c_update": None})
    _exposure_write_json(Path(root) / "training_progress.json",
                         {"jobs": rows, "test_scored": False})
    timings = [row["seconds_per_c_update"] for row in rows
               if isinstance(row.get("seconds_per_c_update"), (int, float))
               and row["seconds_per_c_update"] > 0]
    if timings:
        seconds = sum(timings) / len(timings)
        remaining = sum(cfg["target_c_updates"] - row["c_updates"] for row in rows)
        estimate = {"mean_measured_seconds_per_c_update": seconds,
                    "remaining_c_updates": remaining,
                    "estimated_remaining_gpu_hours": remaining * seconds / 3600,
                    "scope": "measured training throughput only; checkpoint/startup variance excluded; "
                             "new regions can differ; no rental price assumed",
                    "benchmark_jobs_used": len(timings)}
        _exposure_write_json(Path(root) / "benchmark_summary.json", estimate)
        print("Ước lượng từ tốc độ thực:", round(estimate["estimated_remaining_gpu_hours"], 2),
              "GPU-hours còn lại (ước lượng).", flush=True)
    print("Tiến độ:", sum(bool(x["complete"]) for x in rows), "/30 job hoàn tất;",
          sum(int(x["c_updates"]) for x in rows), "/240000 C updates.", flush=True)
    return rows


def _exposure_pump_log(path, offset):
    path = Path(path)
    if not path.is_file():
        return offset
    with path.open("r", encoding="utf-8", errors="replace") as f:
        f.seek(offset)
        text = f.read()
        offset = f.tell()
    if text:
        print(text, end="", flush=True)
    return offset


def _exposure_worker_queues(jobs, gpu_count):
    if gpu_count < 1:
        raise ValueError("Cần ít nhất một GPU worker.")
    queues = [[] for _ in range(gpu_count)]
    pairs = {(region, seed): i for i, (region, seed) in enumerate(
        (region, seed) for region in (1, 2, 3) for seed in range(42, 47))}
    for i, job in enumerate(jobs):
        region, seed, arm = job
        if arm == "test":
            lane = i % gpu_count
        else:
            pair_index = pairs[(region, seed)]
            arm_index = ("blocked", "reuse_control").index(arm)
            # Alternate arms across devices for successive registered pairs;
            # filtering or resuming jobs does not change their assignment.
            lane = (pair_index + arm_index) % gpu_count
        queues[lane].append(list(job))
    return queues


def _exposure_run_workers(root, cfg, jobs, gpu_ids, code_root, runtime, test=False):
    queues = _exposure_worker_queues(jobs, len(gpu_ids))
    children = []
    for worker_id, (gpu, queue) in enumerate(zip(gpu_ids, queues)):
        if not queue:
            continue
        log_path = Path(root) / f"worker_{runtime['action']}_{worker_id}.log"
        log_stream = log_path.open("w", encoding="utf-8")
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        env["TF_USE_LEGACY_KERAS"] = "1"
        env["TF_DETERMINISTIC_OPS"] = "1"
        env["PYTHONUNBUFFERED"] = "1"
        env["EXPOSURE_WORKER_TASK"] = json.dumps({"root": str(root), "jobs": queue,
                                                 "runtime": runtime, "test": test})
        launcher = ("import sys; sys.path.insert(0, " + repr(str(code_root)) + "); "
                    "from gan_exposure_runner import exposure_worker_entry; exposure_worker_entry()")
        child = subprocess.Popen([sys.executable, "-u", "-c", launcher], env=env,
                                 stdout=log_stream, stderr=subprocess.STDOUT)
        children.append({"process": child, "log": log_path, "stream": log_stream,
                         "offset": 0, "gpu": gpu})
    try:
        last_heartbeat = time.monotonic()
        while any(x["process"].poll() is None for x in children):
            for item in children:
                item["offset"] = _exposure_pump_log(item["log"], item["offset"])
            failed = [x for x in children if x["process"].poll() not in (None, 0)]
            if failed:
                for x in failed:
                    x["offset"] = _exposure_pump_log(x["log"], x["offset"])
                raise RuntimeError("Worker lỗi; xem log bên trên. Checkpoint đã commit được giữ. "
                                   + ", ".join(str(x["log"]) for x in failed))
            if time.monotonic() - last_heartbeat >= 30:
                active = [x["gpu"] for x in children if x["process"].poll() is None]
                print("GPU worker còn chạy:", active, "| checkpoint tự lưu.", flush=True)
                last_heartbeat = time.monotonic()
            time.sleep(2)
        for x in children:
            x["offset"] = _exposure_pump_log(x["log"], x["offset"])
            if x["process"].returncode != 0:
                raise RuntimeError(f"Worker exit {x['process'].returncode}: {x['log']}")
    finally:
        for x in children:
            # Stop spawned helpers on interruption; never leave an unseen GPU job.
            if x["process"].poll() is None:
                x["process"].terminate()
                try:
                    x["process"].wait(timeout=20)
                except subprocess.TimeoutExpired:
                    x["process"].kill()
                    x["process"].wait(timeout=10)
            x["stream"].close()




### Code — `gan_exposure_runner.py` · phần 4/5: Entry point prepare/benchmark/train/test

**Mục đích:** Tách action; benchmark400 của cặp region1/seed42, train8000×30jobs; worker smoke trước train và recompute session remaining.

**Đầu vào:** ACTION và options đã khai báo riêng. **Đầu ra:** Launch summary/worker execution.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Default bản review phải không train khi Run All; launch scientific bằng cờ rõ ràng riêng.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_runner.py 4/5 0:3 5fbea8e30aeeb7ed0a15c238d146fc22d05acdae97a5b45688190aece3d00652
def exposure_launch(options, code_root):
    from gan_exposure_data import exposure_discover_raw, exposure_prepare
    from gan_exposure_release import SELECTION_MANIFEST_TEXT
    cfg = _exposure_release_config()
    action = options.get("action", "benchmark")
    if action not in {"prepare", "benchmark", "train", "test"}:
        raise ValueError("ACTION phải là prepare, benchmark, train hoặc test.")
    root = _exposure_restore_suite(options.get("output_root", "/kaggle/working"), cfg,
                                   options.get("resume_run_path"))
    print("Suite:", root, flush=True)
    source = exposure_discover_raw(options.get("hsi_path"), options.get("gt_path"))
    data_cfg = dict(cfg, selection_manifest_text=SELECTION_MANIFEST_TEXT)
    exposure_prepare(source, root / "data", data_cfg)
    _exposure_pin_data(root)
    if action == "prepare":
        print("CPU audit xong. Chưa train hoặc chấm test.", flush=True)
        print("ZIP báo cáo:", _exposure_report_zip(root), flush=True)
        return str(root)
    inventory = _exposure_gpu_inventory()
    wanted = options.get("gpu_ids")
    gpu_ids = [row["index"] for row in inventory] if wanted is None else list(wanted)
    if not gpu_ids or len(gpu_ids) != len(set(gpu_ids)) or not set(gpu_ids) <= {x["index"] for x in inventory}:
        raise ValueError("GPU_IDS không thuộc danh sách GPU hiện có.")
    gpu_ids = gpu_ids[:int(options.get("max_workers", 2))]
    if not gpu_ids:
        raise ValueError("MAX_WORKERS phải >=1.")
    selected_names = {x["name"] for x in inventory if x["index"] in gpu_ids}
    if len(selected_names) != 1:
        raise RuntimeError("GPU được chọn khác model. Chọn GPU_IDS=[0] hoặc một nhóm GPU cùng model "
                           "để tránh khác hardware giữa hai nhánh ghép cặp.")
    if action == "test":
        _exposure_all_test_gate(root, cfg)  # Before loading any test tensor/model.
        _exposure_run_workers(root, cfg, [(0, 0, "test")], gpu_ids[:1], code_root,
                              {"action": action}, test=True)
        print("TEST XONG — xem paired_summary.csv và region_summary.csv.", flush=True)
    else:
        if action == "benchmark":
            jobs = [(1, 42, "blocked"), (1, 42, "reuse_control")]
            stop = int(options.get("benchmark_c_updates", 400))
            if stop < 100 or stop > cfg["target_c_updates"] or stop % 2:
                raise ValueError("BENCHMARK_C_UPDATES là số chẵn 100..8000.")
        else:
            jobs = _exposure_expected_jobs()
            stop = cfg["target_c_updates"]
            if options.get("job_filter") is not None:
                selected = [tuple(x) for x in options["job_filter"]]
                if not selected or len(selected) != len(set(selected)) or any(x not in jobs for x in selected):
                    raise ValueError("JOB_FILTER chứa job ngoài suite hoặc trùng nhau.")
                jobs = [x for x in jobs if x in selected]
        runtime = {"action": action, "stop_c_updates": stop,
                   "session_seconds": float(options.get("session_hours", 8)) * 3600,
                   "save_every": int(options.get("save_every", 100))}
        if runtime["session_seconds"] <= 0 or runtime["save_every"] < 2 or runtime["save_every"] % 2:
            raise ValueError("SESSION_HOURS phải >0; SAVE_EVERY là số chẵn >=2.")
        print(f"{action}: {len(jobs)} jobs, {len(gpu_ids)} GPU worker độc lập; "
              f"stop {stop} C updates/job. Không chấm test.", flush=True)
        try:
            _exposure_run_workers(root, cfg, jobs, gpu_ids, code_root, runtime)
        finally:
            _exposure_training_summary(root, cfg)
            print("ZIP báo cáo:", _exposure_report_zip(root), flush=True)
        # Prefix trace equality is testable only at matching stop counts.
        for region in (1, 2, 3):
            for seed in range(42, 47):
                if all((root / "jobs" / _exposure_job_id(region, seed, arm) / "recovery" / "state.json").exists()
                       for arm in ("blocked", "reuse_control")):
                    _exposure_pair_gate(root, cfg, region, seed)
        if action == "benchmark":
            print(f"BENCHMARK XONG. Đổi ACTION='train' để tiếp tục checkpoint của cùng suite; "
                  f"{stop} bước đầu đã nằm trong ngân sách 8000.", flush=True)
        else:
            print("Lưu Version + Output để giữ checkpoint. Chạy lại ACTION='train' sẽ tiếp tục "
                  "job còn thiếu, bỏ qua job hoàn tất. Chỉ đổi ACTION='test' khi đủ 30/30.", flush=True)
    print("ZIP báo cáo:", _exposure_report_zip(root), flush=True)
    return str(root)


def exposure_worker_entry():
    task = json.loads(os.environ["EXPOSURE_WORKER_TASK"])
    root = Path(task["root"])
    cfg = _exposure_release_config()
    _exposure_validate_suite(root, cfg)
    tf, keras = _exposure_bootstrap_gpu()
    if task.get("test"):
        _exposure_evaluate_suite(tf, keras, root, cfg)
        return
    from gan_exposure_data import exposure_load_region
    from gan_exposure_training import exposure_gpu_smoke, exposure_train_job
    started = time.monotonic()
    deadline = float(task["runtime"]["session_seconds"])
    smoke_regions = set()
    for region, seed, arm in task["jobs"]:
        remaining = deadline - (time.monotonic() - started)
        if remaining <= 30:
            print("Đã tới giới hạn thời gian phiên; giữ checkpoint để tiếp tục.", flush=True)
            break
        data = exposure_load_region(root / "data", region, arm)
        jobdir = root / "jobs" / _exposure_job_id(region, seed, arm)
        existing = _exposure_inspect_job(jobdir, cfg, data)
        if existing.get("c_updates", 0) >= task["runtime"]["stop_c_updates"]:
            print("Đã có đủ bước; bỏ qua:", jobdir.name, existing.get("c_updates"), flush=True)
            continue
        smoke_root = root / "gpu_smoke" / f"worker_gpu{os.environ['CUDA_VISIBLE_DEVICES']}_region{region}_{arm}"
        job_cfg = dict(cfg, gpu_smoke_dir=str(smoke_root))
        if (region, arm) not in smoke_regions:
            exposure_gpu_smoke(tf, keras, data, job_cfg, smoke_root)
            smoke_regions.add((region, arm))
        remaining = deadline - (time.monotonic() - started)
        if remaining <= 30:
            print("Smoke đã qua; gần hết thời gian phiên. Giữ báo cáo để phiên sau tiếp tục.", flush=True)
            break
        print("JOB:", jobdir.name, flush=True)
        exposure_train_job(tf, keras, data, jobdir, job_cfg, seed, region, arm,
                                    max_c_updates=task["runtime"]["stop_c_updates"],
                                    save_every=task["runtime"]["save_every"], wall_seconds=remaining)
        result = _exposure_inspect_job(jobdir, cfg, data)
        print("JOB STATUS:", jobdir.name, result.get("c_updates"),
              "completed=", result.get("completed"), flush=True)




### Code — `gan_exposure_runner.py` · phần 5/5: Độ đo đủ16 output và test paired

**Mục đích:** All16 argmax và stable logsumexp NLL; pooled theo seed751exposed/1109all/358zero; 15region×seedpairs,12true classes, descriptive criterion locked.

**Đầu vào:** 30 final classifiers và identical full test. **Đầu ra:** condition/paired/perclass/bin/region summaries, final manifest/cache.

**Trạng thái:** CPU checks + GPU benchmark400/nhánh đã có; full30jobs/test chưa nhận. **Giới hạn/cần review:** Không bỏ output lớp vắng test; ΔOA=control−blocked, ΔNLL tương tự nhưng NLL thấp tốt hơn. Không diễn giải tCI như spatial-population uncertainty hay dose-bin causal.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_runner.py 5/5 0:1 e838cdb89068cbb5844d91da38398d344bc4616422b4bf98ed3702f132cadacc
def _exposure_metrics(y, prediction, nll):
    import numpy as np
    from sklearn.metrics import f1_score, recall_score, cohen_kappa_score
    y = np.asarray(y, dtype=np.int64)
    prediction = np.asarray(prediction, dtype=np.int64)
    labels = np.unique(y)
    if len(y) == 0:
        return {"samples": 0, "classes": 0, "OA": None, "AA": None,
                "Macro_F1": None, "mean_NLL": None, "balanced_NLL": None, "Kappa": None}
    return {"samples": int(len(y)), "classes": int(len(labels)),
            "OA": float(np.mean(prediction == y)),
            "AA": float(recall_score(y, prediction, labels=labels, average="macro", zero_division=0)),
            "Macro_F1": float(f1_score(y, prediction, labels=labels, average="macro", zero_division=0)),
            "mean_NLL": float(np.mean(nll)),
            "balanced_NLL": float(np.mean([np.mean(nll[y == c]) for c in labels])),
            "Kappa": float(cohen_kappa_score(y, prediction)),
            "true_class_ids": (labels + 1).tolist(),
            "prediction_space": "all16; macro/AA averaged only true-present classes"}


def _exposure_evaluate_suite(tf, keras, root, cfg):
    import numpy as np
    import pandas as pd
    from scipy.special import logsumexp, softmax
    from scipy.stats import t
    from gan_exposure_data import exposure_load_region, exposure_load_test
    from gan_exposure_training import exposure_load_classifier
    _exposure_all_test_gate(root, cfg)
    out = Path(root) / "test_results"
    out.mkdir(exist_ok=True)
    checkpoint_digests = { _exposure_job_id(r,s,a): _exposure_file_hash(
        Path(root) / "jobs" / _exposure_job_id(r,s,a) / "done.json")
        for r,s,a in _exposure_expected_jobs() }
    cache_manifest = out / "manifest.json"
    if cache_manifest.exists():
        cached = _exposure_read_json(cache_manifest)
        if (cached.get("complete") is not True
                or cached.get("suite_spec_sha256") != _exposure_json_hash(_exposure_suite_spec(cfg))
                or cached.get("checkpoint_digests") != checkpoint_digests):
            raise RuntimeError("Test cache không thuộc đúng suite/checkpoints hiện tại.")
        for name,digest in cached["files_sha256"].items():
            if Path(name).name != name or _exposure_file_hash(out / name) != digest:
                raise RuntimeError("Test report cache/hash không khớp.")
        print("Đã có test report đã kiểm hash; dùng lại, không inference lần nữa.", flush=True)
        return
    rows, arrays = [], {}
    for region in (1, 2, 3):
        test = exposure_load_test(Path(root) / "data", region)
        y = np.asarray(test["y_test"], dtype=np.int64)
        e = np.asarray(test["exposure_counts"], dtype=np.int64)
        for seed in range(42, 47):
            for arm in ("blocked", "reuse_control"):
                jobdir = Path(root) / "jobs" / _exposure_job_id(region, seed, arm)
                data = exposure_load_region(Path(root) / "data", region, arm)
                loaded = exposure_load_classifier(tf, keras, jobdir, cfg, data)
                model, logits_model, order = loaded["C"], loaded["logits"], np.asarray(loaded["order"])
                if order.shape != (16,) or set(order.tolist()) != set(range(16)):
                    raise RuntimeError("Classifier class order không đủ 16 lớp.")
                logits_blocks = []
                for start in range(0, len(y), 32):
                    batch = np.asarray(test["X_test"][start:start + 32], dtype=np.float32)
                    block = np.asarray(logits_model(batch, training=False), dtype=np.float64)
                    prob = np.asarray(model(batch, training=False), dtype=np.float64)
                    if block.shape != (len(batch), 16) or not np.isfinite(block).all():
                        raise RuntimeError("Test logits sai shape hoặc nonfinite.")
                    np.testing.assert_allclose(softmax(block, axis=1), prob, atol=2e-6, rtol=2e-5)
                    logits_blocks.append(block)
                sorted_logits = np.concatenate(logits_blocks)
                logits = np.empty_like(sorted_logits)
                logits[:, order] = sorted_logits
                pred = logits.argmax(axis=1)
                nll = logsumexp(logits, axis=1) - logits[np.arange(len(y)), y]
                arrays[(region, seed, arm)] = {"y": y, "pred": pred, "nll": nll, "exposure": e}
                np.savez_compressed(out / f"{_exposure_job_id(region, seed, arm)}.npz",
                                    y=y, prediction=pred, logits=logits, nll=nll,
                                    coords=test["coords_test"], exposure_counts=e)
                for name, mask in (("all", np.ones(len(y), dtype=bool)), ("exposed", e > 0),
                                   ("zero_shared", e == 0)):
                    rows.append({"region": region, "seed": seed, "arm": arm, "subset": name,
                                 **_exposure_metrics(y[mask], pred[mask], nll[mask])})
                print("Test:", region, seed, arm, "|", len(y), "mẫu; đủ16outputs.", flush=True)
                keras.backend.clear_session()
    conditions = pd.DataFrame(rows)
    conditions.to_csv(out / "condition_summary.csv", index=False)
    paired = []
    for region in (1, 2, 3):
        for seed in range(42, 47):
            for subset in ("all", "exposed", "zero_shared"):
                part = conditions[(conditions.region == region) & (conditions.seed == seed)
                                  & (conditions.subset == subset)].set_index("arm")
                row = {"region": region, "seed": seed, "subset": subset,
                       "samples": int(part.loc["blocked", "samples"])}
                for metric in ("OA", "AA", "Macro_F1", "mean_NLL", "balanced_NLL"):
                    row[f"delta_{metric}_control_minus_blocked"] = (
                        float(part.loc["reuse_control", metric] - part.loc["blocked", metric]))
                paired.append(row)
    paired = pd.DataFrame(paired)
    paired.to_csv(out / "paired_by_region_seed.csv", index=False)
    pooled = []
    perclass = []
    dose_rows = []
    for seed in range(42, 47):
        full = {}
        for arm in ("blocked", "reuse_control"):
            full[arm] = {key: np.concatenate([arrays[(r, seed, arm)][key] for r in (1, 2, 3)])
                         for key in ("y", "pred", "nll", "exposure")}
        for subset in ("all", "exposed", "zero_shared"):
            e = full["blocked"]["exposure"]
            mask = np.ones(len(e), bool) if subset == "all" else (e > 0 if subset == "exposed" else e == 0)
            m = {arm: _exposure_metrics(val["y"][mask], val["pred"][mask], val["nll"][mask])
                 for arm, val in full.items()}
            row = {"seed": seed, "subset": subset, "samples": int(mask.sum())}
            for metric in ("OA", "AA", "Macro_F1", "mean_NLL", "balanced_NLL"):
                row[f"blocked_{metric}"] = m["blocked"][metric]
                row[f"control_{metric}"] = m["reuse_control"][metric]
                row[f"delta_{metric}"] = m["reuse_control"][metric] - m["blocked"][metric]
            pooled.append(row)
        for class_id in np.unique(full["blocked"]["y"]):
            mask = full["blocked"]["y"] == class_id
            perclass.append({"seed": seed, "class_id": int(class_id + 1), "samples": int(mask.sum()),
                             "mean_shared_sources": float(full["blocked"]["exposure"][mask].mean()),
                             "delta_recall": float(np.mean(full["reuse_control"]["pred"][mask] == class_id)
                                                   - np.mean(full["blocked"]["pred"][mask] == class_id)),
                             "delta_NLL": float(np.mean(full["reuse_control"]["nll"][mask]
                                                       - full["blocked"]["nll"][mask]))})
        for name, low, high in cfg["exposure_bins"]:
            e = full["blocked"]["exposure"]
            mask = (e >= low) & (e <= high)
            if not mask.any():
                continue
            b, c = full["blocked"], full["reuse_control"]
            dose_rows.append({"seed": seed, "bin": name, "samples": int(mask.sum()),
                              "delta_OA": float(np.mean(c["pred"][mask] == c["y"][mask])
                                                - np.mean(b["pred"][mask] == b["y"][mask])),
                              "delta_NLL": float(np.mean(c["nll"][mask] - b["nll"][mask])),
                              "scope": "descriptive association; not causal dose-response"})
    pooled = pd.DataFrame(pooled)
    pooled.to_csv(out / "paired_summary.csv", index=False)
    pd.DataFrame(perclass).to_csv(out / "per_class_summary.csv", index=False)
    pd.DataFrame(dose_rows).to_csv(out / "exposure_bins.csv", index=False)
    region_summary = paired.groupby(["region", "subset"]).agg(
        mean_delta_OA=("delta_OA_control_minus_blocked", "mean"),
        sd_delta_OA=("delta_OA_control_minus_blocked", "std"),
        positive_seeds=("delta_OA_control_minus_blocked", lambda s: int((s > 0).sum())))
    region_summary.to_csv(out / "region_summary.csv")
    primary = pooled[pooled.subset == "exposed"].delta_OA.to_numpy(dtype=float)
    mean, sd = float(primary.mean()), float(primary.std(ddof=1))
    half = float(t.ppf(.975, len(primary) - 1) * sd / np.sqrt(len(primary)))
    reg = paired[paired.subset == "exposed"]
    positive_regions = int((reg.groupby("region").delta_OA_control_minus_blocked.mean() > 0).sum())
    positive_pairs = int((reg.delta_OA_control_minus_blocked > 0).sum())
    descriptive_signal = (mean >= cfg["practical_delta_oa"]
                          and positive_regions >= cfg["minimum_positive_regions"]
                          and positive_pairs >= cfg["minimum_positive_pairs"])
    summary = {"primary_samples": 751, "all_samples": 1109, "evaluable_classes": 12,
               "mean_delta_OA": mean, "sd_across_paired_seed_suites": sd,
               "conditional_seed_t_interval_95": [mean - half, mean + half],
               "positive_region_means": positive_regions, "positive_region_seed_pairs": positive_pairs,
               "registered_descriptive_signal": bool(descriptive_signal),
               "criterion": {key: cfg[key] for key in ("practical_delta_oa", "minimum_positive_regions",
                                                        "minimum_positive_pairs")},
               "interpretation": "Threshold is descriptive, not a statistical or causal proof. "
                                  "Small-n t CI assumes independent approximately normal paired seed-suite effects, "
                                  "conditional on fixed3regions. "
                                  "Not spatial-population uncertainty; not B inflation or isolated generator effect.",
               "source_intervention": "Different context content/directions remain in contrast.",
               "training_updates_during_test": 0}
    if any(int(row["samples"]) != {"all": 1109, "exposed": 751, "zero_shared": 358}[row["subset"]]
           for row in pooled.to_dict("records")):
        raise RuntimeError("Pooled test counts khác registered geometry.")
    _exposure_write_json(out / "summary.json", summary)
    files = {p.name: _exposure_file_hash(p) for p in out.iterdir()
             if p.is_file() and p.name != "manifest.json"}
    _exposure_write_json(out / "manifest.json",
                         {"complete": True, "suite_spec_sha256": _exposure_json_hash(_exposure_suite_spec(cfg)),
                          "checkpoint_digests": checkpoint_digests,
                          "files_sha256": files, "training_updates": 0, "scope": summary["interpretation"]})
    print("PRIMARY ΔOA:", round(mean * 100, 3), "pp;", positive_regions,
          "/3 region means positive;", positive_pairs, "/15 pairs positive.", flush=True)


### Code — `gan_exposure_region_selection.json` · phần 1/1: Artifact đăng ký đọc được

**Mục đích:** Đăng ký vùng bằng geometry/counts; lưu cả 16 ứng viên và ranking.

**Đầu vào:** GT + tiêu chí khả thi/ranking được khóa. **Đầu ra:** Ba region cố định của suite25d35efe55cc.

**Trạng thái:** Selection artifact đã khóa trước benchmark/test. **Giới hạn/cần review:** Không chọn vùng lại dựa trên accuracy.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma gan_exposure_region_selection.json 1/1 0:1 236b2c38ffda373fad8553fca554c5e67601f2d8c6bd5af4e895adb6e7bf0347
{
  "GT_sha256": "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c",
  "selection_rule": "Among train-all16/min20 geometry-feasible regions: maximize union evaluable classes across three rectangles, then affected test n, then lexicographic rectangles. No model predictions used.",
  "candidate_grid_starts": [
    15,
    45,
    75,
    105
  ],
  "grid": [
    {
      "rectangle": [
        15,
        39,
        15,
        39
      ],
      "test_n": 437,
      "train_n": 9373,
      "train_class_counts": [
        46,
        1047,
        685,
        155,
        473,
        724,
        28,
        478,
        20,
        936,
        2455,
        476,
        205,
        1265,
        297,
        83
      ],
      "test_classes": [
        2,
        3,
        4,
        10,
        12,
        15
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 285,
      "affected_classes": [
        2,
        3,
        4,
        10,
        12,
        15
      ],
      "control_shared_raw_sources": 596,
      "real_train_shared_source_occurrences_per_complete_pass": 15862,
      "shared_sources_per_test_patch_histogram": [
        152,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        1,
        1,
        1,
        37,
        0,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        1,
        3,
        41,
        0,
        0,
        0,
        0,
        1,
        1,
        1,
        0,
        6,
        1,
        43,
        0,
        0,
        1,
        1,
        1,
        0,
        3,
        3,
        1,
        1,
        34,
        2,
        0,
        1,
        1,
        3,
        0,
        4,
        0,
        2,
        3,
        39,
        0,
        3,
        5,
        0,
        0,
        5,
        0,
        2,
        0,
        4,
        0,
        5,
        1,
        2,
        0,
        0,
        2,
        5,
        0,
        2,
        0,
        0,
        0,
        5,
        0,
        1,
        0,
        0,
        0,
        3,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.2107343457457874,
      "mean_shared_fraction_affected_test": 0.32312599681020737,
      "class_mean_shared_fraction": {
        "2": 0.2116014345859972,
        "3": 0.22941730306517424,
        "4": 0.6139315230224321,
        "10": 0.45454545454545453,
        "12": 0.4027047332832457,
        "15": 0.049029622063329927
      },
      "touching_train_n": 543
    },
    {
      "rectangle": [
        15,
        39,
        45,
        69
      ],
      "test_n": 439,
      "train_n": 9515,
      "train_class_counts": [
        46,
        1108,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        970,
        2455,
        274,
        205,
        1265,
        386,
        0
      ],
      "test_classes": [
        2,
        12,
        16
      ],
      "train_classes": 15,
      "minimum_train_class_n": 0,
      "affected_test_n": 318,
      "affected_classes": [
        2,
        12,
        16
      ],
      "control_shared_raw_sources": 452,
      "real_train_shared_source_occurrences_per_complete_pass": 9557,
      "shared_sources_per_test_patch_histogram": [
        121,
        4,
        6,
        7,
        10,
        5,
        11,
        5,
        9,
        5,
        9,
        19,
        10,
        3,
        4,
        6,
        9,
        1,
        5,
        3,
        6,
        5,
        21,
        2,
        3,
        2,
        4,
        2,
        5,
        0,
        4,
        6,
        2,
        19,
        2,
        0,
        2,
        1,
        3,
        0,
        4,
        5,
        2,
        2,
        15,
        2,
        0,
        0,
        2,
        5,
        1,
        4,
        1,
        1,
        1,
        15,
        1,
        2,
        4,
        1,
        1,
        4,
        0,
        1,
        1,
        4,
        0,
        5,
        0,
        0,
        0,
        1,
        2,
        4,
        0,
        1,
        0,
        0,
        0,
        4,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.17116286074662548,
      "mean_shared_fraction_affected_test": 0.23629086750870626,
      "class_mean_shared_fraction": {
        "2": 0.23966942148760334,
        "12": 0.07770139800726036,
        "16": 0.30082644628099175
      },
      "touching_train_n": 377
    },
    {
      "rectangle": [
        15,
        39,
        75,
        99
      ],
      "test_n": 193,
      "train_n": 9775,
      "train_class_counts": [
        46,
        1394,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        699,
        2394,
        593,
        205,
        1174,
        371,
        93
      ],
      "test_classes": [
        10,
        11,
        14
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 147,
      "affected_classes": [
        10,
        11,
        14
      ],
      "control_shared_raw_sources": 437,
      "real_train_shared_source_occurrences_per_complete_pass": 11726,
      "shared_sources_per_test_patch_histogram": [
        46,
        1,
        1,
        1,
        1,
        1,
        0,
        0,
        0,
        0,
        0,
        9,
        1,
        1,
        1,
        1,
        1,
        0,
        0,
        0,
        0,
        0,
        13,
        1,
        2,
        2,
        2,
        2,
        2,
        0,
        0,
        0,
        1,
        16,
        2,
        2,
        2,
        2,
        1,
        1,
        1,
        0,
        2,
        0,
        14,
        1,
        2,
        2,
        3,
        1,
        1,
        1,
        1,
        0,
        0,
        16,
        3,
        2,
        1,
        2,
        2,
        1,
        0,
        2,
        3,
        0,
        0,
        2,
        0,
        0,
        0,
        3,
        0,
        1,
        0,
        1,
        0,
        0,
        1,
        1,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        1,
        0,
        0,
        1,
        0,
        1,
        0,
        1,
        0,
        1,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.2618079047659829,
      "mean_shared_fraction_affected_test": 0.343734187890032,
      "class_mean_shared_fraction": {
        "10": 0.2034416393071437,
        "11": 0.7561983471074382,
        "14": 0.39729893166700264
      },
      "touching_train_n": 503
    },
    {
      "rectangle": [
        15,
        39,
        105,
        129
      ],
      "test_n": 330,
      "train_n": 9554,
      "train_class_counts": [
        46,
        1404,
        830,
        237,
        483,
        730,
        28,
        332,
        20,
        960,
        2252,
        593,
        205,
        955,
        386,
        93
      ],
      "test_classes": [
        8,
        11,
        14
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 260,
      "affected_classes": [
        8,
        11,
        14
      ],
      "control_shared_raw_sources": 515,
      "real_train_shared_source_occurrences_per_complete_pass": 12193,
      "shared_sources_per_test_patch_histogram": [
        70,
        0,
        0,
        0,
        0,
        0,
        0,
        1,
        1,
        1,
        1,
        30,
        0,
        1,
        0,
        0,
        0,
        0,
        2,
        2,
        2,
        2,
        30,
        0,
        1,
        1,
        0,
        2,
        0,
        2,
        2,
        4,
        2,
        29,
        0,
        1,
        3,
        1,
        0,
        0,
        5,
        4,
        2,
        2,
        29,
        1,
        1,
        2,
        1,
        5,
        2,
        5,
        2,
        1,
        2,
        29,
        0,
        4,
        6,
        1,
        1,
        5,
        1,
        0,
        0,
        5,
        0,
        5,
        1,
        0,
        0,
        0,
        3,
        5,
        0,
        0,
        0,
        0,
        0,
        6,
        0,
        0,
        0,
        0,
        0,
        3,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.25900325569747057,
      "mean_shared_fraction_affected_test": 0.32873490146217416,
      "class_mean_shared_fraction": {
        "8": 0.3961776859504132,
        "11": 0.29166666666666663,
        "14": 0.21645021645021645
      },
      "touching_train_n": 433
    },
    {
      "rectangle": [
        45,
        69,
        15,
        39
      ],
      "test_n": 501,
      "train_n": 9331,
      "train_class_counts": [
        46,
        1314,
        698,
        118,
        457,
        460,
        28,
        478,
        0,
        972,
        2316,
        495,
        205,
        1265,
        386,
        93
      ],
      "test_classes": [
        2,
        3,
        4,
        6,
        9,
        11,
        12
      ],
      "train_classes": 15,
      "minimum_train_class_n": 0,
      "affected_test_n": 343,
      "affected_classes": [
        2,
        3,
        4,
        6,
        9,
        11,
        12
      ],
      "control_shared_raw_sources": 600,
      "real_train_shared_source_occurrences_per_complete_pass": 16718,
      "shared_sources_per_test_patch_histogram": [
        158,
        4,
        3,
        1,
        2,
        1,
        2,
        0,
        1,
        1,
        0,
        42,
        3,
        1,
        0,
        2,
        1,
        1,
        0,
        1,
        1,
        3,
        31,
        2,
        1,
        0,
        2,
        0,
        1,
        0,
        2,
        6,
        0,
        41,
        3,
        1,
        0,
        2,
        1,
        1,
        3,
        7,
        1,
        0,
        42,
        3,
        1,
        0,
        2,
        6,
        2,
        6,
        2,
        1,
        0,
        42,
        3,
        4,
        5,
        2,
        3,
        7,
        1,
        0,
        1,
        6,
        1,
        5,
        1,
        1,
        0,
        0,
        4,
        6,
        0,
        0,
        1,
        0,
        1,
        6,
        0,
        0,
        0,
        0,
        0,
        3,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.21881856122465812,
      "mean_shared_fraction_affected_test": 0.31961544948557935,
      "class_mean_shared_fraction": {
        "2": 0.48760330578512395,
        "3": 0.3230653643876784,
        "4": 0.3829201101928374,
        "6": 0.12545725511448313,
        "9": 0.1515151515151515,
        "11": 0.4079639368895567,
        "12": 0.15151515151515155
      },
      "touching_train_n": 619
    },
    {
      "rectangle": [
        45,
        69,
        45,
        69
      ],
      "test_n": 392,
      "train_n": 9465,
      "train_class_counts": [
        46,
        1146,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        926,
        1999,
        593,
        205,
        1265,
        386,
        93
      ],
      "test_classes": [
        2,
        11
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 284,
      "affected_classes": [
        2,
        11
      ],
      "control_shared_raw_sources": 572,
      "real_train_shared_source_occurrences_per_complete_pass": 15701,
      "shared_sources_per_test_patch_histogram": [
        108,
        0,
        0,
        1,
        0,
        0,
        0,
        0,
        3,
        1,
        1,
        31,
        0,
        0,
        1,
        0,
        3,
        0,
        2,
        0,
        1,
        2,
        35,
        0,
        3,
        2,
        0,
        1,
        1,
        0,
        2,
        4,
        4,
        33,
        1,
        0,
        2,
        1,
        1,
        1,
        7,
        4,
        0,
        1,
        35,
        1,
        2,
        1,
        3,
        3,
        1,
        4,
        4,
        1,
        0,
        34,
        1,
        3,
        7,
        1,
        1,
        5,
        0,
        2,
        2,
        3,
        0,
        5,
        2,
        0,
        1,
        0,
        2,
        4,
        0,
        0,
        0,
        0,
        1,
        4,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.23294400404790017,
      "mean_shared_fraction_affected_test": 0.3215283436154115,
      "class_mean_shared_fraction": {
        "2": 0.3484204835096934,
        "11": 0.1895896766710164
      },
      "touching_train_n": 601
    },
    {
      "rectangle": [
        45,
        69,
        75,
        99
      ],
      "test_n": 547,
      "train_n": 9334,
      "train_class_counts": [
        0,
        1176,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        399,
        2411,
        593,
        205,
        1265,
        386,
        93
      ],
      "test_classes": [
        1,
        2,
        10,
        11
      ],
      "train_classes": 15,
      "minimum_train_class_n": 0,
      "affected_test_n": 337,
      "affected_classes": [
        1,
        2,
        10,
        11
      ],
      "control_shared_raw_sources": 578,
      "real_train_shared_source_occurrences_per_complete_pass": 14548,
      "shared_sources_per_test_patch_histogram": [
        210,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        11,
        45,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        1,
        12,
        44,
        0,
        0,
        0,
        0,
        0,
        1,
        0,
        2,
        2,
        11,
        40,
        1,
        0,
        0,
        1,
        0,
        1,
        4,
        3,
        0,
        11,
        38,
        1,
        1,
        0,
        2,
        4,
        3,
        2,
        1,
        2,
        10,
        36,
        2,
        4,
        3,
        1,
        2,
        3,
        0,
        1,
        2,
        5,
        2,
        4,
        0,
        0,
        0,
        1,
        4,
        4,
        0,
        0,
        0,
        0,
        2,
        4,
        0,
        0,
        0,
        0,
        1,
        2,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.1916841675857797,
      "mean_shared_fraction_affected_test": 0.31113127498344656,
      "class_mean_shared_fraction": {
        "1": 0.3921028466483012,
        "2": 0.3255280073461891,
        "10": 0.14354169091685096,
        "11": 0.565162110616656
      },
      "touching_train_n": 511
    },
    {
      "rectangle": [
        45,
        69,
        105,
        129
      ],
      "test_n": 331,
      "train_n": 9587,
      "train_class_counts": [
        43,
        1282,
        830,
        237,
        451,
        730,
        16,
        198,
        20,
        906,
        2332,
        593,
        205,
        1265,
        386,
        93
      ],
      "test_classes": [
        2,
        5,
        8,
        10,
        11
      ],
      "train_classes": 16,
      "minimum_train_class_n": 16,
      "affected_test_n": 217,
      "affected_classes": [
        2,
        5,
        8,
        10,
        11
      ],
      "control_shared_raw_sources": 469,
      "real_train_shared_source_occurrences_per_complete_pass": 12745,
      "shared_sources_per_test_patch_histogram": [
        114,
        0,
        2,
        1,
        2,
        1,
        1,
        1,
        0,
        0,
        0,
        26,
        0,
        2,
        1,
        2,
        1,
        1,
        1,
        0,
        0,
        2,
        26,
        0,
        2,
        1,
        2,
        1,
        1,
        1,
        0,
        4,
        0,
        19,
        0,
        1,
        1,
        1,
        1,
        1,
        3,
        3,
        0,
        0,
        30,
        0,
        2,
        1,
        2,
        4,
        2,
        5,
        1,
        0,
        1,
        29,
        0,
        2,
        4,
        1,
        0,
        4,
        0,
        1,
        0,
        4,
        0,
        3,
        0,
        0,
        0,
        0,
        2,
        3,
        0,
        0,
        0,
        0,
        0,
        3,
        0,
        0,
        0,
        0,
        0,
        1,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.20241691842900303,
      "mean_shared_fraction_affected_test": 0.3087557603686636,
      "class_mean_shared_fraction": {
        "2": 0.4483471074380166,
        "5": 0.3636363636363636,
        "8": 0.22865013774104684,
        "10": 0.14244044725328145,
        "11": 0.07064782724606772
      },
      "touching_train_n": 466
    },
    {
      "rectangle": [
        75,
        99,
        15,
        39
      ],
      "test_n": 361,
      "train_n": 9500,
      "train_class_counts": [
        46,
        1420,
        819,
        237,
        300,
        710,
        28,
        478,
        18,
        972,
        1930,
        593,
        205,
        1265,
        386,
        93
      ],
      "test_classes": [
        5,
        11
      ],
      "train_classes": 16,
      "minimum_train_class_n": 18,
      "affected_test_n": 277,
      "affected_classes": [
        5,
        11
      ],
      "control_shared_raw_sources": 515,
      "real_train_shared_source_occurrences_per_complete_pass": 14769,
      "shared_sources_per_test_patch_histogram": [
        84,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        39,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        3,
        40,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        6,
        0,
        41,
        0,
        0,
        0,
        0,
        0,
        0,
        3,
        6,
        0,
        0,
        41,
        0,
        0,
        0,
        0,
        6,
        0,
        6,
        0,
        0,
        0,
        41,
        0,
        3,
        6,
        0,
        0,
        6,
        0,
        0,
        0,
        6,
        0,
        6,
        0,
        0,
        0,
        0,
        3,
        6,
        0,
        0,
        0,
        0,
        0,
        6,
        0,
        0,
        0,
        0,
        0,
        3,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.25173416359515577,
      "mean_shared_fraction_affected_test": 0.3280723215084882,
      "class_mean_shared_fraction": {
        "5": 0.2871002515271289,
        "11": 0.23963869857752926
      },
      "touching_train_n": 522
    },
    {
      "rectangle": [
        75,
        99,
        45,
        69
      ],
      "test_n": 543,
      "train_n": 9253,
      "train_class_counts": [
        46,
        1381,
        830,
        237,
        483,
        612,
        28,
        478,
        20,
        972,
        1624,
        593,
        205,
        1265,
        386,
        93
      ],
      "test_classes": [
        6,
        11
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 318,
      "affected_classes": [
        6,
        11
      ],
      "control_shared_raw_sources": 600,
      "real_train_shared_source_occurrences_per_complete_pass": 20372,
      "shared_sources_per_test_patch_histogram": [
        225,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        60,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        4,
        51,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        7,
        0,
        29,
        0,
        0,
        0,
        0,
        0,
        0,
        3,
        4,
        0,
        0,
        52,
        0,
        0,
        0,
        0,
        3,
        0,
        7,
        0,
        0,
        0,
        52,
        0,
        1,
        6,
        0,
        0,
        7,
        0,
        0,
        0,
        5,
        0,
        6,
        0,
        0,
        0,
        0,
        4,
        5,
        0,
        0,
        0,
        0,
        0,
        8,
        0,
        0,
        0,
        0,
        0,
        4,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.18507526292558937,
      "mean_shared_fraction_affected_test": 0.31602474141067616,
      "class_mean_shared_fraction": {
        "6": 0.47107438016528924,
        "11": 0.17069232859632014
      },
      "touching_train_n": 709
    },
    {
      "rectangle": [
        75,
        99,
        75,
        99
      ],
      "test_n": 271,
      "train_n": 9742,
      "train_class_counts": [
        35,
        1132,
        830,
        237,
        483,
        574,
        28,
        478,
        20,
        966,
        2417,
        593,
        205,
        1265,
        386,
        93
      ],
      "test_classes": [
        2,
        6
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 220,
      "affected_classes": [
        2,
        6
      ],
      "control_shared_raw_sources": 439,
      "real_train_shared_source_occurrences_per_complete_pass": 10601,
      "shared_sources_per_test_patch_histogram": [
        51,
        0,
        0,
        1,
        1,
        0,
        0,
        1,
        0,
        1,
        0,
        26,
        0,
        0,
        1,
        2,
        0,
        0,
        0,
        2,
        1,
        2,
        25,
        0,
        1,
        1,
        1,
        0,
        0,
        0,
        1,
        5,
        0,
        25,
        0,
        1,
        1,
        1,
        0,
        0,
        2,
        5,
        1,
        0,
        25,
        0,
        1,
        1,
        1,
        4,
        0,
        4,
        1,
        1,
        0,
        35,
        0,
        3,
        5,
        1,
        0,
        4,
        0,
        1,
        1,
        4,
        0,
        4,
        1,
        1,
        0,
        0,
        2,
        4,
        1,
        0,
        0,
        0,
        0,
        5,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.26912872434509466,
      "mean_shared_fraction_affected_test": 0.3315176558978211,
      "class_mean_shared_fraction": {
        "2": 0.23765005048427507,
        "6": 0.4082644628099173
      },
      "touching_train_n": 383
    },
    {
      "rectangle": [
        75,
        99,
        105,
        129
      ],
      "test_n": 43,
      "train_n": 10174,
      "train_class_counts": [
        46,
        1428,
        830,
        237,
        436,
        730,
        0,
        478,
        20,
        972,
        2455,
        593,
        205,
        1265,
        386,
        93
      ],
      "test_classes": [
        5,
        7
      ],
      "train_classes": 15,
      "minimum_train_class_n": 0,
      "affected_test_n": 36,
      "affected_classes": [
        5,
        7
      ],
      "control_shared_raw_sources": 110,
      "real_train_shared_source_occurrences_per_complete_pass": 983,
      "shared_sources_per_test_patch_histogram": [
        7,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        4,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        4,
        0,
        0,
        1,
        0,
        0,
        0,
        0,
        1,
        0,
        0,
        4,
        0,
        2,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        4,
        2,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        6,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        1,
        0,
        0,
        0,
        0,
        1,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.27291946953680574,
      "mean_shared_fraction_affected_test": 0.32598714416896246,
      "class_mean_shared_fraction": {
        "5": 0.20202020202020204,
        "7": 0.3925619834710744
      },
      "touching_train_n": 40
    },
    {
      "rectangle": [
        105,
        129,
        15,
        39
      ],
      "test_n": 391,
      "train_n": 9508,
      "train_class_counts": [
        46,
        1428,
        578,
        237,
        483,
        730,
        28,
        478,
        20,
        972,
        2155,
        593,
        16,
        1265,
        386,
        93
      ],
      "test_classes": [
        3,
        11,
        13
      ],
      "train_classes": 16,
      "minimum_train_class_n": 16,
      "affected_test_n": 211,
      "affected_classes": [
        3,
        11,
        13
      ],
      "control_shared_raw_sources": 326,
      "real_train_shared_source_occurrences_per_complete_pass": 8052,
      "shared_sources_per_test_patch_histogram": [
        180,
        1,
        2,
        2,
        2,
        1,
        3,
        1,
        2,
        2,
        2,
        31,
        2,
        0,
        1,
        1,
        1,
        0,
        2,
        0,
        1,
        3,
        31,
        0,
        1,
        0,
        0,
        1,
        0,
        0,
        1,
        4,
        0,
        31,
        0,
        0,
        0,
        0,
        0,
        0,
        2,
        4,
        0,
        0,
        24,
        0,
        0,
        0,
        0,
        4,
        0,
        3,
        0,
        0,
        0,
        24,
        0,
        2,
        3,
        0,
        0,
        3,
        0,
        0,
        0,
        3,
        0,
        3,
        0,
        0,
        0,
        0,
        1,
        3,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        0,
        1,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.14609710215383312,
      "mean_shared_fraction_affected_test": 0.27072970114762446,
      "class_mean_shared_fraction": {
        "3": 0.15199425080848006,
        "11": 0.21487603305785125,
        "13": 0.07321537522879804
      },
      "touching_train_n": 293
    },
    {
      "rectangle": [
        105,
        129,
        45,
        69
      ],
      "test_n": 342,
      "train_n": 9582,
      "train_class_counts": [
        46,
        1428,
        767,
        237,
        395,
        495,
        28,
        478,
        20,
        972,
        2230,
        593,
        149,
        1265,
        386,
        93
      ],
      "test_classes": [
        3,
        5,
        6,
        11,
        13
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 206,
      "affected_classes": [
        3,
        5,
        6,
        11,
        13
      ],
      "control_shared_raw_sources": 396,
      "real_train_shared_source_occurrences_per_complete_pass": 10911,
      "shared_sources_per_test_patch_histogram": [
        136,
        0,
        0,
        0,
        0,
        0,
        1,
        1,
        1,
        1,
        1,
        30,
        1,
        0,
        1,
        0,
        1,
        0,
        2,
        0,
        1,
        3,
        30,
        0,
        2,
        0,
        0,
        1,
        1,
        0,
        2,
        4,
        1,
        14,
        0,
        1,
        1,
        0,
        0,
        0,
        4,
        3,
        0,
        0,
        28,
        1,
        0,
        0,
        0,
        3,
        1,
        5,
        0,
        0,
        0,
        28,
        0,
        1,
        5,
        0,
        0,
        5,
        0,
        0,
        0,
        4,
        0,
        5,
        0,
        0,
        0,
        0,
        2,
        4,
        0,
        0,
        0,
        0,
        0,
        4,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.18761780484268523,
      "mean_shared_fraction_affected_test": 0.31148198668057453,
      "class_mean_shared_fraction": {
        "3": 0.4896694214876033,
        "5": 0.0340909090909091,
        "6": 0.2126509853782581,
        "11": 0.23057851239669422,
        "13": 0.40909090909090906
      },
      "touching_train_n": 414
    },
    {
      "rectangle": [
        105,
        129,
        75,
        99
      ],
      "test_n": 258,
      "train_n": 9699,
      "train_class_counts": [
        46,
        1428,
        830,
        237,
        483,
        553,
        28,
        478,
        20,
        972,
        2455,
        593,
        205,
        892,
        386,
        93
      ],
      "test_classes": [
        6,
        14
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 163,
      "affected_classes": [
        6,
        14
      ],
      "control_shared_raw_sources": 415,
      "real_train_shared_source_occurrences_per_complete_pass": 10509,
      "shared_sources_per_test_patch_histogram": [
        95,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        24,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        1,
        24,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        24,
        0,
        0,
        0,
        0,
        0,
        0,
        1,
        2,
        0,
        0,
        25,
        0,
        0,
        0,
        0,
        2,
        0,
        3,
        0,
        0,
        0,
        26,
        0,
        1,
        4,
        0,
        0,
        4,
        0,
        0,
        0,
        4,
        0,
        4,
        0,
        0,
        0,
        0,
        2,
        4,
        0,
        0,
        0,
        0,
        0,
        4,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.20991094881158306,
      "mean_shared_fraction_affected_test": 0.33225168584900877,
      "class_mean_shared_fraction": {
        "6": 0.2839840410373326,
        "14": 0.1884297520661157
      },
      "touching_train_n": 351
    },
    {
      "rectangle": [
        105,
        129,
        105,
        129
      ],
      "test_n": 271,
      "train_n": 9788,
      "train_class_counts": [
        46,
        1428,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        972,
        2455,
        593,
        205,
        804,
        386,
        93
      ],
      "test_classes": [
        14
      ],
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "affected_test_n": 150,
      "affected_classes": [
        14
      ],
      "control_shared_raw_sources": 230,
      "real_train_shared_source_occurrences_per_complete_pass": 6287,
      "shared_sources_per_test_patch_histogram": [
        121,
        0,
        0,
        0,
        0,
        0,
        0,
        1,
        1,
        1,
        1,
        21,
        0,
        0,
        1,
        0,
        1,
        0,
        1,
        0,
        1,
        2,
        21,
        0,
        1,
        0,
        0,
        1,
        1,
        0,
        1,
        2,
        1,
        21,
        0,
        1,
        1,
        0,
        0,
        0,
        3,
        2,
        0,
        0,
        21,
        1,
        0,
        0,
        0,
        2,
        1,
        2,
        0,
        0,
        0,
        21,
        0,
        1,
        2,
        0,
        0,
        2,
        0,
        0,
        0,
        2,
        0,
        2,
        0,
        0,
        0,
        0,
        1,
        2,
        0,
        0,
        0,
        0,
        0,
        2,
        0,
        0,
        0,
        0,
        0,
        1,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
      ],
      "mean_shared_fraction_all_test": 0.16467933274374066,
      "mean_shared_fraction_affected_test": 0.2975206611570248,
      "class_mean_shared_fraction": {
        "14": 0.16467933274374066
      },
      "touching_train_n": 202
    }
  ],
  "feasible_triplet_count": 84,
  "selected": {
    "union_classes": [
      2,
      3,
      4,
      5,
      6,
      8,
      10,
      11,
      12,
      13,
      14,
      15
    ],
    "class_count": 12,
    "affected_n": 751,
    "rectangles": [
      [
        15,
        39,
        15,
        39
      ],
      [
        15,
        39,
        105,
        129
      ],
      [
        105,
        129,
        45,
        69
      ]
    ]
  },
  "model_training_performed": false
}


## 07 — Phụ lục code kiểm, builder và thiết kế đã khảo sát

Các script QA/builder giữ nguyên để nhóm kiểm logic chứng nhận. Một số script chỉ chạy trong layout local gốc; không được gọi khi Run All. CrossedM3 chỉ là geometry design archive, không tự thêm36jobs.

### Code — `verify_cells.py` · phần 1/1: functions_only, main

**Mục đích:** Oracle cho ba cell split audit; kiểm count/geometry/pruning.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma verify_cells.py 1/1 0:1 2e8217f03ea442f96cedb2d690b29adbe7ebef8cd2513eeb1dfeb9292d559fdd
"""Local QA: synthetic geometry oracle + execution against the user's archive."""
import ast
import contextlib
import io
import itertools
import json
from pathlib import Path
import sys
import tempfile

import matplotlib
matplotlib.use("Agg")
import numpy as np
from scipy.ndimage import maximum_filter

ROOT = Path(__file__).resolve().parent


def functions_only(path, names):
    tree = ast.parse(path.read_text(encoding="utf-8"))
    tree.body = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name in names]
    ns = dict(np=np, maximum_filter=maximum_filter)
    exec(compile(tree, str(path), "exec"), ns)
    return ns


def main():
    sys.stdout.reconfigure(encoding="utf-8")
    geom = functions_only(ROOT / "cell_2_geometry.py", {"sa_pair_triple", "sa_overlap"})
    prune = functions_only(ROOT / "cell_3_report.py", {"sa_prune"})["sa_prune"]
    rng = np.random.default_rng(917)
    for _ in range(150):
        n = int(rng.integers(0, 12))
        coords = np.unique(rng.integers(0, 18, (n, 2)), axis=0)
        patch = int(rng.choice([1, 3, 5, 11]))
        pair, triple = geom["sa_pair_triple"](coords, patch)
        for size, actual in ((2, pair), (3, triple)):
            expected = any(all(max(abs(coords[a] - coords[b])) >= patch
                               for a, b in itertools.combinations(subset, 2))
                           for subset in itertools.combinations(range(len(coords)), size))
            assert (actual is not None) == expected
            if actual:
                assert all(max(abs(coords[a] - coords[b])) >= patch
                           for a, b in itertools.combinations(actual, 2))
    for patch in [1, 3, 5, 11]:
        original = rng.choice([-1, 0, 1, 2], size=(19, 21))
        for order in itertools.permutations(range(3)):
            mask = prune(original, order, patch)
            assert np.all((mask == -1) | (mask == original))
            assert not any(geom["sa_overlap"](mask, patch).values())
            # Independent oracle: intersect clipped, explicit raw-pixel sets.
            sets = []
            radius = patch // 2
            for k in range(3):
                raw = set()
                for r, c in np.argwhere(mask == k):
                    raw.update((i, j) for i in range(max(0, r-radius), min(mask.shape[0], r+radius+1))
                               for j in range(max(0, c-radius), min(mask.shape[1], c+radius+1)))
                sets.append(raw)
            assert all(not (sets[a] & sets[b]) for a, b in itertools.combinations(range(3), 2))
    # Explicit distance boundary (diagonal included).
    for center, expected in [([10, 0], False), ([10, 10], False), ([11, 0], True), ([0, 11], True)]:
        assert (geom["sa_pair_triple"](np.array([[0, 0], center]), 11)[0] is not None) == expected
    print("PASS: exact pair/triple vs brute-force, clipping and six pruning orders")

    archive = Path(sys.argv[1])
    namespace = {}
    before = (archive.stat().st_size, archive.stat().st_mtime_ns)
    logs = io.StringIO()
    with contextlib.redirect_stdout(logs):
        for index, filename in enumerate(("cell_1_read_run.py", "cell_2_geometry.py", "cell_3_report.py")):
            tree = ast.parse((ROOT / filename).read_text(encoding="utf-8"))
            if index == 0:
                for node in tree.body:
                    if isinstance(node, ast.Assign):
                        ids = [t.id for t in node.targets if isinstance(t, ast.Name)]
                        if "SA_SOURCE" in ids:
                            node.value = ast.Constant(str(archive))
                        elif "SA_OUTPUT_PARENT" in ids:
                            node.value = ast.parse("Path(" + repr(str(ROOT / "verified_output")) + ")", mode="eval").body
                ast.fix_missing_locations(tree)
            exec(compile(tree, filename, "exec"), namespace)
    assert before == (archive.stat().st_size, archive.stat().st_mtime_ns)
    assert namespace["sa_blocks"] == {"DIP_labels": [1, 7, 9, 16], "standard_gt": [1, 7, 9, 16]}
    counts = namespace["sa_variants"]
    assert (counts.classes_present_in_all_three == 0).all()
    assert namespace["sa_label_audit"]["train_vs_standard_gt_disagreements"] == 161
    out = namespace["sa_out"]
    (out / "local_execution.log").write_text(logs.getvalue(), encoding="utf-8")
    print("PASS: all three cells on real archive; archive size/mtime unchanged")
    print("REPORT:", out)
    print("ZIP:", namespace["sa_zip"])
    print("GEOMETRY:", namespace["sa_blocks"])
    print(counts.to_string(index=False))


if __name__ == "__main__":
    main()


### Code — `verify_impact_probe.py` · phần 1/2: main

**Mục đích:** Fixture geometry, metrics và fake orchestration của frozen sensitivity.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma verify_impact_probe.py 1/2 0:3 fef0e23063d2312b79a6e822e0bbeea144bab70ad72e1c17531623b17d770f18
"""Independent CPU geometry/metric and mock-orchestration verification.

Mock predictions are test data, never research results. Real TensorFlow/weights
must pass the saved-prediction reproduction gate on Kaggle.
"""
from pathlib import Path
import ast
import base64
import contextlib
import hashlib
import io
import json
import shutil
import sys
import tempfile
import zlib

import numpy as np
from scipy.io import loadmat
from sklearn.metrics import f1_score

ROOT = Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT))
from impact_probe_core import impact_geometry,impact_paired_masks,impact_metrics,impact_per_sample


def main():
    checks = {}
    cell_path = ROOT/"CELL_KIEM_TRA_DO_NHAY_B.py"
    cell = cell_path.read_text(encoding="utf-8")
    tree = ast.parse(cell)
    checks["standalone_syntax"] = True
    assert "__file__" not in cell
    forbidden = {"fit","fit_transform","train_on_batch","train_step","compile","apply_gradients"}
    assert not any(isinstance(n,ast.Call) and isinstance(n.func,ast.Attribute) and n.func.attr in forbidden
                   for n in ast.walk(tree))
    checks["no_training_or_preprocessing_fit_calls"] = True
    certificate = json.loads((ROOT/"cpu_certificate/certificate.json").read_text(encoding="utf-8"))
    gt_path = Path(certificate["source_gt"])
    assert hashlib.sha256(gt_path.read_bytes()).hexdigest()==certificate["source_gt_sha256"]
    gt = loadmat(gt_path)["indian_pines_gt"]
    source_tree = ast.parse((ROOT.parent/"v2_code.py").read_text(encoding="utf-8"))
    assignment = next(n for n in source_tree.body if isinstance(n,ast.Assign)
                      and any(isinstance(t,ast.Name) and t.id=="DIP_B64" for t in n.targets))
    dip = loadmat(io.BytesIO(zlib.decompress(base64.b64decode(ast.literal_eval(assignment.value)))))["indianpines_disjoint_dset"]
    author = np.array([0,2,3,5,6,8,10,11,12,14,1,4,7,9,13,15,16])[dip]
    train = (author>0)&(gt>0)&(author==gt)
    test = (gt>0)&~train
    train_coords,test_coords = np.argwhere(train),np.argwhere(test)
    y = gt[tuple(test_coords.T)].astype(np.int64)-1
    geometry = impact_geometry(gt.shape,train_coords,test_coords)
    # Direct physical-source enumeration, independent of the geometry function.
    source_seen = set()
    for r,c in train_coords:
        source_seen.update((qr,qc) for qr in range(max(0,r-5),min(145,r+6))
                          for qc in range(max(0,c-5),min(145,c+6)))
    for i in np.linspace(0,len(test_coords)-1,61,dtype=int):
        r,c = test_coords[i]
        for dr in range(-5,6):
            for dc in range(-5,6):
                valid = 0<=r+dr<145 and 0<=c+dc<145
                assert geometry["valid"][i,dr+5,dc+5]==valid
                assert geometry["shared"][i,dr+5,dc+5]==(valid and (r+dr,c+dc) in source_seen)
    checks["direct_physical_source_oracle"] = True
    eligible,masks = impact_paired_masks(geometry,test_coords)
    assert len(eligible)==2962 and np.array_equal(np.setdiff1d(np.arange(16),np.unique(y[eligible])),[8,15])
    assert np.all(masks.sum(axis=(-1,-2))==5) and not masks[...,5,5].any()
    for rep in range(2):
        for j,i in enumerate(eligible):
            shared_coords = np.argwhere(masks[rep,0,j])-5
            nonshared_coords = np.argwhere(masks[rep,1,j])-5
            assert sorted(np.max(np.abs(shared_coords),axis=1))==sorted(np.max(np.abs(nonshared_coords),axis=1))
            r,c = test_coords[i]
            assert all((r+dr,c+dc) in source_seen for dr,dc in shared_coords)
            assert all((r+dr,c+dc) not in source_seen for dr,dc in nonshared_coords)
    eligible2,masks2 = impact_paired_masks(geometry,test_coords)
    assert np.array_equal(eligible,eligible2) and np.array_equal(masks,masks2)
    checks["matched_mask_budget_center_shell_category_determinism"] = True
    metric_y = np.array([0,0,1,1])
    prob = np.zeros((4,16)); prob[np.arange(4),[0,1,1,8]]=1
    metric = impact_metrics(metric_y,prob)
    assert metric["classes_present"]==2 and metric["OA"]==.5
    assert abs(metric["Macro_F1_present_classes"]-f1_score(metric_y,prob.argmax(1),labels=[0,1],average="macro",zero_division=0))<1e-12
    checks["present_class_metric_external_oracle"] = True
    try:
        impact_paired_masks(geometry,test_coords,mask_seeds=())
        raise AssertionError("Empty mask seed list accepted")
    except ValueError:
        pass
    checks["empty_mask_seed_rejected"] = True

    namespace = {"__name__":"impact_probe_verification"}
    exec(compile("\n".join(cell.splitlines()[:-1]),str(cell_path),"exec"),namespace)
    X = np.random.default_rng(71).random((4721,11,11,40,1),dtype=np.float32)*2-1
    order = np.argsort(np.array([21,753,426,138,209,376,16,228,10,469,1390,311,125,720,287,49]),kind="mergesort")
    calls = [0]

    class MockClassifier:
        impact_device = "/CPU:0"
        def __init__(self,method):
            self.scale = .2 if method=="hypergamo" else .1
        def __call__(self,batch,training):
            assert training is False
            calls[0] += 1
            feature = batch[:,:,:,:2,0].mean(axis=(1,2,3))
            logits = feature[:,None]*np.linspace(-1,1,16)+np.linspace(0,.3,16)*self.scale
            probabilities = np.exp(logits-logits.max(axis=1,keepdims=True))
            probabilities /= probabilities.sum(axis=1,keepdims=True)
            return probabilities[:,order].astype(np.float32)
        def get_weights(self):
            return [np.array([self.scale],dtype=np.float32)]

    references = {}
    for method in namespace["IMPACT_METHODS"]:
        sorted_prob = MockClassifier(method)(X,training=False)
        standard_prob = np.empty_like(sorted_prob); standard_prob[:,order] = sorted_prob
        references[(method,42)] = standard_prob
    data = {"source":ROOT,"X_test":X,"y_test":y,"coords_test":test_coords,
            "coords_train":train_coords,"gt":gt,"fingerprint":{"mock":True},
            "fill_vector":np.linspace(-.2,.2,40,dtype=np.float32),
            "reference_probabilities":references}
    namespace["impact_load_source"] = lambda *args,**kwargs:data
    namespace["impact_load_classifier"] = lambda source,method,seed:(MockClassifier(method),(order+1).tolist())
    class MockRuntime:
        __version__ = "mock_CPU_verification"
        @staticmethod
        def device(name):
            return contextlib.nullcontext()
    namespace["impact_keras"] = lambda:(MockRuntime,MockRuntime,{})
    namespace["IMPACT_DEVICE"] = "/CPU:0"
    temporary = Path(tempfile.mkdtemp(prefix="impact_mock_",dir=ROOT)).resolve()
    assert ROOT.resolve() in temporary.parents and Path.cwd().resolve() in temporary.parents
    namespace["IMPACT_OUTPUT_ROOT"] = str(temporary)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            result = Path(namespace["impact_run_probe"]())
        manifest = json.loads((result/"manifest.json").read_text(encoding="utf-8"))
        assert manifest["complete"] is True and manifest["classification_inflation_measured"] is False
        calls_before = calls[0]
        with contextlib.redirect_stdout(io.StringIO()):
            namespace["impact_run_probe"]()
        assert calls[0]==calls_before
        checks["mock_full_orchestration_and_completed_inference_reuse"] = True
        selected = next(result.glob("*_shared_maskseed_*_per_sample.csv"))
        import pandas as pd
        table = pd.read_csv(selected)
        assert len(table)==2962 and np.isfinite(table["perturbation_l2"]).all()
        assert len(pd.read_csv(result/"paired_by_class.csv"))==56
        checks["per_sample_and_per_class_exports"] = True
        # A changed cache must stop the rerun and mark the output incomplete.
        cache = next(result.glob("*_shared_maskseed_*.npz"))
        with cache.open("ab") as stream:
            stream.write(b"changed")
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                namespace["impact_run_probe"]()
            raise AssertionError("Changed inference cache accepted")
        except RuntimeError as error:
            assert "Cache inference" in str(error)
        assert json.loads((result/"manifest.json").read_text(encoding="utf-8"))["complete"] is False
        checks["changed_cache_rejected_and_stale_completion_cleared"] = True
        # Missing references must fail before any model prediction.
        data["reference_probabilities"] = {}
        calls_before = calls[0]
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                namespace["impact_run_probe"]()
            raise AssertionError("Missing baseline references accepted")
        except RuntimeError as error:
            assert "predictions.npz" in str(error)
        assert calls[0]==calls_before
        checks["missing_baseline_reference_rejected_before_inference"] = True
    finally:
        # Resolved target is an explicitly verified temporary descendant.
        assert ROOT.resolve() in temporary.parents and temporary.name.startswith("impact_mock_")
        shutil.rmtree(temporary)
    record = {"checks":checks,"eligible_samples":2962,"eligible_class_count":14,
              "cell_sha256":hashlib.sha256(cell_path.read_bytes()).hexdigest(),
              "tensorflow_checkpoint_inference_tested":False,
              "limitation":"All model outputs in orchestration verification were mocks; no research classification metrics computed."}
    (ROOT/"impact_probe_cpu_verification.json").write_text(json.dumps(record,indent=2),encoding="utf-8")
    print(json.dumps(record,indent=2))




### Code — `verify_impact_probe.py` · phần 2/2: Imports, cấu hình và script body

**Mục đích:** Fixture geometry, metrics và fake orchestration của frozen sensitivity.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma verify_impact_probe.py 2/2 0:1 ffcb9419147370224b7d80c25e07bf934244a2f8092b42d607ed0437e58bd1ac
if __name__=="__main__":
    main()


### Code — `exposure_data_verify.py` · phần 1/1: main

**Mục đích:** Kiểm data geometry, PCA train-only và counterfactual trên nguồn thật.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma exposure_data_verify.py 1/1 0:1 1d4eaff41b4c195e1d4a2bce992469810b6d508db27836f55c575ad5471ef7cf
"""Independent CPU certificate for gan_exposure_data; no GAN or test scoring."""
from pathlib import Path
import json
import time
import numpy as np
import gan_exposure_data as data


def main():
    root = Path.cwd() / "protocol_b_review/leakage_research_2026-10-11"
    matroot = Path("C:/Users/ADMIN/OneDrive/Desktop/reproduct - 3dhyper/3D-HyperGAMO-main/3D-HyperGAMO-main/HSI-datasets/Classification")
    raw = data.exposure_discover_raw(matroot / "indian_pines_corrected.mat",
                                    matroot / "indian_pines_corrected_gt.mat")
    cube,gt = data._exposure_load_raw(raw)
    cube_digest = data.exposure_sha_array(cube)
    selection_bytes = data._exposure_selection_bytes({"selection_manifest_path":str(root / "gan_exposure_region_selection.json")})
    assert len(json.loads(selection_bytes)["grid"]) == 16
    result = {"raw":raw,"region_selection_sha256":data.EXPOSURE_SELECTION_SHA256,
        "training_performed":False,"test_predictions_created":False,"regions":[]}
    for region in (1,2,3):
        started = time.monotonic()
        geometry = data.exposure_build_geometry(gt,region)
        oracle = data.exposure_physical_oracle(gt,region,geometry)
        prep = data.exposure_fit_preprocessing(cube,geometry["coords_train"])
        # Check exact MinMax endpoints on the fitted train-center matrix.
        centers = cube[geometry["coords_train"][:,0],geometry["coords_train"][:,1]]
        center_values = data.exposure_transform_spectra(centers,prep)
        np.testing.assert_allclose(center_values.min(axis=0),-np.ones(40),atol=2e-12,rtol=0)
        np.testing.assert_allclose(center_values.max(axis=0),np.ones(40),atol=2e-12,rtol=0)
        np.testing.assert_allclose(data.exposure_transform_spectra(prep["pca_mean"][None],prep)[0],
                                   prep["missing_scaled_fill"],atol=0,rtol=0)
        cf = data.exposure_counterfactual_audit(cube,gt,region,geometry,prep)
        # Validate masked context and out-of-bounds padding separately, using
        # real examples containing the image boundary and the intervention.
        select = np.unique(np.r_[np.flatnonzero(geometry["touched_train"])[:16],
                                  np.flatnonzero(~geometry["train_valid"].all(axis=(1,2)))[:16]])
        assert len(select) > 0
        for arm,removed_key in (("blocked","s_mask"),("reuse_control","r_mask")):
            output = data.exposure_extract_train(cube,geometry,prep,arm,select)
            valid_removed = geometry["train_valid"][select] & geometry[removed_key][select]
            np.testing.assert_array_equal(output[...,0][valid_removed],
                np.broadcast_to(prep["missing_scaled_fill"].astype(np.float32),
                                 (int(valid_removed.sum()),40)))
            np.testing.assert_array_equal(output[...,0][~geometry["train_valid"][select]],
                np.zeros((int((~geometry["train_valid"][select]).sum()),40),np.float32))
            np.testing.assert_array_equal(output[:,5,5,:,0],
                data.exposure_transform_spectra(centers[select],prep).astype(np.float32))
        row = {"region":region, "oracle":oracle, "counterfactual":cf,
               "scaler_feature_range":[-1,1],"valid_mask_fill_train_mean":True,
               "invalid_image_padding_zero":True,"centers_intact":True,
               "PCA_full40_train_centers_only":True,"seconds":time.monotonic()-started,
               "preprocessing_arrays_sha256":{k:data.exposure_sha_array(v) for k,v in prep.items()}}
        result["regions"].append(row)
        print(json.dumps({"region":region,"oracle":oracle,"seconds":row["seconds"]}),flush=True)
    assert data.exposure_sha_array(cube) == cube_digest
    result["original_raw_arrays_unmodified"] = True
    result["implementation_sha256"] = data.exposure_sha_file(root / "gan_exposure_data.py")
    result["verification_sha256"] = data.exposure_sha_file(root / "exposure_data_verify.py")
    result["status"] = "CPU geometry + independent oracle + preprocessing + all-support counterfactual PASS; TensorFlow/GPU not verified"
    data._exposure_atomic_bytes(root / "exposure_data_cpu_certificate.json",data._exposure_json_bytes(result))
    print(result["status"],flush=True)


if __name__ == "__main__":
    main()


### Code — `exposure_review_verify.py` · phần 1/2: sha, array_sha, physical_oracle, independent_schedule

**Mục đích:** QA exposure trace/contracts/BN/invariants ở CPU; không thay GPU smoke.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma exposure_review_verify.py 1/2 0:3 96e287c67b5fefbb9a7ef9d54e343abdf7bd3525be5402036214ac3b62c436a7
"""Independent CPU coordinate/schedule review; this does not certify TensorFlow.

The physical oracle never calls the runner's source-ID or geometry functions.
It compares its result with the runner only after constructing every source.
"""
from pathlib import Path
import argparse
import hashlib
import importlib.util
import json
import math
import platform
import sys
import tempfile
import threading
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from scipy.io import loadmat

ROOT = Path(__file__).resolve().parent
EXPECTED_RAW = {
    "hsi": "ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939",
    "gt": "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c",
}


def sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4194304), b""):
            digest.update(block)
    return digest.hexdigest()


def array_sha(value):
    value = np.ascontiguousarray(value)
    digest = hashlib.sha256()
    digest.update(str(value.dtype).encode())
    digest.update(json.dumps(list(value.shape), separators=(",", ":")).encode())
    digest.update(value.tobytes())
    return digest.hexdigest()


def physical_oracle(gt, box):
    """Use physical (row,column) tuples, clipping, and reflected sets only."""
    low_r, high_r, low_c, high_c = box
    height, width = gt.shape
    centers = [(r,c) for r in range(height) for c in range(width) if gt[r,c] > 0]
    test = [(r,c) for r,c in centers if low_r <= r <= high_r and low_c <= c <= high_c]

    def patch(r,c):
        return {(rr,cc) for rr in range(max(0,r-5), min(height,r+6))
                for cc in range(max(0,c-5), min(width,c+6))}

    test_sources = set().union(*(patch(r,c) for r,c in test))
    train, masks_s, masks_r, removed_counts = [], [], [], []
    clean_sources, control_sources = set(), set()
    for r,c in centers:
        if (r,c) in test_sources:
            continue
        valid = patch(r,c)
        removed_s = valid & test_sources
        removed_r = {(2*r-rr,2*c-cc) for rr,cc in removed_s}
        if not removed_r <= valid or removed_r & test_sources:
            continue
        assert (r,c) not in removed_s | removed_r
        assert len(valid-removed_s) == len(valid-removed_r)
        train.append((r,c))
        mask_s, mask_r = np.zeros((11,11),bool), np.zeros((11,11),bool)
        for rr,cc in removed_s:
            mask_s[rr-r+5,cc-c+5] = True
        for rr,cc in removed_r:
            mask_r[rr-r+5,cc-c+5] = True
        np.testing.assert_array_equal(mask_s[::-1,::-1],mask_r)
        masks_s.append(mask_s)
        masks_r.append(mask_r)
        removed_counts.append(len(removed_s))
        clean_sources.update(valid-removed_s)
        control_sources.update(valid-removed_r)
    assert not clean_sources & test_sources
    exposure = [len(patch(r,c) & control_sources) for r,c in test]
    return {
        "coords_train":np.asarray(train,np.int32),
        "coords_test":np.asarray(test,np.int32),
        "s_mask":np.asarray(masks_s,bool), "r_mask":np.asarray(masks_r,bool),
        "shared_source_counts_train":np.asarray(removed_counts,np.int64),
        "exposure_counts":np.asarray(exposure,np.int64),
        "clean_shared_n":len(clean_sources & test_sources),
        "control_shared_n":len(control_sources & test_sources),
    }


def independent_schedule(n, sorted_counts, seed, region_id, runner, y_sorted,
                         touched, shared, loops=4000):
    """Replay the registered new stream independently, then compare the runner.

    The order of random draws follows B, but SeedSequence includes the region:
    this deliberately is not a historical B random-stream replay.
    """
    rng = np.random.default_rng(np.random.SeedSequence([seed,region_id,314159]))
    order = rng.permutation(n)
    deficits = sorted_counts[-1] - sorted_counts[:-1]
    assert len(deficits) == 15 and np.all(deficits > 0)
    cdf = np.cumsum(deficits/deficits.sum())
    cdf[-1] = 1.0
    n_batches = math.ceil(n/100)
    real_counts = np.zeros(n,np.int64)
    fake_counts = np.zeros(16,np.int64)
    digest = hashlib.sha256()
    digest.update(order.astype(np.int64).tobytes())
    actual = runner.ExposureTrace(n,sorted_counts,seed,region_id)
    np.testing.assert_array_equal(order,actual.real_order)
    resume_checked = False
    for loop in range(loops):
        j = loop % n_batches
        idx = order[j*100:min((j+1)*100,n)]
        size = len(idx)
        real_counts[idx] += 1
        valid = (1-rng.uniform(0,.1,size=(size,1))).astype(np.float32)
        fake_ids = np.searchsorted(cdf,rng.random(size),side="right")
        fake_counts += np.bincount(fake_ids,minlength=16)
        noise = rng.normal(size=(size,100)).astype(np.float32)
        invalid = rng.uniform(0,.1,size=(size,1)).astype(np.float32)
        expected_record = {"indices":idx,"valid":valid,"fake_ids":fake_ids,
                           "noise":noise,"invalid":invalid}
        head_noises = []
        for value in (idx.astype(np.int64),valid,fake_ids.astype(np.int64),noise,invalid):
            digest.update(value.tobytes())
        for _ in range(15):
            head_noise = rng.normal(size=(7,100)).astype(np.float32)
            digest.update(head_noise.tobytes())
            head_noises.append(head_noise)
        expected_record["head_noise"] = head_noises
        actual_record = actual.next()
        assert len(actual_record["head_noise"]) == 15
        for key in ("indices","valid","fake_ids","noise","invalid"):
            np.testing.assert_array_equal(expected_record[key],actual_record[key])
        for expected_noise,actual_noise in zip(head_noises,actual_record["head_noise"]):
            np.testing.assert_array_equal(expected_noise,actual_noise)
        actual.commit(actual_record,y_sorted,touched,shared)
        if loop == 199:
            state = json.loads(json.dumps(actual.state()))
            restored = runner.ExposureTrace(n,sorted_counts,seed,region_id,state=state)
            assert restored.state() == actual.state()
            actual = restored
            resume_checked = True
    cycles,partial = divmod(loops,n_batches)
    expected = cycles*n + min(partial*100,n)
    assert int(real_counts.sum()) == int(fake_counts.sum()) == expected
    assert fake_counts[-1] == 0
    np.testing.assert_array_equal(actual.real_counts,
        np.bincount(y_sorted,weights=real_counts,minlength=16).astype(np.int64))
    np.testing.assert_array_equal(actual.fake_counts,fake_counts)
    assert int(real_counts@touched) == actual.touched_real_presentations
    assert int(real_counts@shared) == actual.potential_shared_raw_source_occurrences
    assert rng.bit_generator.state == actual.rng.bit_generator.state
    assert actual.loop == loops and resume_checked
    return {"real_order":order,"real_counts":real_counts,"fake_counts":fake_counts,
            "rng_state":rng.bit_generator.state,"digest":digest.hexdigest(),
            "actual_external_trace_matches":True,"external_resume_after400_C_updates":True}


def independent_pair_pin_checks(training):
    """Exercise real exclusive publication under concurrent matching/mixed writers."""
    def initialization(marker):
        components = {"C":marker*64,"D":"d"*64,"G":"f"*64}
        return {"component_sha256":components,"combined_sha256":training._json_hash(components),
                "seed":42,"region_id":1,"class_order":list(range(16)),
                "dropout_initial_state":{"dropout_C":[1,0,0],"dropout_D":[2,0,0]}}
    contract = {"seed":42,"region_id":1,"data":{"class_order":list(range(16))},
                "pair_data_invariants":{"train_coords_sha256":"a"*64,"preprocessing_sha256":"b"*64},
                "training":{"target_c_updates":8000},"runtime":{"device":"CPU fixture; no TensorFlow"}}
    initial_a,initial_b = initialization("a"),initialization("b")
    with tempfile.TemporaryDirectory(prefix="exposure_review_pin_",dir=ROOT) as temp:
        base = Path(temp).resolve()
        base.relative_to(ROOT.resolve())  # Confine automatic recursive cleanup.
        job_a,job_b = base/"jobs"/"blocked",base/"jobs"/"reuse_control"
        pin_a = training._pin_pair_initialization(job_a,initial_a,contract)
        pin_path = Path(pin_a["path"])
        original = pin_path.read_bytes()
        pin_b = training._pin_pair_initialization(job_b,initial_a,contract)
        assert pin_b["sha256"] == pin_a["sha256"]
        try:
            training._pin_pair_initialization(job_b,initial_b,contract)
        except RuntimeError:
            pass
        else:
            raise AssertionError("Different initialization was accepted")
        assert pin_path.read_bytes() == original
        changed_contract = json.loads(json.dumps(contract))
        changed_contract["pair_data_invariants"]["preprocessing_sha256"] = "c"*64
        try:
            training._pin_pair_initialization(job_b,initial_a,changed_contract)
        except RuntimeError:
            pass
        else:
            raise AssertionError("Different paired preprocessing was accepted")
        assert pin_path.read_bytes() == original

        race_base = base/"race"/"jobs"
        count = 16
        barrier = threading.Barrier(count)
        def publish(i):
            init = initial_a if i % 2 == 0 else initial_b
            barrier.wait(timeout=20)
            try:
                training._pin_pair_initialization(race_base/("job_%d"%i),init,contract)
                return True
            except RuntimeError:
                return False
        with ThreadPoolExecutor(max_workers=count) as pool:
            outcomes = list(pool.map(publish,range(count)))
        assert sum(outcomes) == count//2
        race_path = training._pair_pin_path(race_base/"job_0",contract)
        record = json.loads(race_path.read_text(encoding="utf8"))
        accepted_parities = {i % 2 for i,passed in enumerate(outcomes) if passed}
        assert len(accepted_parities) == 1
        winner = initial_a if accepted_parities == {0} else initial_b
        assert record == training._pair_pin_record(winner,contract)
        assert not list(race_path.parent.glob("*.tmp"))
    return {"same_pair_reuses_identical_pin":True,"different_weights_rejected_without_overwrite":True,
            "different_preprocessing_rejected_without_overwrite":True,
            "mixed_16_writer_race_accepts_only_one_initialization":True,
            "first_arm_establishes_counterpart_verifies":True,"both_ready_barrier":False}




### Code — `exposure_review_verify.py` · phần 2/2: independent_queue_checks, main

**Mục đích:** QA exposure trace/contracts/BN/invariants ở CPU; không thay GPU smoke.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma exposure_review_verify.py 2/2 0:1 d267918e1f63522d73f45b62112fb34af73835bb25b6268e82f1b87126706eb7
def independent_queue_checks(runner):
    jobs = runner._exposure_expected_jobs()
    for count in (1,2,3,4,5):
        queues = runner._exposure_worker_queues(jobs,count)
        assert sorted(tuple(job) for queue in queues for job in queue) == sorted(jobs)
        assert max(map(len,queues))-min(map(len,queues)) <= 1
        assignment = {tuple(job):lane for lane,queue in enumerate(queues) for job in queue}
        for r in (1,2,3):
            for seed in range(42,47):
                if count>1:
                    assert assignment[(r,seed,"blocked")] != assignment[(r,seed,"reuse_control")]
        filtered = jobs[::3]
        reduced = runner._exposure_worker_queues(filtered,count)
        for lane,queue in enumerate(reduced):
            for job in queue:
                assert assignment[tuple(job)] == lane
    return {"all30_jobs_assigned_once":True,"balanced_1_to5_workers":True,
            "paired_arms_separate_when_parallel":True,"filter_resume_assignment_stable":True}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output",type=Path,default=ROOT/"exposure_review_certificate.json")
    parser.add_argument("--data-module",type=Path,default=ROOT/"gan_exposure_data.py")
    parser.add_argument("--training-module",type=Path,default=ROOT/"gan_exposure_training.py")
    args = parser.parse_args()
    source = json.loads((ROOT/"cpu_certificate/certificate.json").read_text(encoding="utf8"))
    for kind in EXPECTED_RAW:
        assert sha(source["source_"+kind]) == EXPECTED_RAW[kind]
    document = loadmat(source["source_gt"])
    gt = next(value for key,value in document.items() if not key.startswith("__")
              and isinstance(value,np.ndarray) and value.shape == (145,145))
    assert np.count_nonzero(gt) == 10249
    feasibility = json.loads((ROOT/"gan_training_exposure_feasibility.json").read_text(encoding="utf8"))
    spec = importlib.util.spec_from_file_location("reviewed_data",args.data_module)
    data = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(data)
    training_spec = importlib.util.spec_from_file_location("reviewed_training",args.training_module)
    training = importlib.util.module_from_spec(training_spec)
    training_spec.loader.exec_module(training)
    runner_path = ROOT/"gan_exposure_runner.py"
    runner_spec = importlib.util.spec_from_file_location("reviewed_runner",runner_path)
    runner = importlib.util.module_from_spec(runner_spec)
    runner_spec.loader.exec_module(runner)
    pin_checks = independent_pair_pin_checks(training)
    queue_checks = independent_queue_checks(runner)
    regions = []
    for region_id,expected in enumerate(feasibility["candidate_layouts"],1):
        oracle = physical_oracle(gt,expected["box_inclusive_zero_based"])
        actual = data.exposure_build_geometry(gt,region_id)
        for key in ("coords_train","coords_test","s_mask","r_mask",
                    "shared_source_counts_train","exposure_counts"):
            np.testing.assert_array_equal(oracle[key],actual[key])
        n = len(oracle["coords_train"])
        y = gt[oracle["coords_train"][:,0],oracle["coords_train"][:,1]] - 1
        counts = np.bincount(y.astype(np.int64),minlength=16)
        assert n == expected["train_n"]
        np.testing.assert_array_equal(counts,expected["train_class_counts"])
        assert int((oracle["exposure_counts"]>0).sum()) == expected["affected_test_n"]
        assert oracle["clean_shared_n"] == 0
        assert oracle["control_shared_n"] == expected["control_train_test_shared_raw_pixels"]
        order = np.argsort(counts,kind="mergesort")
        inverse = np.empty(16,np.int64)
        inverse[order] = np.arange(16)
        affected = oracle["shared_source_counts_train"] > 0
        trace = independent_schedule(n,counts[order],42,region_id,training,
            inverse[y.astype(np.int64)],affected,oracle["shared_source_counts_train"])
        regions.append({
            "region_id":region_id,"box":expected["box_inclusive_zero_based"],
            "physical_coordinate_oracle_equals_runner":True,
            "train_n":n,"train_classes":int((counts>0).sum()),
            "test_n":len(oracle["coords_test"]),
            "affected_test_n":int((oracle["exposure_counts"]>0).sum()),
            "clean_shared_raw_n":oracle["clean_shared_n"],
            "control_shared_raw_n":oracle["control_shared_n"],
            "train_coords_sha256":array_sha(oracle["coords_train"]),
            "s_mask_sha256":array_sha(oracle["s_mask"]),
            "r_mask_sha256":array_sha(oracle["r_mask"]),
            "independent_schedule_seed":42,"gan_loops":4000,"C_updates":8000,
            "expected_shared_Adam_calls":136000,
            "C_real_presentations":int(trace["real_counts"].sum()),
            "C_fake_presentations":int(trace["fake_counts"].sum()),
            "C_fake_presentations_sorted_classes":trace["fake_counts"].tolist(),
            "affected_train_C_real_presentations":int(trace["real_counts"]@affected),
            "control_shared_source_occurrence_C_real_exposures":
                int(trace["real_counts"]@oracle["shared_source_counts_train"]),
            "independent_full_external_rng_trace_sha256":trace["digest"],
            "runner_external_trace_arrays_and_class_counters_match":trace["actual_external_trace_matches"],
            "external_resume_after400_C_updates":trace["external_resume_after400_C_updates"],
        })
        print("Independent CPU review passed region%d"%region_id,flush=True)
    result = {"passed":True,"verification_scope":"CPU physical-source geometry and independent external schedule replay",
              "raw_input_sha256":EXPECTED_RAW,"regions":regions,
              "python":platform.python_version(),"numpy":np.__version__,
              "review_script_sha256":sha(__file__),"data_module_sha256":sha(args.data_module),
              "training_module_sha256":sha(args.training_module),
              "runner_module_sha256":sha(runner_path),
              "atomic_pair_pin_CPU_checks":pin_checks,"worker_queue_CPU_checks":queue_checks,
              "training_performed":False,"test_predictions_created":False,
              "TensorFlow_runtime_verified":False,"GPU_smoke_verified":False,
              "limitations":["External streams use SeedSequence([seed,region_id,314159]); this is not historical B bitwise replay.",
                             "Schedule replay does not execute model/dropout streams; TensorFlow runtime must pass the embedded GPU gate.",
                             "CPU certificate does not certify TensorFlow gradients, freeze, optimizer, GPU memory or checkpoint roundtrip."]}
    args.output.write_text(json.dumps(result,indent=2,ensure_ascii=True)+"\n",encoding="utf8")
    print(json.dumps({"passed":True,"regions":3,"certificate":str(args.output),"TensorFlow_runtime_verified":False}))


if __name__ == "__main__":
    sys.stdout.reconfigure(encoding="utf8")
    main()


### Code — `exposure_runner_verify.py` · phần 1/2: sha, run_cell, fake_evaluation_checks, main

**Mục đích:** Kiểm runner queues, recovery/test gates và metrics bằng fixture.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma exposure_runner_verify.py 1/2 0:3 d15a1bd642d2463355cde09f507538a561ef7144301814f76c4c1f265e4d074a
"""Real-data CPU entrypoint/cache checks and synthetic evaluation wiring.

This does not run TensorFlow, GAN training or scientific test inference.
Temporary tensors are removed from an explicitly confined temporary directory.
"""
from pathlib import Path
import ast
import importlib
import json
import tempfile
import time
import hashlib
import sys
import numpy as np

ROOT = Path.cwd() / "protocol_b_review/leakage_research_2026-10-11"
MATROOT = Path("C:/Users/ADMIN/OneDrive/Desktop/reproduct - 3dhyper/3D-HyperGAMO-main/3D-HyperGAMO-main/HSI-datasets/Classification")
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def run_cell(output_root):
    code = (ROOT / "CELL_GAN_TRAIN_EXPOSURE.py").read_text(encoding="utf-8")
    tree = ast.parse(code)
    overrides = {"ACTION":"prepare", "OUTPUT_ROOT":str(output_root),
                 "HSI_PATH":str(MATROOT / "indian_pines_corrected.mat"),
                 "GT_PATH":str(MATROOT / "indian_pines_corrected_gt.mat")}
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            name = node.targets[0].id
            if name in overrides:
                node.value = ast.Constant(overrides[name])
    ast.fix_missing_locations(tree)
    namespace = {"__name__":"__exposure_cpu_verify__"}
    exec(compile(tree, "<bundled_exposure_cell_prepare>", "exec"), namespace)
    return next(output_root.glob("KLTN_GAN_Exposure_*"))


def fake_evaluation_checks(runner, data, training, temp_root, cfg):
    from scipy.special import softmax
    root = temp_root / "synthetic_evaluation_fixture"
    root.mkdir()
    for r,s,a in runner._exposure_expected_jobs():
        job = root / "jobs" / runner._exposure_job_id(r,s,a)
        job.mkdir(parents=True)
        (job / "done.json").write_text(json.dumps({"r":r,"s":s,"a":a}),encoding="utf-8")
    # Exactly registered geometry totals/classes; labels are synthetic, not real GT.
    class_sets = {1:[1,2,3,9,11,14],2:[7,10,13],3:[2,4,5,10,12]}
    ns, exposed = {1:437,2:330,3:342},{1:285,2:260,3:206}
    order = np.array([8,6,0,15,12,3,4,7,14,11,5,2,9,13,1,10])
    fixtures = {}
    for region in (1,2,3):
        y = np.resize(np.array(class_sets[region],dtype=np.int64),ns[region])
        e = np.r_[np.full(exposed[region],30),np.zeros(ns[region]-exposed[region],dtype=int)]
        x = np.zeros((ns[region],11,11,40,1),np.float32)
        x[:,0,0,0,0] = y  # Fixture label feature; explicitly not a research model.
        x[:,0,0,1,0] = np.arange(ns[region])
        fixtures[region] = {"X_test":x,"y_test":y,"coords_test":np.c_[np.arange(ns[region]),np.zeros(ns[region])],
                            "exposure_counts":e,"metadata":{}}
    original_gate = runner._exposure_all_test_gate
    original_load_test,original_load_region = data.exposure_load_test,data.exposure_load_region
    original_load_model = training.exposure_load_classifier
    runner._exposure_all_test_gate = lambda *args: []
    data.exposure_load_test = lambda root,region:fixtures[region]
    data.exposure_load_region = lambda root,region,arm:{"region":region,"arm":arm}
    def load_model(tf,keras,jobdir,cfg,metadata):
        arm,region = metadata["arm"],metadata["region"]
        def logits(batch,training=False):
            labels = batch[:,0,0,0,0].astype(int)
            ids = batch[:,0,0,1,0].astype(int)
            orig = np.full((len(batch),16),-2.0)
            orig[np.arange(len(batch)),labels] = 2.0
            # blocked loses first10 exposed samples to an absent class16.
            if arm == "blocked":
                orig[ids<10,15] = 7.0
            return orig[:,order]
        return {"C":lambda x,training=False:softmax(logits(x),axis=1),
                "logits":logits,"order":order,"state":{}}
    training.exposure_load_classifier = load_model
    class FakeBackend:
        @staticmethod
        def clear_session(): pass
    class FakeKeras:
        backend = FakeBackend()
    try:
        runner._exposure_evaluate_suite(None,FakeKeras(),root,cfg)
        summary = json.loads((root/"test_results/summary.json").read_text(encoding="utf-8"))
        np.testing.assert_allclose(summary["mean_delta_OA"],30/751,rtol=0,atol=1e-15)
        assert summary["positive_region_means"] == 3
        assert summary["positive_region_seed_pairs"] == 15
        assert summary["registered_descriptive_signal"] is True
        p = root/"test_results/region01_seed42_blocked.npz"
        with np.load(p,allow_pickle=False) as f:
            assert np.all(f["prediction"][:10] == 15) # No restricting prediction to present classes.
            assert f["logits"].shape == (437,16)
            assert np.isfinite(f["nll"]).all()
        cache_before = sha(root/"test_results/manifest.json")
        data.exposure_load_test = lambda *args: (_ for _ in ()).throw(AssertionError("cache reloaded test"))
        runner._exposure_evaluate_suite(None,FakeKeras(),root,cfg)
        assert sha(root/"test_results/manifest.json") == cache_before
        # Stale checkpoint binding must fail even when a complete report exists.
        (root/"jobs/region01_seed42_blocked/done.json").write_text('{"changed":true}',encoding="utf-8")
        try:
            runner._exposure_evaluate_suite(None,FakeKeras(),root,cfg)
        except RuntimeError:
            pass
        else:
            raise AssertionError("Stale checkpoint/report mismatch was accepted")
        return {"full16_argmax":True,"class_order_scatter":True,"stable_NLL":True,
                "pooled751_weighting":True,"all15paired_comparisons":True,
                "inference_cache_reuse":True,"stale_checkpoint_rejected":True,
                "fixture_only":True,"scientific_test_scored":False}
    finally:
        runner._exposure_all_test_gate = original_gate
        data.exposure_load_test,data.exposure_load_region = original_load_test,original_load_region
        training.exposure_load_classifier = original_load_model


def main():
    started = time.monotonic()
    temporary = tempfile.TemporaryDirectory(prefix="gan_exposure_cpu_verify_", dir=ROOT)
    temp_root = Path(temporary.name).resolve()
    # Verify the absolute recursive-cleanup target before any writes/cleanup.
    assert temp_root.parent == ROOT.resolve()
    assert temp_root.name.startswith("gan_exposure_cpu_verify_")
    try:
        suite = run_cell(temp_root)
        import gan_exposure_runner as runner
        import gan_exposure_data as data
        import gan_exposure_training as training
        cfg = runner._exposure_release_config()
        queues = runner._exposure_worker_queues(runner._exposure_expected_jobs(), 2)
        assert [len(queue) for queue in queues] == [15, 15]
        assert {job[2] for job in queues[0]} == {"blocked", "reuse_control"}
        assert {job[2] for job in queues[1]} == {"blocked", "reuse_control"}
        assert runner._exposure_worker_queues([(1,43,"blocked")],2) == [[],[[1,43,"blocked"]]]
        manifest_hash = sha(suite/"data/data_manifest.json")
        manifest = json.loads((suite/"data/data_manifest.json").read_text(encoding="utf-8"))
        assert manifest["test_tensor_created"] is False
        assert len(runner._exposure_expected_jobs()) == 30
        for region in (1,2,3):
            left = data.exposure_load_region(suite/"data",region,"blocked")
            right = data.exposure_load_region(suite/"data",region,"reuse_control")
            assert not left["X_train"].flags.writeable and not right["X_train"].flags.writeable
            assert left["metadata"]["pair_invariants"] == right["metadata"]["pair_invariants"]
            assert left["metadata"]["input_sha256"] != right["metadata"]["input_sha256"]
            with np.load(suite/f"data/region_{region}/geometry.npz",allow_pickle=False) as a:
                assert "y_test" not in a.files and "X_test" not in a.files
            left["X_train"]._mmap.close()
            right["X_train"]._mmap.close()
        # Test gate fails before any model/test loading for an untrained suite.
        try:
            runner._exposure_all_test_gate(suite,cfg)
        except RuntimeError:
            pass
        else:
            raise AssertionError("Untrained suite passed test gate")
        run_cell(temp_root) # Run the actual bundled entrypoint again: verified cache reuse.
        assert sha(suite/"data/data_manifest.json") == manifest_hash
        # The bundled cell deliberately reloads its modules on every invocation.
        # Patch the current module objects for the isolated inference fixture.
        runner = sys.modules["gan_exposure_runner"]
        data = sys.modules["gan_exposure_data"]
        training = importlib.import_module("gan_exposure_training")
        cfg = runner._exposure_release_config()
        eval_checks = fake_evaluation_checks(runner,data,training,temp_root,cfg)
        certificate = {"status":"PASS", "bundled_prepare_entrypoint":True,
                       "real_train_cache_readonly":True,"preprocessing_pair_identical":True,
                       "mask_inputs_correctly_different":True,"cached_prepare_reused":True,
                       "incomplete_suite_test_gate_rejected":True,
                       "paired_arm_GPU_assignment_balanced":True,
                       "synthetic_evaluation":eval_checks,
                       "cell_sha256":sha(ROOT/"CELL_GAN_TRAIN_EXPOSURE.py"),
                       "runner_sha256":sha(ROOT/"gan_exposure_runner.py"),
                       "GPU_executed":False,"GAN_trained":False,"scientific_test_scored":False,
                       "seconds":time.monotonic()-started}
        (ROOT/"exposure_runner_cpu_certificate.json").write_text(
            json.dumps(certificate,indent=2)+"\n",encoding="utf-8")
        print(json.dumps(certificate,ensure_ascii=True),flush=True)
    finally:
        # Owned temporary tree; confinement was checked above. No user files removed.
        temporary.cleanup()




### Code — `exposure_runner_verify.py` · phần 2/2: Imports, cấu hình và script body

**Mục đích:** Kiểm runner queues, recovery/test gates và metrics bằng fixture.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma exposure_runner_verify.py 2/2 0:1 9a6a7ffeed1af83ced837054949e0b2fdf0e568fe3d4c2e20c229ec0a9707cc3
if __name__ == "__main__":
    main()


### Code — `audit_received_exposure_report.py` · phần 1/1: digest, main

**Mục đích:** Đọc REPORT.zip để đối chiếu hashes, smoke và benchmark state; không execute archive code.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma audit_received_exposure_report.py 1/1 0:1 69310f7a8a9331079e9dc991bdc8b032a254d346238e2eff97d3b91f96be66f4
"""Preserve and audit the received benchmark reports; never execute archive code."""
from pathlib import Path, PurePosixPath
import csv
import hashlib
import io
import json
import math
import sys
import zipfile

ROOT = Path.cwd() / "protocol_b_review/leakage_research_2026-10-11"
SOURCE = Path("C:/Users/ADMIN/Downloads/KLTN_GAN_Exposure_25d35efe55cc_REPORT.zip")
OUT = ROOT / "received_exposure_25d35efe55cc"
BASE = "KLTN_GAN_Exposure_25d35efe55cc/"
sys.path.insert(0, str(ROOT))
import gan_exposure_runner as runner
import gan_exposure_training as training


def digest(b):
    return hashlib.sha256(b).hexdigest()


def main():
    cfg = runner._exposure_release_config()
    with zipfile.ZipFile(SOURCE) as z:
        names = z.namelist()
        assert len(names) == len(set(names)) == 25 and z.testzip() is None
        for name in names:
            path = PurePosixPath(name)
            assert name.startswith(BASE) and not path.is_absolute() and ".." not in path.parts
        def read(name):
            return z.read(BASE + name)
        def load(name):
            return json.loads(read(name))
        suite = load("suite_manifest.json")
        assert suite["spec"] == runner._exposure_suite_spec(cfg)
        assert suite["suite_id"] == runner._exposure_json_hash(suite["spec"])[:12] == "25d35efe55cc"
        assert load("data_certificate_pin.json")["data_manifest_sha256"] == digest(read("data/data_manifest.json"))
        assert digest(read("data/region_selection.json")) == cfg["region_selection_sha256"]
        data = load("data/data_manifest.json")
        for region in (1,2,3):
            m = load(f"data/region_{region}/region_manifest.json")
            assert data["regions"][str(region)] == m
            assert m["oracle"]["blocked_shared_raw_sources"] == 0
            assert m["oracle"]["control_shared_raw_sources"] == (596,515,396)[region-1]
            assert all(m["counterfactual"][key] is True for key in (
                "all_blocked_train_supports_invariant", "preprocessing_invariant_to_test_source_spectra",
                "heldout_class_ids_without_eligibility_change_train_invariant", "reuse_control_inputs_change"))
        pin_bytes = read("jobs/pair_initialization/region01_seed42.json")
        pin = json.loads(pin_bytes)
        states, histories, smokes, absent_checkpoint_files = [], [], [], []
        for arm, gpu in (("blocked",0),("reuse_control",1)):
            job = f"jobs/region01_seed42_{arm}/"
            state = load(job + "recovery/state.json")
            initial = load(job + "initialization.json")
            contract = load(job + "training_contract.json")
            assert state["contract"] == contract
            assert state["contract_sha256"] == training._json_hash(contract)
            assert initial["combined_sha256"] == training._json_hash(initial["component_sha256"])
            assert state["initial_component_sha256"] == initial["combined_sha256"]
            assert pin == training._pair_pin_record(initial,contract)
            assert state["pair_initialization_sha256"] == digest(pin_bytes)
            assert (state["updates"],state["loop"],state["optimizer_clock"]) == (400,200,6800)
            smoke = load(f"gpu_smoke/worker_gpu{gpu}_region1_{arm}/gpu_smoke.json")
            assert all(smoke[key] is True for key in (
                "passed","all_15_heads_exercised","resume_exact_NumPy_and_dropout_streams",
                "resume_exact_component_and_optimizer_hashes",
                "inference_generation_keeps_BN_RNG_optimizer_unchanged"))
            assert all(route["passed"] for route in smoke["frozen_variable_routes"])
            assert smoke["contract"]["data"] == contract["data"]
            assert smoke["contract"]["runtime"] == contract["runtime"]
            assert smoke["test_evaluated"] is False
            history = list(csv.DictReader(io.StringIO(read(job+"training_history.csv").decode("utf-8"))))
            assert [int(row["classifier_updates"]) for row in history] == [100,200,300,400]
            assert all(math.isfinite(float(row[key])) for row in history for key in
                       ("C_loss","D_loss","G_vs_C_loss","G_vs_D_loss"))
            absent_checkpoint_files.extend(job + relative for relative in state["checkpoint"]["files"]
                                           if BASE+job+relative not in names)
            states.append(state); histories.append(history); smokes.append(smoke)
        assert states[0]["trace"] == states[1]["trace"]
        assert states[0]["dropout_state"] == states[1]["dropout_state"]
        assert states[0]["contract"]["pair_data_invariants"] == states[1]["contract"]["pair_data_invariants"]
        assert states[0]["actual_shared_raw_source_occurrences_presented"] == 0
        assert states[1]["actual_shared_raw_source_occurrences_presented"] == 33820
        progress = load("training_progress.json")
        assert len(progress["jobs"]) == 30 and sum(row["c_updates"] for row in progress["jobs"]) == 800
        assert progress["test_scored"] is False and not any("test_results" in name for name in names)
        bench = load("benchmark_summary.json")
        assert bench["remaining_c_updates"] == 239200
        speeds = []
        for history in histories:
            steady = [row for row in history if row["warmup_interval"] == "False"]
            speeds.append(sum(float(row["interval_seconds"]) for row in steady)
                          / (2*sum(int(row["interval_loops"]) for row in steady)))
        assert math.isclose(sum(speeds)/2, bench["mean_measured_seconds_per_c_update"],abs_tol=1e-12)
        payload_present = [name for name in names if name.endswith((".index",".npy",".npz",".h5"))
                           or ".data-" in name]
        assert not payload_present and len(absent_checkpoint_files) == 4
        full_gpu_hours = 240000 * bench["mean_measured_seconds_per_c_update"] / 3600
        result = {"report_metadata_audit_passed":True,"ZIP_CRC":"PASS","entries":len(names),
                  "zip_sha256":digest(SOURCE.read_bytes()),"release_id":cfg["implementation_release"],
                  "suite_id":suite["suite_id"],"pair_updates":400,"pair_GAN_loops":200,
                  "Adam_calls_per_arm":6800,"pair_initialization_data_trace_dropout_match":True,
                  "GPU_smoke_reported_passed_both_arms":True,"losses_finite":True,
                  "GPU_runtime":smokes[0]["contract"]["runtime"],
                  "real_C_presentations_per_arm":sum(states[0]["trace"]["real_counts"]),
                  "fake_C_presentations_per_arm":sum(states[0]["trace"]["fake_counts"]),
                  "shared_real_source_occurrences_blocked_control":[0,33820],
                  "measured_remaining_GPU_hours_if_checkpoint_survives":bench["estimated_remaining_gpu_hours"],
                  "estimated_GPU_hours_from_zero_if_checkpoint_lost":full_gpu_hours,
                  "ideal_two_T4_wall_hours_from_zero":full_gpu_hours/2,
                  "timing_scope":bench["scope"],"weights_and_arrays_in_report":False,
                  "checkpoint_files_verified_locally":False,"resume_from_report_alone":False,
                  "referenced_checkpoint_files_absent_from_ZIP":absent_checkpoint_files,
                  "classification_test_scored":False,"leakage_effect_estimated":False,
                  "archive_member_sha256":{name:digest(z.read(name)) for name in names}}
    OUT.mkdir(parents=True,exist_ok=True)
    received = OUT / SOURCE.name
    if received.exists():
        assert digest(received.read_bytes()) == result["zip_sha256"]
    else:
        received.write_bytes(SOURCE.read_bytes())
    (OUT/"audit.json").write_text(json.dumps(result,indent=2)+"\n",encoding="utf-8")
    print(json.dumps({key:value for key,value in result.items() if key != "archive_member_sha256"}))


if __name__ == "__main__":
    main()


### Code — `build_impact_probe.py` · phần 1/1: Imports, cấu hình và script body

**Mục đích:** Builder ghép ba fragment thành cell/notebook inference độc lập.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma build_impact_probe.py 1/1 0:1 65ff4127536c24badcd7a8b1efc2cd013df7ff124bdd113a8446ba6676a33e8e
"""Assemble a complete restart-independent Kaggle inference cell."""
from pathlib import Path
import ast
import hashlib
import json

ROOT = Path(__file__).resolve().parent
fragments = [ROOT/name for name in ("impact_probe_core.py","impact_probe_bootstrap_fragment.py",
                                   "impact_probe_main_fragment.py")]
payload = "\n\n".join(path.read_text(encoding="utf-8") for path in fragments)
implementation = hashlib.sha256(payload.encode()).hexdigest()
header = '''# DÁN TOÀN BỘ VÀO MỘT CELL KAGGLE MỚI; chạy riêng cell này với GPU.
# Dùng Notebook Outputs của B đã test xong; không cần globals/kernel của cell cũ.
# Chỉ inference C đã lưu. Không train, không load hoặc chạy G/D.
IMPACT_SOURCE_RUN_PATH = None
IMPACT_OUTPUT_ROOT = "/kaggle/working"
IMPACT_DEVICE = "/GPU:0"  # Có thể dùng /CPU:0 cho inference, tốc độ chậm hơn.
IMPACT_METHODS = ("c_real_only_matched", "hypergamo")
IMPACT_MODEL_SEEDS = (42,)
IMPACT_K = 5
IMPACT_MASK_SEEDS = (173, 947)
IMPACT_EVAL_BATCH = 64
'''
cell = header+"IMPACT_IMPLEMENTATION_SHA256 = "+repr(implementation)+"\n\n"+payload+"\n\nimpact_run_probe()\n"
ast.parse(cell)
source_path = ROOT/"CELL_KIEM_TRA_DO_NHAY_B.py"
source_path.write_text(cell,encoding="utf-8")
notebook = {
    "nbformat":4,"nbformat_minor":5,
    "metadata":{"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},
                "language_info":{"name":"python"}},
    "cells":[{"cell_type":"code","id":"b-impact-inference","metadata":{},
              "execution_count":None,"outputs":[],"source":cell.splitlines(keepends=True)}],
}
(ROOT/"CELL_KIEM_TRA_DO_NHAY_B.ipynb").write_text(json.dumps(notebook,ensure_ascii=False,indent=1),encoding="utf-8")
manifest = {"implementation_sha256":implementation,"fragments":{path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in fragments},
            "cell_sha256":hashlib.sha256(source_path.read_bytes()).hexdigest(),
            "runtime_note":"CPU algorithm/mock orchestration verified separately; real TF/GPU checkpoint inference needs Kaggle reproduction gate"}
(ROOT/"impact_probe_build_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print(json.dumps(manifest,indent=2))


### Code — `build_exposure_notebook.py` · phần 1/1: sha, main

**Mục đích:** Builder release và cell Kaggle độc lập; chưa cần chạy lại để review.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma build_exposure_notebook.py 1/1 0:1 153c30fdc87a3368751f080efdb4b268e51f892d5e3860f0cdfc3d64fe5e776a
"""Build one self-contained Kaggle code cell from reviewed local helpers."""
from pathlib import Path
import hashlib
import json
import ast
import zipfile

ROOT = Path.cwd() / "protocol_b_review/leakage_research_2026-10-11"


def sha(data):
    return hashlib.sha256(data).hexdigest()


def main():
    files = ["gan_exposure_data.py", "gan_exposure_training.py", "gan_exposure_runner.py"]
    code = {name: (ROOT / name).read_bytes().decode("utf-8") for name in files}
    for name, text in code.items():
        ast.parse(text, filename=name)
    hashes = {name: sha(text.encode("utf-8")) for name, text in code.items()}
    selection = (ROOT / "gan_exposure_region_selection.json").read_bytes().decode("utf-8")
    assert sha(selection.encode("utf-8")) == "c91ea95a6d91df3751b3b13643fb4545234b5e7565960e6b255cef4a887c912c"
    cfg = {
        "protocol": "registered_HyperGAMO_train_source_intervention_v1",
        "target_c_updates": 8000, "batch": 100, "latent": 100, "gen_points": 7,
        "learning_rate": 2e-4, "beta1": .5, "loss": "mse",
        "patch_size": 11, "pca_components": 40, "pca_solver": "full",
        "scaler_range": [-1, 1], "masked_context_fill": "scaled_train_center_mean",
        "image_padding_fill": 0, "preprocessing_fit": "train_center_spectra_only",
        "checkpoint_selection": "fixed8000_Cupdates", "seeds": [42,43,44,45,46],
        "regions": [[15,39,15,39],[15,39,105,129],[105,129,45,69]],
        "arms": ["blocked", "reuse_control"], "test_input": "identical_full11x11x40",
        "primary_metric": "OA_on_geometry_exposed751_pooled_perseed",
        "practical_delta_oa": .01, "minimum_positive_regions": 2,
        "minimum_positive_pairs": 12,
        "criterion_scope": "registered descriptive rule; not statistical/causal proof",
        "exposure_bins": [["zero",0,0],["1_to_20",1,20],["21_to_40",21,40],["41_to_121",41,121]],
        "evaluable_classes": [2,3,4,5,6,8,10,11,12,13,14,15],
        "gan_specific_claim": False, "historical_B_inflation_claim": False,
        "data_implementation_sha256": hashes["gan_exposure_data.py"],
        "training_implementation_sha256": hashes["gan_exposure_training.py"],
        "region_selection_sha256": sha(selection.encode("utf-8")),
        "raw_hsi_sha256": "ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939",
        "raw_gt_sha256": "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c",
    }
    release_id = sha(json.dumps({"components": hashes, "scientific": cfg},
                               sort_keys=True, separators=(",", ":")).encode())[:12]
    release_code = ("# Generated frozen scientific configuration.\n"
                    + "RELEASE_ID = " + repr(release_id) + "\n"
                    + "COMPONENT_HASHES = " + repr(hashes) + "\n"
                    + "SCIENTIFIC_CONFIG = " + repr(cfg) + "\n"
                    + "SELECTION_MANIFEST_TEXT = " + repr(selection) + "\n")
    (ROOT / "gan_exposure_release.py").write_text(release_code, encoding="utf-8", newline="\n")
    code["gan_exposure_release.py"] = release_code
    hashes["gan_exposure_release.py"] = sha(release_code.encode())
    header = '''# CHẠY RIÊNG CELL NÀY TRÊN KAGGLE; bật GPU trước.
# Không cần RUN/keras/checkpoint cũ; tạo thí nghiệm mới, không train lại B.
# Input cần Indian Pines corrected HSI + GT như notebook B, không chỉ ZIP báo cáo.
ACTION = "benchmark"       # benchmark -> train -> test. prepare chỉ kiểm CPU.
BENCHMARK_C_UPDATES = 400   # 400 đầu tiên tính trong ngân sách 8000; tiếp tục từ checkpoint.
MAX_WORKERS = 2            # T4 x2: hai process độc lập; P100: tự dùng một.
GPU_IDS = None             # None tự chọn; ví dụ [0] để chỉ dùng GPU 0.
SESSION_HOURS = 8          # Hết thời gian: checkpoint, lần sau tiếp tục.
SAVE_EVERY = 100           # C updates; checkpoint theo cặp cập nhật.
HSI_PATH = None            # Tự tìm file .mat đúng hash trong /kaggle/input.
GT_PATH = None
RESUME_RUN_PATH = None     # Tự khôi phục một Output suite đã lưu; nhiều snapshot thì điền rõ.
OUTPUT_ROOT = "/kaggle/working"
JOB_FILTER = None          # train mặc định tất cả30; tùy chọn [(1,42,"blocked"), ...].

'''
    # repr strings preserve CRLF and Unicode without relying on notebook globals.
    bundle = "EXPOSURE_COMPONENTS = {\n" + "".join(
        "    " + repr(name) + ": " + repr(source) + ",\n" for name,source in code.items()) + "}\n"
    bootstrap = '''
from pathlib import Path as _ExposurePath
import hashlib as _exposure_hashlib
import sys as _exposure_sys

_exposure_release_id = RELEASE_TAG
_exposure_code_root = _ExposurePath(OUTPUT_ROOT) / ("_gan_exposure_code_" + _exposure_release_id)
_exposure_code_root.mkdir(parents=True, exist_ok=True)
for _exposure_name, _exposure_source in EXPOSURE_COMPONENTS.items():
    _exposure_path = _exposure_code_root / _exposure_name
    _exposure_bytes = _exposure_source.encode("utf-8")
    if _exposure_path.exists() and _exposure_path.read_bytes() != _exposure_bytes:
        raise RuntimeError("Runner source khác release: " + str(_exposure_path))
    if not _exposure_path.exists():
        _exposure_path.write_bytes(_exposure_bytes)
_exposure_sys.path.insert(0, str(_exposure_code_root))
# Avoid importing an older helper cached by a previously executed release.
for _exposure_module in ("gan_exposure_data", "gan_exposure_training", "gan_exposure_runner", "gan_exposure_release"):
    _exposure_sys.modules.pop(_exposure_module, None)
from gan_exposure_runner import exposure_launch as _exposure_launch
_exposure_launch({"action":ACTION,"benchmark_c_updates":BENCHMARK_C_UPDATES,
    "max_workers":MAX_WORKERS,"gpu_ids":GPU_IDS,"session_hours":SESSION_HOURS,
    "save_every":SAVE_EVERY,"hsi_path":HSI_PATH,"gt_path":GT_PATH,
    "resume_run_path":RESUME_RUN_PATH,"output_root":OUTPUT_ROOT,"job_filter":JOB_FILTER},
    _exposure_code_root)
'''.replace("RELEASE_TAG", repr(release_id))
    cell = header + bundle + bootstrap
    ast.parse(cell, filename="CELL_GAN_TRAIN_EXPOSURE.py")
    (ROOT / "CELL_GAN_TRAIN_EXPOSURE.py").write_text(cell, encoding="utf-8", newline="\n")
    markdown = ("# HyperGAMO — kiểm can thiệp nguồn lúc train\n\n"
        "Bật GPU, giữ Input Indian Pines corrected HSI và GT. Chạy một cell code bên dưới.\n\n"
        "`ACTION=benchmark`: 400 C updates/nhánh của region1/seed42, lưu đầy đủ state. "
        "`ACTION=train`: tiếp tục cùng checkpoint tới 8.000, bỏ qua job đã xong; suite 30 jobs. "
        "`ACTION=test`: chỉ mở sau 30/30 và pair/hash gates. Không chấm test trong benchmark/train.\n\n"
        "Sau mỗi phiên Save Version + Save Output. Sau restart thêm Output suite đã lưu vào Input. "
        "Không Run All notebook B. Hai GPU dùng hai worker riêng; kernel cũ không quyết định memory growth của worker.\n\n"
        "CPU geometry/preprocessing/counterfactual được kiểm local; GPU freeze/optimizer/resume "
        "phải qua smoke trong môi trường Kaggle trước training. Không coi kiểm CPU là xác nhận GPU.\n")
    notebook = {"cells":[{"cell_type":"markdown","id":"exposure-instructions","metadata":{},"source":markdown.splitlines(True)},
                          {"cell_type":"code","id":"exposure-runner","metadata":{},"execution_count":None,"outputs":[],
                           "source":cell.splitlines(True)}],
                "metadata":{"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},
                            "language_info":{"name":"python","version":"3.12"},
                            "exposure_release":{"release_id":release_id,"GPU_executed_locally":False}},
                "nbformat":4,"nbformat_minor":5}
    (ROOT / "KAGGLE_GAN_TRAIN_EXPOSURE.ipynb").write_text(
        json.dumps(notebook, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    manifest = {"release_id":release_id,"scientific_config":cfg,"component_sha256":hashes,
                "cell_sha256":sha(cell.encode()),"notebook_sha256":sha(
                    (ROOT/"KAGGLE_GAN_TRAIN_EXPOSURE.ipynb").read_bytes()),
                "syntax_passed":True,"GPU_executed_locally":False,"test_scored_locally":False}
    (ROOT/"exposure_build_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
    print(json.dumps({"release_id":release_id,"cell_bytes":len(cell.encode()),"syntax":"PASS"}))


if __name__ == "__main__":
    main()


### Code — `package_exposure_release.py` · phần 1/1: sha, main

**Mục đích:** Đóng gói code release/checks để lưu; không phải weights training.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma package_exposure_release.py 1/1 0:1 f6c4d7f64e41bcfaf5b0209e41f381cf7c7ea8650645b2fd72bcda4ad1791362
"""Check and package the single-cell deliverable and its CPU audit evidence."""
from pathlib import Path
import ast
import hashlib
import json
import zipfile
import nbformat

ROOT = Path.cwd() / "protocol_b_review/leakage_research_2026-10-11"


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def main():
    build = json.loads((ROOT / "exposure_build_manifest.json").read_text(encoding="utf-8"))
    cell_path = ROOT / "CELL_GAN_TRAIN_EXPOSURE.py"
    cell = cell_path.read_text(encoding="utf-8")
    tree = ast.parse(cell)
    embedded = next(ast.literal_eval(node.value) for node in tree.body
                    if isinstance(node, ast.Assign)
                    and any(isinstance(target, ast.Name) and target.id == "EXPOSURE_COMPONENTS"
                            for target in node.targets))
    for name, source in embedded.items():
        ast.parse(source, filename=name)
        assert source.encode("utf-8") == (ROOT / name).read_bytes()
        assert hashlib.sha256(source.encode("utf-8")).hexdigest() == build["component_sha256"][name]
    assert sha(cell_path) == build["cell_sha256"]
    notebook_path = ROOT / "KAGGLE_GAN_TRAIN_EXPOSURE.ipynb"
    nb = nbformat.read(notebook_path, as_version=4)
    nbformat.validate(nb)
    code_cells = [entry for entry in nb.cells if entry.cell_type == "code"]
    assert len(code_cells) == 1 and code_cells[0].source == cell
    assert not code_cells[0].outputs and code_cells[0].execution_count is None
    assert sha(notebook_path) == build["notebook_sha256"]
    data = json.loads((ROOT / "exposure_data_cpu_certificate.json").read_text(encoding="utf-8"))
    review = json.loads((ROOT / "exposure_review_certificate.json").read_text(encoding="utf-8"))
    runner = json.loads((ROOT / "exposure_runner_cpu_certificate.json").read_text(encoding="utf-8"))
    assert data["implementation_sha256"] == sha(ROOT / "gan_exposure_data.py")
    assert review["passed"] is True
    for key, name in (("data_module_sha256", "gan_exposure_data.py"),
                      ("training_module_sha256", "gan_exposure_training.py"),
                      ("runner_module_sha256", "gan_exposure_runner.py")):
        assert review[key] == sha(ROOT / name)
    assert runner["status"] == "PASS" and runner["cell_sha256"] == sha(cell_path)
    assert runner["runner_sha256"] == sha(ROOT / "gan_exposure_runner.py")
    assert runner["GPU_executed"] is False and runner["scientific_test_scored"] is False
    files = ["KAGGLE_GAN_TRAIN_EXPOSURE.ipynb", "CELL_GAN_TRAIN_EXPOSURE.py",
             "HUONG_DAN_GAN_TRAIN_EXPOSURE.md", "DE_XUAT_KIEM_DINH_GAN_MANH.md",
             "gan_exposure_region_selection.json", "exposure_build_manifest.json",
             "exposure_data_cpu_certificate.json", "exposure_review_certificate.json",
             "exposure_runner_cpu_certificate.json"]
    delivery = {"release_id": build["release_id"], "single_code_cell": True,
                "notebook_schema_valid": True, "CPU_checks_passed": True,
                "GPU_executed_locally": False, "GAN_trained_locally": False,
                "scientific_test_scored_locally": False,
                "files_sha256": {name: sha(ROOT / name) for name in files}}
    manifest = ROOT / "release_delivery_manifest.json"
    manifest.write_text(json.dumps(delivery, indent=2) + "\n", encoding="utf-8")
    archive = ROOT / "BO_GAN_TRAIN_EXPOSURE.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for name in files + [manifest.name]:
            z.write(ROOT / name, name)
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
        for name, digest in delivery["files_sha256"].items():
            assert hashlib.sha256(z.read(name)).hexdigest() == digest
    print(json.dumps({"release_id": build["release_id"], "ZIP_CRC": "PASS",
                      "single_code_cell": True, "notebook_schema": "PASS",
                      "archive_bytes": archive.stat().st_size, "archive_sha256": sha(archive)}))


if __name__ == "__main__":
    main()


### Code — `plot_certificate.py` · phần 1/1: Imports, cấu hình và script body

**Mục đích:** Vẽ hình từ certificate CPU M3/M5; không tính classification.

**Đầu vào:** Fixtures/artifacts và đường dẫn local của từng script. **Đầu ra:** QA certificate, package hoặc hình tùy script.

**Trạng thái:** Phụ lục code kiểm/đóng gói; chỉ được lưu khi Run All review. **Giới hạn/cần review:** Một số script có đường dẫn Windows/ROOT local; driver nghiên cứu đã tách phía cuối notebook.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma plot_certificate.py 1/1 0:1 a1ecf1e62af54450dc4e4611da448e118a5b9600c9c3df956545c48e9f0bc569
"""Static geometry illustration; no classification measurements."""
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from spatial_leakage_core import Lattice, source_indices

ROOT = Path(__file__).resolve().parent
lattice = Lattice(3, (0, 1, 2))
fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.8))
fig.suptitle("Cửa sổ giao nhau, các pixel thực sự được dùng tách biệt", fontsize=17, y=.97)
colors = ["#1976D2", "#E97824"]
centers = np.array([[6, 6], [6, 7]])
names = ["Tâm train", "Tâm validation"]
ax = axes[0]
rr, cc = np.indices((14, 15))
ax.scatter(cc.ravel(), rr.ravel(), s=20, c="#e4e8ec", marker="s")
for i, center in enumerate(centers):
    ids = source_indices((14, 15), center[None], lattice)[0]
    ids = ids[ids >= 0]
    used_r, used_c = np.unravel_index(ids, (14, 15))
    ax.scatter(used_c, used_r, s=48, c=colors[i], marker="s", label=f"Pixel dùng bởi {names[i].lower()}")
    ax.add_patch(Rectangle((center[1]-5.5, center[0]-5.5), 11, 11,
                           fill=False, edgecolor=colors[i], lw=2, linestyle="--"))
    ax.scatter(center[1], center[0], s=170, c=colors[i], edgecolor="#20262c", marker="*", zorder=5)
ax.set(xlim=(-.7, 14.7), ylim=(13.7, -.7), aspect="equal", xlabel="Cột", ylabel="Hàng")
ax.set_title("Hai tâm sát nhau; giao nguồn quan sát = 0", fontsize=12, pad=12)
ax.legend(loc="lower center", bbox_to_anchor=(.5, -.25), frameon=False, fontsize=10)

ax = axes[1]
mask = lattice.offset_mask()
ax.imshow(mask, cmap=matplotlib.colors.ListedColormap(["#edf0f3", colors[0]]), interpolation="nearest")
ax.scatter(5, 5, s=170, c="#fff", edgecolor="#20262c", marker="*", zorder=5)
ax.set_xticks(np.arange(11), np.arange(-5, 6))
ax.set_yticks(np.arange(11), np.arange(-5, 6))
ax.set(xlabel="Độ lệch cột so với tâm", ylabel="Độ lệch hàng so với tâm")
ax.set_title("Cùng một mask cho mọi tâm bên trong ảnh", fontsize=12, pad=12)
ax.text(.5, -.17, "41 pixel quan sát / 121 vị trí trong cửa sổ 11×11\nVị trí xám là phần thiếu, không phải phổ thật",
        ha="center", va="top", transform=ax.transAxes, fontsize=11)
fig.text(.5, .035, "Đã kiểm trên Indian Pines: đủ 16 lớp ở cả ba tập. Tâm vẫn có thể cách nhau 1 pixel; chưa có độc lập không gian.",
         ha="center", fontsize=10, color="#414b56")
fig.subplots_adjust(left=.06, right=.98, top=.84, bottom=.27, wspace=.28)
for suffix in ("png", "svg"):
    fig.savefig(ROOT / f"NGUON_PIXEL_TACH_BIET.{suffix}", dpi=180, facecolor="white")
plt.close(fig)


### Code — `strong_gan_design_geometry.json` · phần 1/1: Artifact đăng ký đọc được

**Mục đích:** Lưu phương án crossed M3 để nhóm đọc thiết kế đã khảo sát.

**Đầu vào:** Artifact geometry lịch sử. **Đầu ra:** JSON mô tả feasibility.

**Trạng thái:** Có khảo sát geometry; không có runner36jobs trong nguồn lưu. **Giới hạn/cần review:** Phụ lục thiết kế, không thêm vào suite30jobs và không giả là code đã train.

Chạy cell này chỉ lưu source; lệnh thực nghiệm ở cuối notebook.

In [ ]:
%%luu_ma strong_gan_design_geometry.json 1/1 0:0 d02d972f2c75334f103f739fd10a84cb9da18ec461aa10922d161603507c0425
{
  "GT_sha256": "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c",
  "candidate_geometries": [
    {
      "rectangle": [
        15,
        39,
        15,
        39
      ],
      "test_n": 437,
      "test_classes": [
        2,
        3,
        4,
        10,
        12,
        15
      ],
      "train_n": 9373,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 543,
      "affected_test_n": 285,
      "affected_classes": [
        2,
        3,
        4,
        10,
        12,
        15
      ],
      "train_class_counts": [
        46,
        1047,
        685,
        155,
        473,
        724,
        28,
        478,
        20,
        936,
        2455,
        476,
        205,
        1265,
        297,
        83
      ]
    },
    {
      "rectangle": [
        15,
        39,
        45,
        69
      ],
      "test_n": 439,
      "test_classes": [
        2,
        12,
        16
      ],
      "train_n": 9461,
      "train_classes": 15,
      "minimum_train_class_n": 0,
      "touching_train_n": 416,
      "affected_test_n": 283,
      "affected_classes": [
        2,
        12,
        16
      ],
      "train_class_counts": [
        46,
        1084,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        940,
        2455,
        274,
        205,
        1265,
        386,
        0
      ]
    },
    {
      "rectangle": [
        15,
        39,
        75,
        99
      ],
      "test_n": 193,
      "test_classes": [
        10,
        11,
        14
      ],
      "train_n": 9724,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 585,
      "affected_test_n": 142,
      "affected_classes": [
        10,
        11,
        14
      ],
      "train_class_counts": [
        46,
        1389,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        699,
        2394,
        585,
        205,
        1174,
        333,
        93
      ]
    },
    {
      "rectangle": [
        15,
        39,
        105,
        129
      ],
      "test_n": 330,
      "test_classes": [
        8,
        11,
        14
      ],
      "train_n": 9541,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 434,
      "affected_test_n": 250,
      "affected_classes": [
        8,
        11,
        14
      ],
      "train_class_counts": [
        46,
        1398,
        830,
        237,
        483,
        730,
        28,
        332,
        20,
        957,
        2248,
        593,
        205,
        955,
        386,
        93
      ]
    },
    {
      "rectangle": [
        45,
        69,
        15,
        39
      ],
      "test_n": 501,
      "test_classes": [
        2,
        3,
        4,
        6,
        9,
        11,
        12
      ],
      "train_n": 9312,
      "train_classes": 15,
      "minimum_train_class_n": 0,
      "touching_train_n": 619,
      "affected_test_n": 324,
      "affected_classes": [
        2,
        3,
        4,
        6,
        9,
        11,
        12
      ],
      "train_class_counts": [
        46,
        1295,
        698,
        118,
        457,
        460,
        28,
        478,
        0,
        972,
        2316,
        495,
        205,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        45,
        69,
        45,
        69
      ],
      "test_n": 392,
      "test_classes": [
        2,
        11
      ],
      "train_n": 9442,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 613,
      "affected_test_n": 283,
      "affected_classes": [
        2,
        11
      ],
      "train_class_counts": [
        46,
        1123,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        926,
        1999,
        593,
        205,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        45,
        69,
        75,
        99
      ],
      "test_n": 547,
      "test_classes": [
        1,
        2,
        10,
        11
      ],
      "train_n": 9334,
      "train_classes": 15,
      "minimum_train_class_n": 0,
      "touching_train_n": 511,
      "affected_test_n": 337,
      "affected_classes": [
        1,
        2,
        10,
        11
      ],
      "train_class_counts": [
        0,
        1176,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        399,
        2411,
        593,
        205,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        45,
        69,
        105,
        129
      ],
      "test_n": 331,
      "test_classes": [
        2,
        5,
        8,
        10,
        11
      ],
      "train_n": 9577,
      "train_classes": 16,
      "minimum_train_class_n": 16,
      "touching_train_n": 456,
      "affected_test_n": 215,
      "affected_classes": [
        2,
        5,
        8,
        10,
        11
      ],
      "train_class_counts": [
        33,
        1282,
        830,
        237,
        451,
        730,
        16,
        198,
        20,
        906,
        2332,
        593,
        205,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        75,
        99,
        15,
        39
      ],
      "test_n": 361,
      "test_classes": [
        5,
        11
      ],
      "train_n": 9500,
      "train_classes": 16,
      "minimum_train_class_n": 18,
      "touching_train_n": 522,
      "affected_test_n": 277,
      "affected_classes": [
        5,
        11
      ],
      "train_class_counts": [
        46,
        1420,
        819,
        237,
        300,
        710,
        28,
        478,
        18,
        972,
        1930,
        593,
        205,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        75,
        99,
        45,
        69
      ],
      "test_n": 543,
      "test_classes": [
        6,
        11
      ],
      "train_n": 9253,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 709,
      "affected_test_n": 318,
      "affected_classes": [
        6,
        11
      ],
      "train_class_counts": [
        46,
        1381,
        830,
        237,
        483,
        612,
        28,
        478,
        20,
        972,
        1624,
        593,
        205,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        75,
        99,
        75,
        99
      ],
      "test_n": 271,
      "test_classes": [
        2,
        6
      ],
      "train_n": 9728,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 383,
      "affected_test_n": 210,
      "affected_classes": [
        2,
        6
      ],
      "train_class_counts": [
        21,
        1132,
        830,
        237,
        483,
        574,
        28,
        478,
        20,
        966,
        2417,
        593,
        205,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        105,
        129,
        15,
        39
      ],
      "test_n": 391,
      "test_classes": [
        3,
        11,
        13
      ],
      "train_n": 9508,
      "train_classes": 16,
      "minimum_train_class_n": 16,
      "touching_train_n": 293,
      "affected_test_n": 211,
      "affected_classes": [
        3,
        11,
        13
      ],
      "train_class_counts": [
        46,
        1428,
        578,
        237,
        483,
        730,
        28,
        478,
        20,
        972,
        2155,
        593,
        16,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        105,
        129,
        45,
        69
      ],
      "test_n": 342,
      "test_classes": [
        3,
        5,
        6,
        11,
        13
      ],
      "train_n": 9582,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 414,
      "affected_test_n": 206,
      "affected_classes": [
        3,
        5,
        6,
        11,
        13
      ],
      "train_class_counts": [
        46,
        1428,
        767,
        237,
        395,
        495,
        28,
        478,
        20,
        972,
        2230,
        593,
        149,
        1265,
        386,
        93
      ]
    },
    {
      "rectangle": [
        105,
        129,
        75,
        99
      ],
      "test_n": 258,
      "test_classes": [
        6,
        14
      ],
      "train_n": 9699,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 351,
      "affected_test_n": 163,
      "affected_classes": [
        6,
        14
      ],
      "train_class_counts": [
        46,
        1428,
        830,
        237,
        483,
        553,
        28,
        478,
        20,
        972,
        2455,
        593,
        205,
        892,
        386,
        93
      ]
    },
    {
      "rectangle": [
        105,
        129,
        105,
        129
      ],
      "test_n": 271,
      "test_classes": [
        14
      ],
      "train_n": 9788,
      "train_classes": 16,
      "minimum_train_class_n": 20,
      "touching_train_n": 202,
      "affected_test_n": 150,
      "affected_classes": [
        14
      ],
      "train_class_counts": [
        46,
        1428,
        830,
        237,
        483,
        730,
        28,
        478,
        20,
        972,
        2455,
        593,
        205,
        804,
        386,
        93
      ]
    }
  ],
  "crossed_geometry": [
    {
      "split_sign": 1,
      "mask_sign": 1,
      "split_counts": [
        3412,
        3424,
        3413
      ],
      "class_counts": [
        [
          16,
          475,
          275,
          81,
          162,
          244,
          10,
          162,
          7,
          323,
          813,
          197,
          66,
          421,
          128,
          32
        ],
        [
          15,
          475,
          281,
          77,
          160,
          246,
          9,
          159,
          6,
          327,
          819,
          199,
          71,
          422,
          128,
          30
        ],
        [
          15,
          478,
          274,
          79,
          161,
          240,
          9,
          157,
          7,
          322,
          823,
          197,
          68,
          422,
          130,
          31
        ]
      ],
      "shared_raw_pixels_pairs": [
        0,
        0,
        0
      ],
      "mean_sources": 39.94116499170651,
      "sources_min": 1
    },
    {
      "split_sign": 1,
      "mask_sign": -1,
      "split_counts": [
        3412,
        3424,
        3413
      ],
      "class_counts": [
        [
          16,
          475,
          275,
          81,
          162,
          244,
          10,
          162,
          7,
          323,
          813,
          197,
          66,
          421,
          128,
          32
        ],
        [
          15,
          475,
          281,
          77,
          160,
          246,
          9,
          159,
          6,
          327,
          819,
          199,
          71,
          422,
          128,
          30
        ],
        [
          15,
          478,
          274,
          79,
          161,
          240,
          9,
          157,
          7,
          322,
          823,
          197,
          68,
          422,
          130,
          31
        ]
      ],
      "shared_raw_pixels_pairs": [
        15492,
        15496,
        15503
      ],
      "mean_sources": 39.94116499170651,
      "sources_min": 1
    },
    {
      "split_sign": -1,
      "mask_sign": 1,
      "split_counts": [
        3418,
        3413,
        3418
      ],
      "class_counts": [
        [
          13,
          479,
          277,
          79,
          160,
          243,
          10,
          159,
          7,
          324,
          818,
          198,
          69,
          422,
          129,
          31
        ],
        [
          18,
          474,
          278,
          79,
          161,
          244,
          9,
          160,
          6,
          325,
          819,
          196,
          68,
          419,
          127,
          30
        ],
        [
          15,
          475,
          275,
          79,
          162,
          243,
          9,
          159,
          7,
          323,
          818,
          199,
          68,
          424,
          130,
          32
        ]
      ],
      "shared_raw_pixels_pairs": [
        15494,
        15496,
        15492
      ],
      "mean_sources": 39.94116499170651,
      "sources_min": 1
    },
    {
      "split_sign": -1,
      "mask_sign": -1,
      "split_counts": [
        3418,
        3413,
        3418
      ],
      "class_counts": [
        [
          13,
          479,
          277,
          79,
          160,
          243,
          10,
          159,
          7,
          324,
          818,
          198,
          69,
          422,
          129,
          31
        ],
        [
          18,
          474,
          278,
          79,
          161,
          244,
          9,
          160,
          6,
          325,
          819,
          196,
          68,
          419,
          127,
          30
        ],
        [
          15,
          475,
          275,
          79,
          162,
          243,
          9,
          159,
          7,
          323,
          818,
          199,
          68,
          424,
          130,
          32
        ]
      ],
      "shared_raw_pixels_pairs": [
        0,
        0,
        0
      ],
      "mean_sources": 39.94116499170651,
      "sources_min": 1
    }
  ],
  "reflection_equal_source_count": true,
  "symmetrically_clipped_boundaries": true,
  "model_training_performed": false
}

## 08 — Cấu hình release nguyên bản và cửa chạy thực nghiệm

### Code — tái tạo `gan_exposure_release.py` đúng SHA

**Mục đích:** trình bày cấu hình đọc được thay vì một dòng repr JSON rất dài. **Đầu vào:** constants dưới đây và JSON region selection đã lưu. **Đầu ra:** module release có bytes giống bản đã benchmark. Chạy cell chỉ ghi module; không train. Không sửa LR, seeds, regions, ngân sách hoặc tiêu chí sau khi test.


In [ ]:
REVIEW_RELEASE_VALUES = {'RELEASE_ID': '4c08e7e92312',
 'COMPONENT_HASHES': {'gan_exposure_data.py': 'bf7cde0d7305f430d8ab85866b7e3b66ea99903cf7d5ff8fac5d6bbe71920733',
                      'gan_exposure_training.py': '8a1ef7dd13702665cfa56c30ef2653b43076a9c2af67d6a31af17aea50ad5c63',
                      'gan_exposure_runner.py': 'bc5616400f49be3a40d86fa39f69ce019941f2ac9f1886414e2a22a3f465f5be'},
 'SCIENTIFIC_CONFIG': {'protocol': 'registered_HyperGAMO_train_source_intervention_v1',
                       'target_c_updates': 8000,
                       'batch': 100,
                       'latent': 100,
                       'gen_points': 7,
                       'learning_rate': 0.0002,
                       'beta1': 0.5,
                       'loss': 'mse',
                       'patch_size': 11,
                       'pca_components': 40,
                       'pca_solver': 'full',
                       'scaler_range': [-1, 1],
                       'masked_context_fill': 'scaled_train_center_mean',
                       'image_padding_fill': 0,
                       'preprocessing_fit': 'train_center_spectra_only',
                       'checkpoint_selection': 'fixed8000_Cupdates',
                       'seeds': [42, 43, 44, 45, 46],
                       'regions': [[15, 39, 15, 39], [15, 39, 105, 129], [105, 129, 45, 69]],
                       'arms': ['blocked', 'reuse_control'],
                       'test_input': 'identical_full11x11x40',
                       'primary_metric': 'OA_on_geometry_exposed751_pooled_perseed',
                       'practical_delta_oa': 0.01,
                       'minimum_positive_regions': 2,
                       'minimum_positive_pairs': 12,
                       'criterion_scope': 'registered descriptive rule; not statistical/causal proof',
                       'exposure_bins': [['zero', 0, 0],
                                         ['1_to_20', 1, 20],
                                         ['21_to_40', 21, 40],
                                         ['41_to_121', 41, 121]],
                       'evaluable_classes': [2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15],
                       'gan_specific_claim': False,
                       'historical_B_inflation_claim': False,
                       'data_implementation_sha256': 'bf7cde0d7305f430d8ab85866b7e3b66ea99903cf7d5ff8fac5d6bbe71920733',
                       'training_implementation_sha256': '8a1ef7dd13702665cfa56c30ef2653b43076a9c2af67d6a31af17aea50ad5c63',
                       'region_selection_sha256': 'c91ea95a6d91df3751b3b13643fb4545234b5e7565960e6b255cef4a887c912c',
                       'raw_hsi_sha256': 'ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939',
                       'raw_gt_sha256': '65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c'}}
_selection = (REVIEW_CODE_ROOT/"gan_exposure_region_selection.json").read_bytes().decode("utf-8")
_release_text = ("# Generated frozen scientific configuration.\n"
    + "RELEASE_ID = " + repr(REVIEW_RELEASE_VALUES["RELEASE_ID"]) + "\n"
    + "COMPONENT_HASHES = " + repr(REVIEW_RELEASE_VALUES["COMPONENT_HASHES"]) + "\n"
    + "SCIENTIFIC_CONFIG = " + repr(REVIEW_RELEASE_VALUES["SCIENTIFIC_CONFIG"]) + "\n"
    + "SELECTION_MANIFEST_TEXT = " + repr(_selection) + "\n")
_review_write_exact("gan_exposure_release.py", _release_text.encode("utf-8"))


### Code — kiểm đủ source và định nghĩa cửa chạy

**Mục đích:** xác nhận bản chia sẻ đủ module, nối driver, giữ runtime GAN trong subprocess GPU của runner gốc. **Đầu vào:** source được lưu và cấu hình đầu notebook. **Đầu ra:** hàm `review_launch`; chưa thực thi. Paths notebook dùng `Path.cwd()`, không dùng `__file__` tại notebook. Các module `.py` được import bình thường nên vẫn có `__file__`.


In [ ]:
for _name, _spec in REVIEW_SOURCE_SPECS.items():
    _path = REVIEW_CODE_ROOT/_name
    if not _path.is_file() or hashlib.sha256(_path.read_bytes()).hexdigest() != _spec["sha256"]:
        raise RuntimeError("Thiếu/sai source: " + _name)
sys.path.insert(0, str(REVIEW_CODE_ROOT.resolve()))
# Discard an older cached module from a different release; importing below is lazy.
for _name in ("team_review_adapter", "spatial_leakage_core", "audit_real_data", "context_retention_bound",
              "impact_probe_feasibility", "gan_exposure_data", "gan_exposure_training",
              "gan_exposure_runner", "gan_exposure_release"):
    sys.modules.pop(_name, None)


def review_launch(action):
    actions = {"read_only", "split_audit", "identity", "m3_m5", "context_bound", "probe_feasibility",
               "frozen_sensitivity", "gan_prepare", "gan_benchmark", "gan_train", "gan_test"}
    if action not in actions:
        raise ValueError("REVIEW_ACTION không hợp lệ: " + str(action))
    if action == "read_only":
        print("REVIEW XONG: chỉ lưu/kiểm source. Chưa mở data, checkpoint, train hoặc test.")
        return
    import team_review_adapter as adapter
    out = Path(OUTPUT_ROOT)
    out.mkdir(parents=True, exist_ok=True)
    if action == "split_audit":
        return adapter.run_split_audit(REVIEW_CODE_ROOT, COMPARISON_SOURCE, out)
    if action == "frozen_sensitivity":
        return adapter.run_frozen_sensitivity(REVIEW_CODE_ROOT, B_OUTPUT_SOURCE, out, IMPACT_DEVICE)
    if action.startswith("gan_"):
        from gan_exposure_runner import exposure_launch
        return exposure_launch({"action": action.removeprefix("gan_"),
            "benchmark_c_updates": BENCHMARK_C_UPDATES, "max_workers": MAX_WORKERS,
            "gpu_ids": GPU_IDS, "session_hours": SESSION_HOURS, "save_every": SAVE_EVERY,
            "hsi_path": HSI_PATH, "gt_path": GT_PATH, "resume_run_path": RESUME_RUN_PATH,
            "output_root": str(out), "job_filter": JOB_FILTER}, REVIEW_CODE_ROOT)
    search_roots = ["/kaggle/input"]
    hsi = adapter.find_mat(HSI_PATH, adapter.RAW_HSI_SHA256, search_roots)
    gt_path = adapter.find_mat(GT_PATH, adapter.RAW_GT_SHA256, search_roots)
    if action == "m3_m5":
        return adapter.run_m3_m5(REVIEW_CODE_ROOT, hsi, gt_path, COMPARISON_ZIP, out)
    cube, gt, train, test, dev_train, dev_val, identity = adapter.build_identity(hsi, gt_path)
    if action == "identity":
        return adapter.write_report(out/"protocol_identity.json", identity)
    if action == "context_bound":
        return adapter.run_bound(gt, out)
    if action == "probe_feasibility":
        return adapter.run_feasibility(gt, train, test, dev_train, dev_val, out)


### Cell cuối — thực thi action đã chọn

**Mặc định:** chỉ thông báo kiểm nguồn hoàn tất. Để chạy một bước, chỉnh `REVIEW_ACTION` ở cell cấu hình rồi chạy notebook trong kernel sạch. Các bước lịch sử không bắt buộc chạy lại để tiếp tục suite GAN.

**Benchmark đã xong; không cần chạy benchmark lại.** Bước hiện tại là `gan_train` → `gan_test`. Trainer tiếp tục checkpoint đến 8.000 C updates, bỏ qua jobs đã xong; nếu checkpoint không còn thì train từ khởi tạo đã khóa. GPU smoke vẫn tự kiểm trước training. Test gates yêu cầu 30/30 jobs complete và pair/hash đúng trước khi mở test. Code benchmark được giữ để nhóm đọc cách đã kiểm runtime; 400 updates đã chạy nằm trong ngân sách 8.000 nếu checkpoint còn.

Save Version + Save Output sau mỗi phiên và giữ **Output đầy đủ** để resume; tải REPORT.zip riêng để gửi phân tích. REPORT.zip không khôi phục weights. Nhóm chỉ đọc hoặc Run All với `read_only`; không cần chạy lại các kiểm tra lịch sử.

**Đánh giá đã khóa:** ΔOA = reuse_control − blocked trên751exposed, pooled theo từng seed; toàn1109/zero358/perregion/perclass/NLL là bổ sung. Quy tắc1điểmOA,≥2/3regionmeanspositive,≥12/15pairspositive là tiêu chí mô tả, không là chứng minh thống kê/nhân quả. Test vẫn argmax đầy16outputs.


In [ ]:
review_launch(REVIEW_ACTION)


## Gợi ý phân công review nhóm

1. Data/geometry: kiểm raw-source IDs, boundary/padding, center retention, PCA train-only và oracle độc lập.
2. GAN: kiểm supports đúng arm, online fake exposure, freeze/update routes, shared Adam và checkpoint RNG/Dropout.
3. Runner/test: kiểm pair initialization/data/trace, ngân sách8000, resume,30jobgates và identical full test inputs.
4. Diễn giải: phân biệt source overlap, sensitivity, ownership feasibility và training intervention; xem đủ lớp/mẫu và giới hạn inference.

Kết quả sâu sẽ bổ sung từ CSV/JSON/ZIP thực tế; các dòng trạng thái trong notebook là mốc đã xác minh, không phải outputs được chạy lại khi đóng gói. Không có weights hoặc dữ liệu ảnh trong file notebook.
